In [1]:
# Establishing project paths, Azure connection and source file registry

import os
import csv
import pandas as pd
import numpy as np

from io import BytesIO
from pathlib import Path

from azure.storage.blob import BlobServiceClient

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 180)

current_working_directory = Path.cwd().resolve()
project_root = current_working_directory.parent if current_working_directory.name == "notebooks" else current_working_directory

storage_connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")

if not storage_connection_string:
    raise ValueError("AZURE_STORAGE_CONNECTION_STRING environment variable is not set.")

blob_service_client = BlobServiceClient.from_connection_string(storage_connection_string)

container_names = {"raw": "raw",
                   "curated": "curated",
                   "presentation": "presentation",
                   "model_output": "model-output"
}

raw_dataset_prefix = "KuaiRand-Pure/data"

source_blob_paths = {"interaction_log_standard_20220408_20220421": f"{raw_dataset_prefix}/log_standard_4_08_to_4_21_pure.csv",
                     "interaction_log_standard_20220422_20220508": f"{raw_dataset_prefix}/log_standard_4_22_to_5_08_pure.csv",
                     "interaction_log_random_20220422_20220508": f"{raw_dataset_prefix}/log_random_4_22_to_5_08_pure.csv",
                     "user_dimension": f"{raw_dataset_prefix}/user_features_pure.csv",
                     "video_dimension_basic": f"{raw_dataset_prefix}/video_features_basic_pure.csv",
                     "video_dimension_statistic": f"{raw_dataset_prefix}/video_features_statistic_pure.csv"
}

# Verifying that all required raw blobs are available
def blob_exists(container_name, blob_name):
    blob_client = blob_service_client.get_blob_client(container = container_name, blob = blob_name)
    return blob_client.exists()

missing_source_blobs = [source_name
                        for source_name, blob_name in source_blob_paths.items()
                        if not blob_exists(container_names["raw"], blob_name)
]

if missing_source_blobs:
    raise FileNotFoundError(f"Missing expected source blobs : {missing_source_blobs}")

print(f"Project root : {project_root}")
print(f"Raw container : {container_names['raw']}")
print("Blob connection established successfully.")

Project root : C:\Users\azureuser\Documents\Short-Video Engagement Analysis
Raw container : raw
Blob connection established successfully.


In [2]:
# Auditing source file structure and size before loading full data

def download_blob_bytes(container_name, blob_name):
    blob_client = blob_service_client.get_blob_client(container = container_name, blob = blob_name)
    return blob_client.download_blob().readall()

def read_csv_header_from_blob(container_name, blob_name):
    blob_bytes = download_blob_bytes(container_name, blob_name)
    first_line = blob_bytes.decode("utf-8").splitlines()[0]
    return next(csv.reader([first_line]))

def count_csv_rows_in_blob(container_name, blob_name):
    blob_bytes = download_blob_bytes(container_name, blob_name)
    line_count = len(blob_bytes.decode("utf-8").splitlines())
    return max(line_count - 1, 0)

source_file_inventory = []

for source_name, blob_name in source_blob_paths.items():
    header_columns = read_csv_header_from_blob(container_names["raw"], blob_name)
    file_row_count = count_csv_rows_in_blob(container_names["raw"], blob_name)

    source_file_inventory.append(
        {
            "source_name": source_name,
            "blob_name": blob_name,
            "row_count": file_row_count,
            "column_count": len(header_columns),
            "columns": ", ".join(header_columns),
        }
    )

source_file_inventory = pd.DataFrame(source_file_inventory).sort_values("source_name").reset_index(drop = True)
source_file_inventory

,source_name,blob_name,row_count,column_count,columns
0,interaction_log_random_20220422_20220508,KuaiRand-Pure/data/log_random_4_22_to_5_08_pur...,1186059,19,"user_id, video_id, date, hourmin, time_ms, is_..."
1,interaction_log_standard_20220408_20220421,KuaiRand-Pure/data/log_standard_4_08_to_4_21_p...,1141112,19,"user_id, video_id, date, hourmin, time_ms, is_..."
2,interaction_log_standard_20220422_20220508,KuaiRand-Pure/data/log_standard_4_22_to_5_08_p...,295497,19,"user_id, video_id, date, hourmin, time_ms, is_..."
3,user_dimension,KuaiRand-Pure/data/user_features_pure.csv,27285,31,"user_id, user_active_degree, is_lowactive_peri..."
4,video_dimension_basic,KuaiRand-Pure/data/video_features_basic_pure.csv,7583,12,"video_id, author_id, video_type, upload_dt, up..."
5,video_dimension_statistic,KuaiRand-Pure/data/video_features_statistic_pu...,7583,52,"video_id, counts, show_cnt, show_user_num, pla..."


In [3]:
# Defining reusable Blob read and write helpers for the workflow

def read_csv_from_blob(container_name, blob_name, **read_csv_kwargs):
    blob_bytes = download_blob_bytes(container_name, blob_name)
    return pd.read_csv(BytesIO(blob_bytes), **read_csv_kwargs)

def read_parquet_from_blob(container_name, blob_name, **read_parquet_kwargs):
    blob_bytes = download_blob_bytes(container_name, blob_name)
    return pd.read_parquet(BytesIO(blob_bytes), **read_parquet_kwargs)

def upload_dataframe_to_parquet_blob(dataframe, container_name, blob_name):
    buffer = BytesIO()
    dataframe.to_parquet(buffer, index = False)
    buffer.seek(0)

    blob_client = blob_service_client.get_blob_client(container = container_name, blob = blob_name)
    blob_client.upload_blob(buffer.getvalue(), overwrite = True)

In [4]:
# Loading source tables with controlled data types for stable ingestion

interaction_dtype_map = {"time_ms": "int64",
                         "user_id": "int32", "video_id": "int32", "date": "int32", "hourmin": "int32", 
                         "play_time_ms": "int32", "duration_ms": "int32", "profile_stay_time": "int32", 
                         "comment_stay_time": "int32",
                         "is_click": "int8", "is_like": "int8", "is_follow": "int8", "is_comment": "int8",
                         "is_forward": "int8", "is_hate": "int8", "long_view": "int8", "is_profile_enter": "int8",
                         "is_rand": "int8", "tab": "int8"
}

interaction_log_standard_20220408_20220421 = read_csv_from_blob(
    container_names["raw"],
    source_blob_paths["interaction_log_standard_20220408_20220421"],
    dtype = interaction_dtype_map
)

interaction_log_standard_20220422_20220508 = read_csv_from_blob(
    container_names["raw"],
    source_blob_paths["interaction_log_standard_20220422_20220508"],
    dtype = interaction_dtype_map
)

interaction_log_random_20220422_20220508 = read_csv_from_blob(
    container_names["raw"],
    source_blob_paths["interaction_log_random_20220422_20220508"],
    dtype = interaction_dtype_map
)

user_dimension = read_csv_from_blob(
    container_names["raw"],
    source_blob_paths["user_dimension"]
)

video_dimension_basic = read_csv_from_blob(
    container_names["raw"],
    source_blob_paths["video_dimension_basic"]
)

video_dimension_statistic = read_csv_from_blob(
    container_names["raw"],
    source_blob_paths["video_dimension_statistic"]
)

print(interaction_log_standard_20220408_20220421.shape)
print(interaction_log_standard_20220422_20220508.shape)
print(interaction_log_random_20220422_20220508.shape)
print(user_dimension.shape)
print(video_dimension_basic.shape)
print(video_dimension_statistic.shape)

(1141112, 19)
(295497, 19)
(1186059, 19)
(27285, 31)
(7583, 12)
(7583, 52)


In [5]:
# Combining interaction logs into one traceable raw event base

interaction_frames = {"log_standard_4_08_to_4_21_pure": interaction_log_standard_20220408_20220421.copy(),
                      "log_standard_4_22_to_5_08_pure": interaction_log_standard_20220422_20220508.copy(),
                      "log_random_4_22_to_5_08_pure": interaction_log_random_20220422_20220508.copy()
}

interaction_source_metadata = {
    "log_standard_4_08_to_4_21_pure": {"traffic_source": "standard",
                                       "source_period": "20220408_20220421"
    },
    "log_standard_4_22_to_5_08_pure": {"traffic_source": "standard",
                                       "source_period": "20220422_20220508"
    },
    "log_random_4_22_to_5_08_pure": {"traffic_source": "random",
                                     "source_period": "20220422_20220508"
    },
}

raw_interaction_log = pd.concat(
    [
        frame.assign(source_file = source_name,
                     traffic_source = interaction_source_metadata[source_name]["traffic_source"],
                     source_period = interaction_source_metadata[source_name]["source_period"]
        )
        for source_name, frame in interaction_frames.items()
    ],
    ignore_index = True
)

print(raw_interaction_log.shape)
raw_interaction_log.head()

(2622668, 22)


,user_id,video_id,date,hourmin,time_ms,is_click,is_like,is_follow,is_comment,is_forward,is_hate,long_view,play_time_ms,duration_ms,profile_stay_time,comment_stay_time,is_profile_enter,is_rand,tab,source_file,traffic_source,source_period
0,0,1527,20220411,1900,1649675512388,0,0,0,0,0,0,0,1385,209900,0,0,0,0,1,log_standard_4_08_to_4_21_pure,standard,20220408_20220421
1,0,7405,20220416,2000,1650111976017,0,0,0,0,0,0,0,0,65400,0,0,0,0,0,log_standard_4_08_to_4_21_pure,standard,20220408_20220421
2,0,6026,20220420,1600,1650444367095,0,0,0,0,0,0,0,1405,170833,0,0,0,0,1,log_standard_4_08_to_4_21_pure,standard,20220408_20220421
3,1,6354,20220411,1100,1649645295928,0,0,0,0,0,0,0,0,255160,0,0,0,0,8,log_standard_4_08_to_4_21_pure,standard,20220408_20220421
4,1,3645,20220411,1100,1649648827559,0,0,0,0,0,0,0,1970,79733,0,0,0,0,1,log_standard_4_08_to_4_21_pure,standard,20220408_20220421


In [6]:
# Auditing duplicate behavior at full-row and event-key level

interaction_columns = list(raw_interaction_log.columns)
duplicate_key_columns = ["user_id", "video_id", "time_ms"]

interaction_audit_table = raw_interaction_log.copy()

# Hashing full rows for separating exact duplicates from conflicting records
interaction_audit_table["record_signature"] = pd.util.hash_pandas_object(
    interaction_audit_table[interaction_columns],
    index = False
)

duplicate_event_key_summary = (
    interaction_audit_table
    .groupby(duplicate_key_columns, as_index = False)
    .agg(record_count = ("user_id", "size"),
         distinct_record_count = ("record_signature", "nunique"),
         source_file_count = ("source_file", "nunique")
    )
)

duplicate_diagnostic_summary = pd.DataFrame(
    {
        "check_name": [
            "full_row_duplicate_count",
            "duplicate_user_video_time_count",
            "duplicate_keys_with_identical_records_only",
            "duplicate_keys_with_conflicting_records",
            "row_count",
            "distinct_user_count",
            "distinct_video_count"
        ],
        "value": [
            int(raw_interaction_log.duplicated().sum()),
            int(raw_interaction_log.duplicated(subset = duplicate_key_columns).sum()),
            int(((duplicate_event_key_summary["record_count"] > 1) & (duplicate_event_key_summary["distinct_record_count"] == 1)).sum()),
            int(((duplicate_event_key_summary["record_count"] > 1) & (duplicate_event_key_summary["distinct_record_count"] > 1)).sum()),
            int(len(raw_interaction_log)),
            int(raw_interaction_log["user_id"].nunique()),
            int(raw_interaction_log["video_id"].nunique())
        ]
    }
)

duplicate_diagnostic_summary

,check_name,value
0,full_row_duplicate_count,21997
1,duplicate_user_video_time_count,23483
2,duplicate_keys_with_identical_records_only,21916
3,duplicate_keys_with_conflicting_records,1486
4,row_count,2622668
5,distinct_user_count,27285
6,distinct_video_count,7583


In [7]:
# Inspecting example duplicate event keys

duplicate_event_key_examples = (
    duplicate_event_key_summary.loc[duplicate_event_key_summary["record_count"] > 1]
    .sort_values(["distinct_record_count", "record_count"], ascending = [False, False])
    .head(10)
)

duplicate_event_key_examples

,user_id,video_id,time_ms,record_count,distinct_record_count,source_file_count
2114425,22155,1957,1651937555119,5,2,1
10730,110,7044,1649597988214,3,2,1
84440,881,3740,1649774581330,3,2,1
869106,9122,3858,1649774809577,3,2,1
1171812,12242,527,1651236457849,3,2,1
1598584,16705,124,1649684535855,3,2,1
1924022,20134,1052,1649597929901,3,2,1
1725,20,7086,1649859935878,2,2,1
2239,27,728,1650642130406,2,2,1
3487,39,1052,1651362622018,2,2,1


In [8]:
# Checking date coverage and source consistency across interaction files

interaction_file_coverage_summary = (interaction_audit_table
                                     .groupby("source_file", as_index = False)
                                     .agg(row_count = ("user_id", "size"),
                                          minimum_date = ("date", "min"),
                                          maximum_date = ("date", "max"),
                                          minimum_time_ms = ("time_ms", "min"),
                                          maximum_time_ms = ("time_ms", "max"),
                                          is_rand_rate = ("is_rand", "mean")
    )
)

interaction_file_coverage_summary["is_rand_rate"] = interaction_file_coverage_summary["is_rand_rate"].round(4)
interaction_file_coverage_summary

,source_file,row_count,minimum_date,maximum_date,minimum_time_ms,maximum_time_ms,is_rand_rate
0,log_random_4_22_to_5_08_pure,1186059,20220422,20220508,1650595332831,1652025128763,1.0
1,log_standard_4_08_to_4_21_pure,1141112,20220409,20220421,1649475963278,1650556328855,0.0
2,log_standard_4_22_to_5_08_pure,295497,20220422,20220508,1650552767170,1652025130564,0.0


In [9]:
# Summarizing key integrity and missingness in dimension tables

def summarize_dimension_quality(table_name, frame, key_column):
    return {"table_name": table_name,
            "row_count": len(frame),
            "column_count": frame.shape[1],
            "distinct_key_count": frame[key_column].nunique(dropna = False),
            "duplicate_key_count": int(frame.duplicated(subset = [key_column]).sum()),
            "rows_with_any_missing": int(frame.isna().any(axis = 1).sum()),
            "total_missing_values": int(frame.isna().sum().sum())
    }

dimension_quality_summary = pd.DataFrame(
    [
        summarize_dimension_quality("user_dimension", user_dimension, "user_id"),
        summarize_dimension_quality("video_dimension_basic", video_dimension_basic, "video_id"),
        summarize_dimension_quality("video_dimension_statistic", video_dimension_statistic, "video_id")
    ]
)

dimension_quality_summary

,table_name,row_count,column_count,distinct_key_count,duplicate_key_count,rows_with_any_missing,total_missing_values
0,user_dimension,27285,31,27285,0,1563,5158
1,video_dimension_basic,7583,12,7583,0,533,538
2,video_dimension_statistic,7583,52,7583,0,0,0


In [10]:
# Measuring column-level missingness in supporting dimension tables

dimension_missingness = []

for table_name, frame in {"user_dimension": user_dimension,
                          "video_dimension_basic": video_dimension_basic,
                          "video_dimension_statistic": video_dimension_statistic
}.items():
    missing_summary = pd.DataFrame({"table_name": table_name,
                                    "column_name": frame.columns,
                                    "missing_count": frame.isna().sum().values,
                                    "missing_rate": (frame.isna().mean().values * 100).round(4)
        }
    )
    dimension_missingness.append(missing_summary)

dimension_missingness = (pd.concat(dimension_missingness, ignore_index = True)
                         .query("missing_count > 0")
                         .sort_values(["missing_rate", "table_name", "column_name"], ascending = [False, True, True])
                         .reset_index(drop = True)
)

dimension_missingness

,table_name,column_name,missing_count,missing_rate
0,user_dimension,onehot_feat4,874,3.2032
1,video_dimension_basic,video_duration,239,3.1518
2,video_dimension_basic,music_type,203,2.6770
3,user_dimension,onehot_feat12,714,2.6168
4,user_dimension,onehot_feat13,714,2.6168
5,user_dimension,onehot_feat14,714,2.6168
6,user_dimension,onehot_feat15,714,2.6168
7,user_dimension,onehot_feat16,714,2.6168
8,user_dimension,onehot_feat17,714,2.6168
9,video_dimension_basic,tag,96,1.2660


In [11]:
# Profiling outcome event frequency to confirm analytical usefulness

event_columns = ["is_click", "long_view", "is_like", "is_follow",
                 "is_comment", "is_forward", "is_hate", "is_profile_enter"
]

event_audit = pd.DataFrame(
    {
        "event_name": event_columns,
        "positive_count": [int(raw_interaction_log[column].sum()) for column in event_columns],
        "positive_rate": [(raw_interaction_log[column].mean() * 100).round(4) for column in event_columns],
        "min_value": [raw_interaction_log[column].min() for column in event_columns],
        "max_value": [raw_interaction_log[column].max() for column in event_columns]
    }
).sort_values("positive_rate", ascending = False).reset_index(drop = True)

event_audit

,event_name,positive_count,positive_rate,min_value,max_value
0,is_click,869276,33.1447,0,1
1,long_view,577493,22.0193,0,1
2,is_profile_enter,41093,1.5668,0,1
3,is_like,32237,1.2292,0,1
4,is_comment,4068,0.1551,0,1
5,is_hate,2061,0.0786,0,1
6,is_follow,1853,0.0707,0,1
7,is_forward,1784,0.0680,0,1


In [12]:
# Verifying join coverage from events to dimension tables

user_join_coverage_rate = raw_interaction_log["user_id"].isin(user_dimension["user_id"]).mean()
video_basic_join_coverage_rate = raw_interaction_log["video_id"].isin(video_dimension_basic["video_id"]).mean()
video_statistic_join_coverage_rate = raw_interaction_log["video_id"].isin(video_dimension_statistic["video_id"]).mean()

join_coverage_summary = pd.DataFrame(
    {
        "dimension_table": ["user_dimension",
                            "video_dimension_basic",
                            "video_dimension_statistic"
        ],
        "join_coverage_rate_percent": [round(user_join_coverage_rate * 100, 4),
                                       round(video_basic_join_coverage_rate * 100, 4),
                                       round(video_statistic_join_coverage_rate * 100, 4)
        ]
    }
)

join_coverage_summary

,dimension_table,join_coverage_rate_percent
0,user_dimension,100.0
1,video_dimension_basic,100.0
2,video_dimension_statistic,100.0


In [13]:
# Checking raw date, hour, play-time and duration ranges

date_time_summary = pd.DataFrame(
    {
        "metric_name": ["minimum_date", "maximum_date",
                        "minimum_hourmin", "maximum_hourmin",
                        "minimum_play_time_ms", "maximum_play_time_ms",
                        "minimum_duration_ms", "maximum_duration_ms"
        ],
        "value": [raw_interaction_log["date"].min(), raw_interaction_log["date"].max(),
                  raw_interaction_log["hourmin"].min(), raw_interaction_log["hourmin"].max(),
                  raw_interaction_log["play_time_ms"].min(), raw_interaction_log["play_time_ms"].max(),
                  raw_interaction_log["duration_ms"].min(), raw_interaction_log["duration_ms"].max()
        ]
    }
)

date_time_summary

,metric_name,value
0,minimum_date,20220409
1,maximum_date,20220508
2,minimum_hourmin,0
3,maximum_hourmin,2300
4,minimum_play_time_ms,0
5,maximum_play_time_ms,1023809
6,minimum_duration_ms,0
7,maximum_duration_ms,1177720


In [14]:
# Validating raw temporal field quality against parsed values

raw_event_date_parsed = pd.to_datetime(raw_interaction_log["date"].astype(str),
                                       format = "%Y%m%d",
                                       errors = "coerce"
)

raw_temporal_field_quality = pd.DataFrame(
    {
        "metric_name": ["minimum_raw_event_date", "maximum_raw_event_date",
                        "invalid_raw_event_date_count",
                        "minimum_raw_event_hour_bucket", "maximum_raw_event_hour_bucket",
                        "raw_event_hour_bucket_out_of_range_count", "raw_event_hour_bucket_not_hour_aligned_count",
                        "minimum_event_timestamp_utc", "maximum_event_timestamp_utc"
        ],
        "value": [int(raw_interaction_log["date"].min()), int(raw_interaction_log["date"].max()),
                  int(raw_event_date_parsed.isna().sum()), 
                  int(raw_interaction_log["hourmin"].min()), int(raw_interaction_log["hourmin"].max()),
                  int((~raw_interaction_log["hourmin"].between(0, 2300)).sum()),
                  int((raw_interaction_log["hourmin"] % 100 != 0).sum()),
                  pd.to_datetime(raw_interaction_log["time_ms"].min(), unit = "ms", utc = True),
                  pd.to_datetime(raw_interaction_log["time_ms"].max(), unit = "ms", utc = True)
        ]
    }
)

raw_temporal_field_quality

,metric_name,value
0,minimum_raw_event_date,20220409
1,maximum_raw_event_date,20220508
2,invalid_raw_event_date_count,0
3,minimum_raw_event_hour_bucket,0
4,maximum_raw_event_hour_bucket,2300
5,raw_event_hour_bucket_out_of_range_count,0
6,raw_event_hour_bucket_not_hour_aligned_count,0
7,minimum_event_timestamp_utc,2022-04-09 03:46:03.278000+00:00
8,maximum_event_timestamp_utc,2022-05-08 15:52:10.564000+00:00


In [15]:
# Removing exact duplicate rows from the raw interaction table

interaction_log_exact_deduplicated = raw_interaction_log.loc[
    ~raw_interaction_log.duplicated()
].copy()

pd.DataFrame(
    {
        "metric_name": ["row_count_before_exact_deduplication",
                        "row_count_after_exact_deduplication",
                        "exact_duplicates_removed"
        ],
        "value": [int(len(raw_interaction_log)),
                  int(len(interaction_log_exact_deduplicated)),
                  int(len(raw_interaction_log) - len(interaction_log_exact_deduplicated))
        ]
    }
)

,metric_name,value
0,row_count_before_exact_deduplication,2622668
1,row_count_after_exact_deduplication,2600671
2,exact_duplicates_removed,21997


In [16]:
# Isolating unresolved duplicate event-key groups after exact deduplication

duplicate_key_columns = ["user_id", "video_id", "time_ms"]

unresolved_duplicate_rows = interaction_log_exact_deduplicated.loc[
    interaction_log_exact_deduplicated.duplicated(subset = duplicate_key_columns, keep = False)
].copy()

remaining_duplicate_key_summary = pd.DataFrame(
    {
        "metric_name": ["rows_in_remaining_duplicate_groups", 
                        "remaining_duplicate_event_key_count"
        ],
        "value": [int(len(unresolved_duplicate_rows)),
                  int(unresolved_duplicate_rows[duplicate_key_columns]
                      .drop_duplicates()
                      .shape[0]
            )
        ]
    }
)

remaining_duplicate_key_summary

,metric_name,value
0,rows_in_remaining_duplicate_groups,2972
1,remaining_duplicate_event_key_count,1486


In [17]:
# Separating temporal-only duplicate differences from true event conflicts

non_temporal_columns = ["is_click", "is_like", "is_follow", "is_comment", "is_forward", "is_hate",
                        "long_view", "play_time_ms", "duration_ms", "profile_stay_time", "comment_stay_time",
                        "is_profile_enter", "is_rand", "tab", "traffic_source"
]

unresolved_duplicate_key_profile = (unresolved_duplicate_rows
                                    .groupby(duplicate_key_columns, as_index = False)
                                    .agg(
                                        record_count = ("user_id", "size"),
                                        distinct_raw_date_count = ("date", "nunique"),
                                        distinct_raw_hourmin_count = ("hourmin", "nunique"),
                                        **{
                                            f"distinct_{column}_count": (column, "nunique")
                                            for column in non_temporal_columns
        }
    )
)

duplicate_conflict_profile = pd.DataFrame(
    {
        "metric_name": ["remaining_duplicate_key_count_after_exact_deduplication",
                        "duplicate_keys_with_raw_date_difference",
                        "duplicate_keys_with_raw_hourmin_difference",
                        "duplicate_keys_with_non_temporal_difference"
        ],
        "value": [int(len(unresolved_duplicate_key_profile)),
                  int((unresolved_duplicate_key_profile["distinct_raw_date_count"] > 1).sum()),
                  int((unresolved_duplicate_key_profile["distinct_raw_hourmin_count"] > 1).sum()),
                  int(
                        unresolved_duplicate_key_profile[
                            [f"distinct_{column}_count" for column in non_temporal_columns]
                        ].gt(1).any(axis = 1).sum()
            )
        ]
    }
)

duplicate_conflict_profile

,metric_name,value
0,remaining_duplicate_key_count_after_exact_dedu...,1486
1,duplicate_keys_with_raw_date_difference,110
2,duplicate_keys_with_raw_hourmin_difference,1461
3,duplicate_keys_with_non_temporal_difference,27


In [18]:
# Quantifying duplicate groups that remain analytically unsafe.

unresolved_duplicate_keys = unresolved_duplicate_key_profile.loc[
    unresolved_duplicate_key_profile[
        [f"distinct_{column}_count" for column in non_temporal_columns]
    ].gt(1).any(axis = 1),
    duplicate_key_columns
].copy()

duplicate_resolution_summary = pd.DataFrame(
    {
        "metric_name": ["remaining_duplicate_key_count_after_exact_deduplication",
                        "duplicate_keys_with_only_raw_date_or_hourmin_difference",
                        "duplicate_keys_with_non_temporal_conflicts"
        ],
        "value": [int(len(unresolved_duplicate_key_profile)),
                  int(len(unresolved_duplicate_key_profile) - len(unresolved_duplicate_keys)),
                  int(len(unresolved_duplicate_keys))
        ]
    }
)

duplicate_resolution_summary

,metric_name,value
0,remaining_duplicate_key_count_after_exact_dedu...,1486
1,duplicate_keys_with_only_raw_date_or_hourmin_d...,1459
2,duplicate_keys_with_non_temporal_conflicts,27


In [19]:
# Testing whether temporal-only duplicates align under plausible UTC offsets

temporal_only_duplicate_keys = unresolved_duplicate_key_profile.loc[
    ~unresolved_duplicate_key_profile[
        [f"distinct_{column}_count" for column in non_temporal_columns]
    ].gt(1).any(axis = 1),
    duplicate_key_columns
].copy()

temporal_only_duplicate_rows = (interaction_log_exact_deduplicated
                                .merge(temporal_only_duplicate_keys, on = duplicate_key_columns, how = "inner")
                                .copy()
)

temporal_only_duplicate_rows["timestamp_utc"] = pd.to_datetime(temporal_only_duplicate_rows["time_ms"],
                                                               unit = "ms",
                                                               utc = True
)

temporal_only_duplicate_rows["raw_event_date"] = pd.to_datetime(temporal_only_duplicate_rows["date"].astype(str),
                                                                format = "%Y%m%d",
                                                                errors = "coerce"
)

temporal_only_duplicate_rows["raw_event_hour"] = (
    temporal_only_duplicate_rows["hourmin"] // 100
).astype("int16")

temporal_alignment_rows = []


# Comparing raw date and hour fields against shifted event timestamps
for utc_offset_hours in range(-12, 15):
    shifted_timestamp = (
        temporal_only_duplicate_rows["timestamp_utc"] + pd.to_timedelta(utc_offset_hours, unit = "h")
    )

    derived_event_date = pd.to_datetime(shifted_timestamp.dt.date)
    derived_event_hour = shifted_timestamp.dt.hour

    temporal_alignment_rows.append(
        {
            "utc_offset_hours": utc_offset_hours,
            "date_match_rate_percent": round(
                (derived_event_date == temporal_only_duplicate_rows["raw_event_date"]).mean() * 100,
                4
            ),
            "hour_match_rate_percent": round(
                (derived_event_hour == temporal_only_duplicate_rows["raw_event_hour"]).mean() * 100,
                4
            ),
            "date_and_hour_match_rate_percent": round(
                (
                    (derived_event_date == temporal_only_duplicate_rows["raw_event_date"])
                    & (derived_event_hour == temporal_only_duplicate_rows["raw_event_hour"])
                ).mean() * 100,
                4
            )
        }
    )

temporal_alignment_summary = (
    pd.DataFrame(temporal_alignment_rows)
    .sort_values(
        [
            "date_and_hour_match_rate_percent",
            "date_match_rate_percent",
            "hour_match_rate_percent"
        ],
        ascending = False
    )
    .reset_index(drop = True)
)

temporal_duplicate_conflict_shape = (
    temporal_only_duplicate_rows
    .groupby(duplicate_key_columns, as_index = False)
    .agg(
        raw_date_count = ("date", "nunique"),
        raw_hourmin_count = ("hourmin", "nunique")
    )
)

temporal_duplicate_validation_summary = pd.DataFrame(
    {
        "metric_name": ["temporal_only_duplicate_event_key_count",
                        "temporal_only_duplicate_row_count",
                        "keys_with_date_difference",
                        "keys_with_hourmin_difference",
                        "keys_with_both_date_and_hourmin_difference"
        ],
        "value": [int(len(temporal_only_duplicate_keys)),
                  int(len(temporal_only_duplicate_rows)),
                  int((temporal_duplicate_conflict_shape["raw_date_count"] > 1).sum()),
                  int((temporal_duplicate_conflict_shape["raw_hourmin_count"] > 1).sum()),
                  int(
                        (
                            (temporal_duplicate_conflict_shape["raw_date_count"] > 1)
                            & (temporal_duplicate_conflict_shape["raw_hourmin_count"] > 1)
                        ).sum()
                )
        ]
    }
)

display(temporal_duplicate_validation_summary)
display(temporal_alignment_summary.head(10))

,metric_name,value
0,temporal_only_duplicate_event_key_count,1459
1,temporal_only_duplicate_row_count,2918
2,keys_with_date_difference,110
3,keys_with_hourmin_difference,1459
4,keys_with_both_date_and_hourmin_difference,110


,utc_offset_hours,date_match_rate_percent,hour_match_rate_percent,date_and_hour_match_rate_percent
0,8,96.2303,49.8972,49.8972
1,9,96.2303,49.7944,49.7944
2,10,84.5099,0.3084,0.3084
3,7,93.0775,0.0000,0.0000
4,6,91.5010,0.0000,0.0000
5,5,90.1302,0.0000,0.0000
6,4,89.3763,0.0000,0.0000
7,3,88.6223,0.0000,0.0000
8,2,86.7718,0.0000,0.0000
9,1,84.0987,0.0000,0.0000


In [20]:
# Comparing duplicate handling options before final event cleaning

strict_exclusion_key_count = int(len(unresolved_duplicate_key_profile))
strict_exclusion_row_count = int(len(unresolved_duplicate_rows))

targeted_conflict_key_count = int(len(unresolved_duplicate_keys))
targeted_conflict_row_count = int(interaction_log_exact_deduplicated
                                  .merge(unresolved_duplicate_keys, on = duplicate_key_columns, how = "inner")
                                  .shape[0]
)

duplicate_strategy = pd.DataFrame(
    {
        "approach": ["strict_exclusion_of_all_unresolved_duplicate_event_keys",
                     "exclude_only_non_temporal_conflicts_and_keep_one_row_for_temporal_only_duplicates"
        ],
        "excluded_event_key_count": [strict_exclusion_key_count,
                                     targeted_conflict_key_count
        ],
        "excluded_row_count": [strict_exclusion_row_count,
                               targeted_conflict_row_count
        ],
        "retained_row_count": [int(len(interaction_log_exact_deduplicated) - strict_exclusion_row_count),
                               int(len(interaction_log_exact_deduplicated) - targeted_conflict_row_count)
        ]
    }
)

duplicate_strategy["excluded_row_rate_percent"] = (
    duplicate_strategy["excluded_row_count"] / len(interaction_log_exact_deduplicated) * 100
).round(4)

duplicate_strategy

,approach,excluded_event_key_count,excluded_row_count,retained_row_count,excluded_row_rate_percent
0,strict_exclusion_of_all_unresolved_duplicate_e...,1486,2972,2597699,0.1143
1,exclude_only_non_temporal_conflicts_and_keep_o...,27,54,2600617,0.0021


In [21]:
# Finalizing the clean interaction event base with conservative exclusions
interaction_events = (interaction_log_exact_deduplicated
                      .merge(unresolved_duplicate_key_profile[duplicate_key_columns].assign(exclude_from_analysis = 1),
                             on = duplicate_key_columns,
                             how = "left"
    )
)

interaction_events = (interaction_events.loc[interaction_events["exclude_from_analysis"].isna()]
                      .drop(columns = ["exclude_from_analysis"])
                      .copy()
)

interaction_events["event_timestamp_utc"] = pd.to_datetime(interaction_events["time_ms"],
                                                           unit = "ms",
                                                           utc = True
)

interaction_events["event_date"] = pd.to_datetime(interaction_events["date"].astype(str),
                                                  format = "%Y%m%d",
                                                  errors = "coerce"
)

interaction_events["event_date_key"] = interaction_events["date"].astype("int32")
interaction_events["event_hour_bucket"] = interaction_events["hourmin"].astype("int32")

interaction_events = (interaction_events
                      .sort_values(["event_date", "event_timestamp_utc", "user_id", "video_id", "date", "hourmin"])
                      .reset_index(drop = True)
)

pd.DataFrame(
    {
        "metric_name": ["row_count_after_exact_deduplication",
                        "excluded_temporal_only_duplicate_event_key_count",
                        "excluded_conflicting_duplicate_event_key_count",
                        "excluded_unresolved_duplicate_row_count",
                        "clean_interaction_row_count",
                        "remaining_duplicate_event_key_count"
        ],
        "value": [int(len(interaction_log_exact_deduplicated)),
                  int(len(temporal_only_duplicate_keys)),
                  int(len(unresolved_duplicate_keys)),
                  int(len(unresolved_duplicate_rows)),
                  int(len(interaction_events)),
                  int(interaction_events.duplicated(subset = duplicate_key_columns).sum())
        ]
    }
)

,metric_name,value
0,row_count_after_exact_deduplication,2600671
1,excluded_temporal_only_duplicate_event_key_count,1459
2,excluded_conflicting_duplicate_event_key_count,27
3,excluded_unresolved_duplicate_row_count,2972
4,clean_interaction_row_count,2597699
5,remaining_duplicate_event_key_count,0


In [22]:
# Creating quality flags and core derived engagement outcomes

interaction_events["has_zero_duration"] = interaction_events["duration_ms"].eq(0).astype("int8")
interaction_events["has_zero_play_time"] = interaction_events["play_time_ms"].eq(0).astype("int8")
interaction_events["has_play_time_exceeding_duration"] = (
    interaction_events["play_time_ms"] > interaction_events["duration_ms"]
).astype("int8")

interaction_events["meaningful_watch"] = interaction_events["long_view"].astype("int8")

interaction_events["high_value_engagement"] = (interaction_events["is_follow"].eq(1) 
                                               | interaction_events["is_comment"].eq(1)
                                               | interaction_events["is_forward"].eq(1)
).astype("int8")

In [23]:
# Recasting the event date field into final analytical format

interaction_events["event_date"] = pd.to_datetime(interaction_events["event_date_key"].astype(str),
                                                  format = "%Y%m%d",
                                                  errors = "coerce"
)

In [24]:
# Narrowing the clean event base to the required analytical fields

interaction_events_clean = interaction_events[["user_id", "video_id", 
                                               "time_ms", "event_timestamp_utc", "event_date_key", "event_date", "event_hour_bucket",
                                               "traffic_source", "is_rand", "tab", "is_click", "meaningful_watch", "is_like",
                                               "is_follow", "is_comment", "is_forward", "is_hate", "is_profile_enter",
                                               "play_time_ms", "duration_ms", "profile_stay_time", "comment_stay_time",
                                               "has_zero_duration", "has_zero_play_time", "has_play_time_exceeding_duration",
                                               "high_value_engagement"
    ]
].copy()

interaction_events_clean = (interaction_events_clean
                            .sort_values(["event_date", "event_timestamp_utc", "user_id", "video_id"])
                            .reset_index(drop = True)
)

In [25]:
# Checking whether downstream actions appear without click events

event_dependency_summary = pd.DataFrame(
    {
        "metric_name": ["meaningful_watch_without_click_count",
                        "profile_enter_without_click_count",
                        "like_without_click_count",
                        "follow_without_click_count",
                        "comment_without_click_count",
                        "forward_without_click_count",
                        "high_value_engagement_without_click_count"
        ],
        "value": [int(((interaction_events_clean["meaningful_watch"] == 1) & (interaction_events_clean["is_click"] == 0)).sum()),
                  int(((interaction_events_clean["is_profile_enter"] == 1) & (interaction_events_clean["is_click"] == 0)).sum()),
                  int(((interaction_events_clean["is_like"] == 1) & (interaction_events_clean["is_click"] == 0)).sum()),
                  int(((interaction_events_clean["is_follow"] == 1) & (interaction_events_clean["is_click"] == 0)).sum()),
                  int(((interaction_events_clean["is_comment"] == 1) & (interaction_events_clean["is_click"] == 0)).sum()),
                  int(((interaction_events_clean["is_forward"] == 1) & (interaction_events_clean["is_click"] == 0)).sum()),
                  int(((interaction_events_clean["high_value_engagement"] == 1) & (interaction_events_clean["is_click"] == 0)).sum())
        ]
    }
)

event_dependency_summary["rate_percent"] = (
    event_dependency_summary["value"] / len(interaction_events_clean) * 100
).round(4)

event_dependency_summary

,metric_name,value,rate_percent
0,meaningful_watch_without_click_count,2283,0.0879
1,profile_enter_without_click_count,4208,0.1620
2,like_without_click_count,6275,0.2416
3,follow_without_click_count,247,0.0095
4,comment_without_click_count,115,0.0044
5,forward_without_click_count,272,0.0105
6,high_value_engagement_without_click_count,633,0.0244


In [26]:
# Comparing key engagement rates across traffic sources

event_rate_by_traffic_source = (interaction_events_clean
                                .groupby("traffic_source", as_index = False)
                                .agg(
                                    row_count = ("user_id", "size"),
                                    click_rate_percent = ("is_click", lambda s: round(s.mean() * 100, 4)),
                                    meaningful_watch_rate_percent = ("meaningful_watch", lambda s: round(s.mean() * 100, 4)),
                                    like_rate_percent = ("is_like", lambda s: round(s.mean() * 100, 4)),
                                    follow_rate_percent = ("is_follow", lambda s: round(s.mean() * 100, 4)),
                                    comment_rate_percent = ("is_comment", lambda s: round(s.mean() * 100, 4)),
                                    forward_rate_percent = ("is_forward", lambda s: round(s.mean() * 100, 4)),
                                    high_value_engagement_rate_percent = ("high_value_engagement", lambda s: round(s.mean() * 100, 4))
    )
)

event_rate_by_traffic_source

,traffic_source,row_count,click_rate_percent,meaningful_watch_rate_percent,like_rate_percent,follow_rate_percent,comment_rate_percent,forward_rate_percent,high_value_engagement_rate_percent
0,random,1186049,17.616,8.4962,0.4798,0.0261,0.0347,0.0339,0.0942
1,standard,1411650,46.456,33.6059,1.8665,0.1070,0.2571,0.0973,0.4575


In [27]:
# Comparing playback-quality exceptions across traffic sources

playback_quality_by_traffic_source = (interaction_events_clean
                                      .groupby("traffic_source", as_index = False)
                                      .agg(
                                          row_count = ("user_id", "size"),
                                          zero_duration_rate_percent = ("has_zero_duration", lambda s: round(s.mean() * 100, 4)),
                                          zero_play_time_rate_percent = ("has_zero_play_time", lambda s: round(s.mean() * 100, 4)),
                                          play_time_exceeds_duration_rate_percent = ("has_play_time_exceeding_duration", lambda s: round(s.mean() * 100, 4))
    )
)

playback_quality_by_traffic_source

,traffic_source,row_count,zero_duration_rate_percent,zero_play_time_rate_percent,play_time_exceeds_duration_rate_percent
0,random,1186049,3.1129,0.6672,6.3101
1,standard,1411650,2.0443,13.3686,16.8289


In [28]:
# Reviewing basic video fields for completeness and analytical value

video_basic_field_review = pd.DataFrame(
    {
        "column_name": video_dimension_basic.columns,
        "missing_count": video_dimension_basic.isna().sum().values,
        "missing_rate_percent": (video_dimension_basic.isna().mean().values * 100).round(4),
        "distinct_value_count": [video_dimension_basic[column].nunique(dropna = False) for column in video_dimension_basic.columns],
        "sample_value": [video_dimension_basic[column].dropna().iloc[0]
                         if video_dimension_basic[column].notna().any()
                         else np.nan
                         for column in video_dimension_basic.columns
        ]
    }
).sort_values(["missing_rate_percent", "column_name"], ascending = [False, True]).reset_index(drop = True)

video_basic_field_review

,column_name,missing_count,missing_rate_percent,distinct_value_count,sample_value
0,video_duration,239,3.1518,5757,87433.0
1,music_type,203,2.6770,6,9.0
2,tag,96,1.2660,111,39
3,author_id,0,0.0000,6510,7349781
4,music_id,0,0.0000,7202,9155697141
5,server_height,0,0.0000,120,1280.0
6,server_width,0,0.0000,156,720.0
7,upload_dt,0,0.0000,3,2022-04-10
8,upload_type,0,0.0000,14,LongImport
9,video_id,0,0.0000,7583,0


In [29]:
# Reviewing user fields for completeness and segment usefulness

user_field_review = pd.DataFrame(
    {
        "column_name": user_dimension.columns,
        "missing_count": user_dimension.isna().sum().values,
        "missing_rate_percent": (user_dimension.isna().mean().values * 100).round(4),
        "distinct_value_count": [user_dimension[column].nunique(dropna = False) for column in user_dimension.columns],
        "sample_value": [user_dimension[column].dropna().iloc[0]
                         if user_dimension[column].notna().any()
                         else np.nan
                         for column in user_dimension.columns
        ]
    }
).sort_values(["missing_rate_percent", "column_name"], ascending = [False, True]).reset_index(drop = True)

user_field_review

,column_name,missing_count,missing_rate_percent,distinct_value_count,sample_value
0,onehot_feat4,874,3.2032,16,1.0
1,onehot_feat12,714,2.6168,3,0.0
2,onehot_feat13,714,2.6168,3,1.0
3,onehot_feat14,714,2.6168,3,0.0
4,onehot_feat15,714,2.6168,3,0.0
5,onehot_feat16,714,2.6168,3,0.0
6,onehot_feat17,714,2.6168,3,0.0
7,fans_user_num,0,0.0000,2210,150
8,fans_user_num_range,0,0.0000,9,"[100,1k)"
9,follow_user_num,0,0.0000,2562,514


In [30]:
# Reviewing aggregate video statistics before deciding whether to exclude them

video_statistic_field_review = pd.DataFrame(
    {
        "column_name": video_dimension_statistic.columns,
        "missing_count": video_dimension_statistic.isna().sum().values,
        "missing_rate_percent": (video_dimension_statistic.isna().mean().values * 100).round(4),
        "distinct_value_count": [video_dimension_statistic[column].nunique(dropna = False)
                                 for column in video_dimension_statistic.columns
        ],
        "sample_value": [video_dimension_statistic[column].dropna().iloc[0]
                         if video_dimension_statistic[column].notna().any()
                         else np.nan
                         for column in video_dimension_statistic.columns
        ]
    }
).sort_values(["missing_rate_percent", "column_name"], ascending = [False, True]).reset_index(drop = True)

video_statistic_field_review

,column_name,missing_count,missing_rate_percent,distinct_value_count,sample_value
0,cancel_collect_cnt,0,0.0,6680,2.352941e-01
1,cancel_collect_user_num,0,0.0,6599,2.352941e-01
2,cancel_follow_cnt,0,0.0,5999,5.196078e-01
3,cancel_follow_user_num,0,0.0,5968,5.098039e-01
4,cancel_like_cnt,0,0.0,7136,6.960784e-01
5,cancel_like_user_num,0,0.0,7107,6.568627e-01
6,click_like_cnt,0,0.0,7488,2.921569e+00
7,collect_cnt,0,0.0,7290,9.019608e-01
8,collect_user_num,0,0.0,7274,8.823529e-01
9,comment_cnt,0,0.0,6906,1.960784e-01


In [31]:
# Renaming and narrowing the cleaned event table for impression-level analysis

impression_events = (
    interaction_events_clean
    .rename(
        columns = {
            "time_ms": "event_timestamp_ms",
            "tab": "feed_tab_id",
            "meaningful_watch": "is_meaningful_watch",
            "high_value_engagement": "is_high_value_engagement"
        }
    )
    .drop(columns = ["is_rand"])
    .copy()
)

impression_events = impression_events[
    [
        "user_id", "video_id", "event_timestamp_ms", "event_timestamp_utc", "event_date_key", "event_date", "event_hour_bucket",
        "traffic_source", "feed_tab_id",
        "is_click", "is_meaningful_watch", "is_high_value_engagement", "is_like", "is_follow", "is_comment", "is_forward", "is_hate",
        "is_profile_enter", "play_time_ms", "duration_ms", "profile_stay_time", "comment_stay_time",
        "has_zero_duration", "has_zero_play_time", "has_play_time_exceeding_duration"
    ]
].copy()

In [32]:
# Joining selected user and video attributes to the impression base for business analysis

user_columns = ["user_id", "user_active_degree", "is_video_author", 
                "follow_user_num", "follow_user_num_range", "fans_user_num", "fans_user_num_range",
                "friend_user_num", "friend_user_num_range", "register_days", "register_days_range"
]

video_columns = ["video_id", "author_id", "video_type", "upload_dt", "upload_type", "video_duration",
                 "server_width", "server_height", "music_id", "music_type", "tag"
]

# Standardizing the user attributes for consistent downstream grouping
user_attributes = user_dimension[user_columns].copy()

for column_name in ["user_active_degree", "follow_user_num_range", "fans_user_num_range",
                    "friend_user_num_range", "register_days_range"
]:
    user_attributes[column_name] = (user_attributes[column_name]
                                    .astype("string")
                                    .str.strip()
                                    .replace({"": pd.NA})
    )

# Standardizing the video attributes for business-ready field names
video_attributes = (video_dimension_basic[video_columns]
                    .rename(
                        columns = {"author_id": "creator_user_id",
                                   "upload_dt": "video_upload_date",
                                   "upload_type": "video_upload_type",
                                   "video_duration": "video_duration_ms",
                                   "server_width": "video_width",
                                   "server_height": "video_height",
                                   "tag": "video_tag"
        }
    )
    .copy()
)

video_attributes["video_upload_date"] = pd.to_datetime(video_attributes["video_upload_date"],
                                                       errors = "coerce"
)

for column_name in ["video_type", "video_upload_type", "video_tag"]:
    video_attributes[column_name] = (video_attributes[column_name]
                                     .astype("string")
                                     .str.strip()
                                     .replace({"": pd.NA})
    )

# Building the main impression fact table at the analytical grain
impression_fact = (impression_events
                   .merge(user_attributes, on = "user_id", how = "left")
                   .merge(video_attributes, on = "video_id", how = "left")
                   .rename(columns = {"duration_ms": "event_content_duration_ms"})
                   .copy()
)


# Deriving reporting fields that are usable for segmentation and interpretation
impression_fact["event_hour_of_day"] = (
    impression_fact["event_hour_bucket"] // 100
).astype("int8")

impression_fact["event_weekday_name"] = (
    impression_fact["event_date"]
    .dt.day_name()
    .astype("string")
)

impression_fact["is_weekend"] = (
    impression_fact["event_date"]
    .dt.dayofweek
    .isin([5, 6])
    .astype("int8")
)

impression_fact["video_age_days"] = (
    impression_fact["event_date"] - impression_fact["video_upload_date"]
).dt.days

impression_fact["video_orientation"] = np.select(
    [
        impression_fact["video_height"] > impression_fact["video_width"],
        impression_fact["video_width"] > impression_fact["video_height"],
        impression_fact["video_width"] == impression_fact["video_height"]
    ],
    ["vertical", "horizontal", "square"],
    default = "unknown"
)

impression_fact["playback_completion_rate"] = np.where(
    impression_fact["event_content_duration_ms"] > 0,
    impression_fact["play_time_ms"] / impression_fact["event_content_duration_ms"],
    np.nan
)

impression_fact["playback_completion_rate_capped"] = (
    impression_fact["playback_completion_rate"]
    .clip(lower = 0, upper = 1)
)

impression_fact["is_social_engagement"] = (
    impression_fact[["is_like", "is_follow", "is_comment", "is_forward"]]
    .max(axis = 1)
    .astype("int8")
)

impression_fact = (
    impression_fact[
        [
            "user_id", "video_id",
            "event_timestamp_ms", "event_timestamp_utc", "event_date_key", "event_date",
            "event_hour_bucket", "event_hour_of_day", "event_weekday_name", "is_weekend", "traffic_source", "feed_tab_id",
            "is_click", "is_meaningful_watch", "is_social_engagement", "is_high_value_engagement", "is_like", "is_follow", "is_comment",
            "is_forward", "is_hate", "is_profile_enter", "play_time_ms", "event_content_duration_ms", "video_duration_ms", 
            "playback_completion_rate", "playback_completion_rate_capped", "profile_stay_time", "comment_stay_time",
            "has_zero_duration", "has_zero_play_time", "has_play_time_exceeding_duration",
            "user_active_degree", "is_video_author", "follow_user_num", "follow_user_num_range",
            "fans_user_num", "fans_user_num_range", "friend_user_num", "friend_user_num_range", "register_days",
            "register_days_range", "creator_user_id", "video_type", "video_upload_date", "video_upload_type", "video_width",
            "video_height", "video_orientation", "music_id", "music_type", "video_tag", "video_age_days"
        ]
    ]
    .sort_values(["event_date", "event_timestamp_ms", "user_id", "video_id"])
    .reset_index(drop = True)
)

print(impression_fact.shape)
impression_fact.head()

(2597699, 53)


,user_id,video_id,event_timestamp_ms,event_timestamp_utc,event_date_key,event_date,event_hour_bucket,event_hour_of_day,event_weekday_name,is_weekend,traffic_source,feed_tab_id,is_click,is_meaningful_watch,is_social_engagement,is_high_value_engagement,is_like,is_follow,is_comment,is_forward,is_hate,is_profile_enter,play_time_ms,event_content_duration_ms,video_duration_ms,playback_completion_rate,playback_completion_rate_capped,profile_stay_time,comment_stay_time,has_zero_duration,has_zero_play_time,has_play_time_exceeding_duration,user_active_degree,is_video_author,follow_user_num,follow_user_num_range,fans_user_num,fans_user_num_range,friend_user_num,friend_user_num_range,register_days,register_days_range,creator_user_id,video_type,video_upload_date,video_upload_type,video_width,video_height,video_orientation,music_id,music_type,video_tag,video_age_days
0,20853,222,1649475963278,2022-04-09 03:46:03.278000+00:00,20220409,2022-04-09,1200,12,Saturday,1,standard,1,0,0,0,0,0,0,0,0,0,0,1286,289866,289866.0,0.004437,0.004437,0,0,0,0,0,full_active,1,78,"(50,100]",66,"[10,100)",0,0,2479,730+,6705448,NORMAL,2022-04-09,Web,720.0,1280.0,vertical,9138394916,9.0,3,0
1,8286,222,1649476012164,2022-04-09 03:46:52.164000+00:00,20220409,2022-04-09,1200,12,Saturday,1,standard,1,0,0,0,0,0,0,0,0,0,0,1857,289866,289866.0,0.006406,0.006406,0,0,0,0,0,2_14_day_new,1,64,"(50,100]",47,"[10,100)",36,"[30,60)",1420,730+,6705448,NORMAL,2022-04-09,Web,720.0,1280.0,vertical,9138394916,9.0,3,0
2,7406,222,1649476421916,2022-04-09 03:53:41.916000+00:00,20220409,2022-04-09,1200,12,Saturday,1,standard,4,1,1,0,0,0,0,0,0,0,1,25750,289866,289866.0,0.088834,0.088834,0,0,0,0,0,full_active,1,32,"(10,50]",53,"[10,100)",32,"[30,60)",102,91-180,6705448,NORMAL,2022-04-09,Web,720.0,1280.0,vertical,9138394916,9.0,3,0
3,25934,6580,1649476523839,2022-04-09 03:55:23.839000+00:00,20220409,2022-04-09,1200,12,Saturday,1,standard,0,0,0,0,0,0,0,0,0,0,0,0,55960,55960.0,0.000000,0.000000,0,0,0,1,0,full_active,0,78,"(50,100]",0,0,0,0,89,61-90,4854468,NORMAL,2022-04-09,LongImport,720.0,1280.0,vertical,9138293546,9.0,"12,60",0
4,2073,4566,1649476630340,2022-04-09 03:57:10.340000+00:00,20220409,2022-04-09,1200,12,Saturday,1,standard,2,0,1,0,0,0,0,0,0,0,0,14904,14700,14700.0,1.013878,1.000000,0,1641,0,0,1,high_active,1,153,"(150,250]",11,"[10,100)",1,"[1,5)",1711,730+,5597283,NORMAL,2022-04-09,Web,720.0,1280.0,vertical,5128196645,4.0,20,0


In [33]:
# Rebuilding the final video attribute view for later tag and content analysis

video_attributes = (
    video_dimension_basic[
        [
            "video_id", "author_id", "video_type", "upload_dt", "upload_type",
            "video_duration", "server_width", "server_height", "music_id", "music_type", "tag"
        ]
    ]
    .rename(
        columns = {"author_id": "creator_user_id",
                   "upload_dt": "video_upload_date",
                   "upload_type": "video_upload_type",
                   "video_duration": "video_duration_ms",
                   "server_width": "video_width",
                   "server_height": "video_height",
                   "tag": "video_tag"
        }
    )
    .copy()
)

video_attributes["video_upload_date"] = pd.to_datetime(
    video_attributes["video_upload_date"],
    errors = "coerce"
)

video_attributes["video_tag"] = (
    video_attributes["video_tag"]
    .astype("string")
    .str.strip()
    .replace({"": pd.NA})
)

In [34]:
# Checking how well raw event dates align with the timestamp field

event_date_reference = impression_fact["event_date"]
event_hour_reference = impression_fact["event_hour_bucket"].floordiv(100)

alignment_rows = []

for utc_offset_hours in range(-12, 15):
    shifted_timestamp = (
        impression_fact["event_timestamp_utc"] + pd.to_timedelta(utc_offset_hours, unit = "h")
    )

    same_date = shifted_timestamp.dt.date == event_date_reference.dt.date
    same_hour = shifted_timestamp.dt.hour == event_hour_reference

    alignment_rows.append(
        {
            "utc_offset_hours": utc_offset_hours,
            "event_date_match_rate_percent": round(same_date.mean() * 100, 4),
            "event_hour_match_rate_percent": round(same_hour.mean() * 100, 4),
            "event_date_and_hour_match_rate_percent": round((same_date & same_hour).mean() * 100, 4)
        }
    )

timestamp_alignment_summary = (
    pd.DataFrame(alignment_rows)
    .sort_values(
        [
            "event_date_and_hour_match_rate_percent",
            "event_date_match_rate_percent",
            "event_hour_match_rate_percent"
        ],
        ascending = False
    )
    .reset_index(drop = True)
)

timestamp_alignment_by_traffic_source = []

for traffic_source, source_frame in impression_fact.groupby("traffic_source"):
    source_event_date_reference = source_frame["event_date"]
    source_event_hour_reference = source_frame["event_hour_bucket"].floordiv(100)

    for utc_offset_hours in range(-12, 15):
        shifted_timestamp = (
            source_frame["event_timestamp_utc"] + pd.to_timedelta(utc_offset_hours, unit = "h")
        )

        same_date = shifted_timestamp.dt.date == source_event_date_reference.dt.date
        same_hour = shifted_timestamp.dt.hour == source_event_hour_reference

        timestamp_alignment_by_traffic_source.append(
            {
                "traffic_source": traffic_source,
                "utc_offset_hours": utc_offset_hours,
                "event_date_match_rate_percent": round(same_date.mean() * 100, 4),
                "event_hour_match_rate_percent": round(same_hour.mean() * 100, 4),
                "event_date_and_hour_match_rate_percent": round((same_date & same_hour).mean() * 100, 4)
            }
        )

timestamp_alignment_by_traffic_source = (
    pd.DataFrame(timestamp_alignment_by_traffic_source)
    .sort_values(
        ["traffic_source", "event_date_and_hour_match_rate_percent"],
        ascending = [True, False]
    )
    .groupby("traffic_source", as_index = False)
    .head(5)
    .reset_index(drop = True)
)

display(timestamp_alignment_summary.head(10))
display(timestamp_alignment_by_traffic_source)

,utc_offset_hours,event_date_match_rate_percent,event_hour_match_rate_percent,event_date_and_hour_match_rate_percent
0,8,99.2199,82.6684,82.6684
1,9,95.4601,17.3201,17.3201
2,10,88.5976,0.0107,0.0107
3,11,81.5587,0.0008,0.0008
4,7,95.9592,0.0000,0.0000
5,6,93.9913,0.0000,0.0000
6,5,92.7199,0.0000,0.0000
7,4,91.8476,0.0000,0.0000
8,3,91.0904,0.0000,0.0000
9,2,90.0086,0.0000,0.0000


,traffic_source,utc_offset_hours,event_date_match_rate_percent,event_hour_match_rate_percent,event_date_and_hour_match_rate_percent
0,random,8,99.2727,83.0680,83.0680
1,random,9,95.7302,16.9281,16.9281
2,random,10,89.3626,0.0026,0.0026
3,random,11,82.9107,0.0013,0.0013
4,random,-12,23.4322,0.0000,0.0000
5,standard,8,99.1755,82.3327,82.3327
6,standard,9,95.2332,17.6493,17.6493
7,standard,10,87.9549,0.0175,0.0175
8,standard,11,80.4228,0.0004,0.0004
9,standard,-12,26.3259,0.0000,0.0000


In [35]:
# Profiling playback anomalies to understand watch-quality edge cases

playback_exception_summary = (
    impression_fact
    .assign(
        click_status = np.where(impression_fact["is_click"].eq(1), "clicked", "not_clicked")
    )
    .groupby(["traffic_source", "click_status"], as_index = False)
    .agg(
        impression_count = ("user_id", "size"),
        zero_duration_count = ("has_zero_duration", "sum"),
        zero_play_time_count = ("has_zero_play_time", "sum"),
        play_time_exceeds_duration_count = ("has_play_time_exceeding_duration", "sum"),
        median_play_time_ms = ("play_time_ms", "median"),
        median_event_content_duration_ms = ("event_content_duration_ms", "median"),
        median_playback_completion_rate = ("playback_completion_rate_capped", "median")
    )
)

playback_exception_summary["zero_duration_rate_percent"] = (
    playback_exception_summary["zero_duration_count"] / playback_exception_summary["impression_count"] * 100
).round(4)

playback_exception_summary["zero_play_time_rate_percent"] = (
    playback_exception_summary["zero_play_time_count"] / playback_exception_summary["impression_count"] * 100
).round(4)

playback_exception_summary["play_time_exceeds_duration_rate_percent"] = (
    playback_exception_summary["play_time_exceeds_duration_count"] / playback_exception_summary["impression_count"] * 100
).round(4)

playback_exception_summary = playback_exception_summary[
    [
        "traffic_source", "click_status", "impression_count", "zero_duration_count", "zero_duration_rate_percent", "zero_play_time_count", 
        "zero_play_time_rate_percent", "play_time_exceeds_duration_count", "play_time_exceeds_duration_rate_percent",
        "median_play_time_ms", "median_event_content_duration_ms", "median_playback_completion_rate"
    ]
]

playback_exception_summary

,traffic_source,click_status,impression_count,zero_duration_count,zero_duration_rate_percent,zero_play_time_count,zero_play_time_rate_percent,play_time_exceeds_duration_count,play_time_exceeds_duration_rate_percent,median_play_time_ms,median_event_content_duration_ms,median_playback_completion_rate
0,random,clicked,208934,2792,1.3363,376,0.1800,40844,19.5488,14982.0,76400.0,0.299041
1,random,not_clicked,977115,34128,3.4927,7537,0.7714,33997,3.4793,1754.0,76920.0,0.024861
2,standard,clicked,655796,6511,0.9928,5056,0.7710,219385,33.4532,27682.0,72266.0,0.698498
3,standard,not_clicked,755854,22347,2.9565,183662,24.2986,18180,2.4052,1645.0,69033.0,0.020530


In [36]:
# Checking how much creator profile coverage is available from the user table

creator_user_ids = (video_attributes["creator_user_id"]
                    .dropna()
                    .drop_duplicates()
)

creator_profile_coverage = pd.DataFrame(
    {
        "metric_name": ["distinct_creator_count",
                        "distinct_creators_present_in_user_dimension",
                        "distinct_creators_missing_from_user_dimension",
                        "distinct_creator_coverage_rate_percent",
                        "impression_rows_with_creator_present_in_user_dimension",
                        "impression_row_creator_coverage_rate_percent"
        ],
        "value": [int(creator_user_ids.nunique()),
                  int(creator_user_ids.isin(user_dimension["user_id"]).sum()),
                  int((~creator_user_ids.isin(user_dimension["user_id"])).sum()),
                  round(creator_user_ids.isin(user_dimension["user_id"]).mean() * 100, 4),
                  int(impression_fact["creator_user_id"].isin(user_dimension["user_id"]).sum()),
                  round(impression_fact["creator_user_id"].isin(user_dimension["user_id"]).mean() * 100, 4)
        ]
    }
)

creator_profile_coverage

,metric_name,value
0,distinct_creator_count,6510.0000
1,distinct_creators_present_in_user_dimension,104.0000
2,distinct_creators_missing_from_user_dimension,6406.0000
3,distinct_creator_coverage_rate_percent,1.5975
4,impression_rows_with_creator_present_in_user_d...,51663.0000
5,impression_row_creator_coverage_rate_percent,1.9888


In [37]:
# Profiling the final fact fields to confirm readiness and data coverage

final_field_profile = pd.DataFrame(
    {
        "field_name": impression_fact.columns,
        "data_type": impression_fact.dtypes.astype(str).values,
        "missing_count": impression_fact.isna().sum().values,
        "missing_rate_percent": (impression_fact.isna().mean().values * 100).round(4),
        "distinct_value_count": [impression_fact[column].nunique(dropna = False)
                                 for column in impression_fact.columns
        ],
        "sample_value": [impression_fact[column].dropna().iloc[0]
                         if impression_fact[column].notna().any()
                         else np.nan
                         for column in impression_fact.columns
        ]
    }
).sort_values(["missing_rate_percent", "field_name"],
              ascending = [False, True]
).reset_index(drop = True)


final_field_profile

,field_name,data_type,missing_count,missing_rate_percent,distinct_value_count,sample_value
0,music_type,float64,79102,3.0451,6,9.0
1,playback_completion_rate,float64,65778,2.5322,2171493,0.004437
2,playback_completion_rate_capped,float64,65778,2.5322,1933750,0.004437
3,video_duration_ms,float64,65778,2.5322,5757,289866.0
4,video_tag,string,27593,1.0622,111,3
5,comment_stay_time,int32,0,0.0000,24967,0
6,creator_user_id,int64,0,0.0000,6510,6705448
7,event_content_duration_ms,int32,0,0.0000,5757,289866
8,event_date,datetime64[us],0,0.0000,30,2022-04-09 00:00:00
9,event_date_key,int32,0,0.0000,30,20220409


In [38]:
# Reviewing the main categorical fields for reporting usefulness

business_categorical_fields = ["traffic_source", "feed_tab_id", "user_active_degree", "follow_user_num_range",
                               "fans_user_num_range", "friend_user_num_range", "register_days_range", "video_type",
                               "video_upload_type", "music_type"
]

categorical_field_summary = pd.DataFrame(
    {
        "field_name": business_categorical_fields,
        "distinct_value_count": [impression_fact[field].nunique(dropna = False)
                                 for field in business_categorical_fields
        ],
        "missing_count": [int(impression_fact[field].isna().sum())
                          for field in business_categorical_fields
        ],
        "top_value": [impression_fact[field].value_counts(dropna = False).index[0]
                      for field in business_categorical_fields
        ],
        "top_value_count": [int(impression_fact[field].value_counts(dropna = False).iloc[0])
                            for field in business_categorical_fields
        ]
    }
)

categorical_field_summary

,field_name,distinct_value_count,missing_count,top_value,top_value_count
0,traffic_source,2,0,standard,1411650
1,feed_tab_id,15,0,1,2236411
2,user_active_degree,9,0,full_active,1889845
3,follow_user_num_range,8,0,500+,537421
4,fans_user_num_range,9,0,"[10,100)",991288
5,friend_user_num_range,7,0,"[5,30)",726049
6,register_days_range,8,0,730+,1909229
7,video_type,3,0,NORMAL,2568173
8,video_upload_type,14,0,LongImport,1070378
9,music_type,6,79102,9.0,2292634


In [39]:
# Inspecting the video tag structure before splitting tags for analysis

video_tag_profile = video_attributes[["video_id", "video_tag"]].copy()

video_tag_profile["tag_count"] = np.where(
    video_tag_profile["video_tag"].notna(),
    video_tag_profile["video_tag"].astype("string").str.count(",") + 1,
    np.nan
)

video_tag_structure_summary = pd.DataFrame(
    {
        "metric_name": ["video_count", "videos_with_missing_tag", "videos_with_single_tag",
                        "videos_with_multiple_tags", "maximum_tag_count_per_video"
        ],
        "value": [int(len(video_tag_profile)), 
                  int(video_tag_profile["video_tag"].isna().sum()),
                  int((video_tag_profile["tag_count"] == 1).sum()),
                  int((video_tag_profile["tag_count"] > 1).sum()),
                  int(video_tag_profile["tag_count"].max())
        ]
    }
)

exploded_video_tags = (video_tag_profile.loc[video_tag_profile["video_tag"].notna(), "video_tag"]
                       .astype("string")
                       .str.split(",")
                       .explode()
                       .str.strip()
)

top_video_tags = (exploded_video_tags.value_counts()
                  .head(20)
                  .rename_axis("video_tag")
                  .reset_index(name = "video_count")
)

video_tag_structure_summary

,metric_name,value
0,video_count,7583
1,videos_with_missing_tag,96
2,videos_with_single_tag,5696
3,videos_with_multiple_tags,1791
4,maximum_tag_count_per_video,3


In [40]:
# Surfacing the most common tags for later content-level reporting
top_video_tags

,video_tag,video_count
0,39,1446
1,3,732
2,20,675
3,12,606
4,62,489
5,43,428
6,68,428
7,9,419
8,6,400
9,7,383


In [41]:
# Exposing the truly conflicting duplicates as evidence for the exclusion policy

conflicting_duplicate_rows = (interaction_log_exact_deduplicated
                              .merge(unresolved_duplicate_keys, on = duplicate_key_columns, how = "inner")
                              .sort_values(duplicate_key_columns + ["date", "hourmin", "source_file"])
                              .reset_index(drop = True)
)

conflicting_duplicate_rows[
    [
        "user_id", "video_id", "time_ms", "date", "hourmin",
        "is_click", "is_like", "is_follow", "is_comment", "is_forward", "is_hate",
        "long_view", "play_time_ms", "duration_ms", "profile_stay_time", "comment_stay_time", 
        "is_profile_enter", "is_rand", "tab", "source_file", "traffic_source", "source_period"
    ]
]

,user_id,video_id,time_ms,date,hourmin,is_click,is_like,is_follow,is_comment,is_forward,is_hate,long_view,play_time_ms,duration_ms,profile_stay_time,comment_stay_time,is_profile_enter,is_rand,tab,source_file,traffic_source,source_period
0,1631,2329,1649908983373,20220414,1200,0,0,0,0,0,0,0,787,78733,0,0,0,0,6,log_standard_4_08_to_4_21_pure,standard,20220408_20220421
1,1631,2329,1649908983373,20220414,1200,0,0,0,0,0,0,0,4960,78733,0,0,0,0,6,log_standard_4_08_to_4_21_pure,standard,20220408_20220421
2,1756,4745,1651310345755,20220430,1700,1,0,0,0,0,0,1,48585,66700,0,0,0,0,5,log_standard_4_22_to_5_08_pure,standard,20220422_20220508
3,1756,4745,1651310345755,20220430,1700,1,0,0,0,0,0,1,18726,66700,0,0,0,0,5,log_standard_4_22_to_5_08_pure,standard,20220422_20220508
4,2037,527,1649905720525,20220414,1100,0,0,0,0,0,0,0,854,10388,0,0,0,0,6,log_standard_4_08_to_4_21_pure,standard,20220408_20220421
5,2037,527,1649905720525,20220414,1100,0,0,0,0,0,0,0,1588,10388,0,0,0,0,6,log_standard_4_08_to_4_21_pure,standard,20220408_20220421
6,2116,5319,1649512329194,20220409,2200,1,0,0,0,0,0,1,55927,7720,0,0,0,0,6,log_standard_4_08_to_4_21_pure,standard,20220408_20220421
7,2116,5319,1649512329194,20220409,2200,1,0,0,0,0,0,1,55927,7720,0,47537,0,0,6,log_standard_4_08_to_4_21_pure,standard,20220408_20220421
8,2373,2055,1649660927424,20220411,1500,1,0,0,0,0,0,0,1182,15800,0,0,0,0,6,log_standard_4_08_to_4_21_pure,standard,20220408_20220421
9,2373,2055,1649660927424,20220411,1500,1,0,0,0,0,0,0,12451,15800,0,0,0,0,6,log_standard_4_08_to_4_21_pure,standard,20220408_20220421


In [42]:
# Summarizing whether the current fact table is ready for analysis use

creator_profile_coverage_rate = (impression_fact["creator_user_id"]
                                 .isin(user_dimension["user_id"])
                                 .mean()
)

dataset_readiness = pd.DataFrame(
    {
        "metric_name": ["row_count", "column_count", 
                        "duplicate_event_key_count", "resolved_temporal_only_duplicate_event_key_count",
                        "excluded_conflicting_duplicate_event_key_count", "excluded_conflicting_duplicate_row_count",
                        "missing_user_attribute_count", "missing_creator_user_id_count", "missing_event_content_duration_count",
                        "missing_video_duration_count", "missing_music_type_count", "missing_video_tag_count",
                        "negative_video_age_count", "creator_profile_coverage_rate_percent", "click_rate_percent",
                        "meaningful_watch_rate_percent", "social_engagement_rate_percent", "high_value_engagement_rate_percent"
        ],
        "value": [int(len(impression_fact)), 
                  int(impression_fact.shape[1]),
                  int(impression_fact.duplicated(subset = ["user_id", "video_id", "event_timestamp_ms"]).sum()),
                  int(len(temporal_only_duplicate_keys)),
                  int(len(unresolved_duplicate_keys)),
                  int(interaction_log_exact_deduplicated
                      .merge(unresolved_duplicate_keys, on = duplicate_key_columns, how = "inner")
                      .shape[0]),
                  int(impression_fact["user_active_degree"].isna().sum()),
                  int(impression_fact["creator_user_id"].isna().sum()),
                  int(impression_fact["event_content_duration_ms"].isna().sum()),
                  int(impression_fact["video_duration_ms"].isna().sum()),
                  int(impression_fact["music_type"].isna().sum()),
                  int(impression_fact["video_tag"].isna().sum()),
                  int((impression_fact["video_age_days"] < 0).fillna(False).sum()),
                  round(creator_profile_coverage_rate * 100, 4),
                  round(impression_fact["is_click"].mean() * 100, 4),
                  round(impression_fact["is_meaningful_watch"].mean() * 100, 4),
                  round(impression_fact["is_social_engagement"].mean() * 100, 4),
                  round(impression_fact["is_high_value_engagement"].mean() * 100, 4)
        ]
    }
)

field_missingness = pd.DataFrame(
    {
        "field_name": impression_fact.columns,
        "data_type": impression_fact.dtypes.astype(str).values,
        "missing_count": impression_fact.isna().sum().values,
        "missing_rate_percent": (impression_fact.isna().mean().values * 100).round(4)
    }
)

field_missingness = (field_missingness
                     .query("missing_count > 0")
                     .sort_values(["missing_rate_percent", "field_name"], ascending = [False, True])
                     .reset_index(drop = True)
)

display(dataset_readiness)
display(field_missingness)

,metric_name,value
0,row_count,2.597699e+06
1,column_count,5.300000e+01
2,duplicate_event_key_count,0.000000e+00
3,resolved_temporal_only_duplicate_event_key_count,1.459000e+03
4,excluded_conflicting_duplicate_event_key_count,2.700000e+01
5,excluded_conflicting_duplicate_row_count,5.400000e+01
6,missing_user_attribute_count,0.000000e+00
7,missing_creator_user_id_count,0.000000e+00
8,missing_event_content_duration_count,0.000000e+00
9,missing_video_duration_count,6.577800e+04


,field_name,data_type,missing_count,missing_rate_percent
0,music_type,float64,79102,3.0451
1,playback_completion_rate,float64,65778,2.5322
2,playback_completion_rate_capped,float64,65778,2.5322
3,video_duration_ms,float64,65778,2.5322
4,video_tag,string,27593,1.0622


In [43]:
# Saving the full curated impression fact table for reproducible downstream use

upload_dataframe_to_parquet_blob(impression_fact,
                                 container_names["curated"],
                                 "impression_fact.parquet"
)

pd.DataFrame(
    {
        "dataset_name": ["impression_fact"],
        "row_count": [len(impression_fact)],
        "column_count": [impression_fact.shape[1]]
    }
)

,dataset_name,row_count,column_count
0,impression_fact,2597699,53


In [44]:
# Verifying that the attribute joins are not changing the event grain

pre_join_row_count = len(impression_events)

post_join_row_count = len(impression_fact)

join_validation_summary = pd.DataFrame(
    {
        "metric_name": ["pre_join_row_count",
                        "post_join_row_count",
                        "row_count_difference",
                        "duplicate_event_key_count_after_join"
        ],
        "value": [int(pre_join_row_count),
                  int(post_join_row_count),
                  int(post_join_row_count - pre_join_row_count),
                  int(impression_fact.duplicated(
                        subset = ["user_id", "video_id", "event_timestamp_ms"]
                    ).sum()
                )
        ]
    }
)

binary_flag_columns = ["is_click", "is_meaningful_watch", "is_social_engagement", "is_high_value_engagement", "is_like",
                       "is_follow", "is_comment", "is_forward", "is_hate", "is_profile_enter", 
                       "has_zero_duration", "has_zero_play_time", "has_play_time_exceeding_duration", "is_weekend"
]

binary_flag_validation = pd.DataFrame(
    {
        "field_name": binary_flag_columns,
        "invalid_value_count": [
            int((~impression_fact[column].isin([0, 1])).sum())
            for column in binary_flag_columns
        ]
    }
)

display(join_validation_summary)
display(binary_flag_validation.sort_values("invalid_value_count", ascending = False))

,metric_name,value
0,pre_join_row_count,2597699
1,post_join_row_count,2597699
2,row_count_difference,0
3,duplicate_event_key_count_after_join,0


,field_name,invalid_value_count
0,is_click,0
1,is_meaningful_watch,0
2,is_social_engagement,0
3,is_high_value_engagement,0
4,is_like,0
5,is_follow,0
6,is_comment,0
7,is_forward,0
8,is_hate,0
9,is_profile_enter,0


In [45]:
# Confirming whether the two duration fields are materially different

duration_comparison = impression_fact[
    [
        "event_content_duration_ms", "video_duration_ms"
    ]
].copy()

duration_comparison = duration_comparison[
    duration_comparison["video_duration_ms"].notna()
].copy()

duration_comparison["duration_difference_ms"] = (
    duration_comparison["event_content_duration_ms"] - duration_comparison["video_duration_ms"]
)

duration_comparison["absolute_duration_difference_ms"] = (
    duration_comparison["duration_difference_ms"].abs()
)

duration_concordance_summary = pd.DataFrame(
    {
        "metric_name": ["rows_with_both_duration_fields",
                        "exact_match_count", "exact_match_rate_percent", "median_absolute_difference_ms",
                        "p95_absolute_difference_ms", "rows_with_difference_over_1000_ms", "rate_over_1000_ms_percent"
        ],
        "value": [int(len(duration_comparison)),
                  int((duration_comparison["duration_difference_ms"] == 0).sum()),
                  round((duration_comparison["duration_difference_ms"] == 0).mean() * 100, 4),
                  float(duration_comparison["absolute_duration_difference_ms"].median()),
                  float(duration_comparison["absolute_duration_difference_ms"].quantile(0.95)),
                  int((duration_comparison["absolute_duration_difference_ms"] > 1000).sum()),
                  round((duration_comparison["absolute_duration_difference_ms"] > 1000).mean() * 100, 4)
        ]
    }
)

duration_difference_examples = (
    impression_fact.loc[
        impression_fact["video_duration_ms"].notna() & (impression_fact["event_content_duration_ms"] != impression_fact["video_duration_ms"]),
        [
            "video_id", "event_content_duration_ms", "video_duration_ms"
        ]
    ]
    .drop_duplicates()
    .head(20)
)

display(duration_concordance_summary)
display(duration_difference_examples)

,metric_name,value
0,rows_with_both_duration_fields,2531921.0
1,exact_match_count,2531921.0
2,exact_match_rate_percent,100.0
3,median_absolute_difference_ms,0.0
4,p95_absolute_difference_ms,0.0
5,rows_with_difference_over_1000_ms,0.0
6,rate_over_1000_ms_percent,0.0


,video_id,event_content_duration_ms,video_duration_ms


In [46]:
# Recording why the aggregate video statistics table is being excluded

excluded_source_review = pd.DataFrame(
    {
        "source_table": ["video_dimension_statistic"],
        "current_use_in_impression_fact": ["excluded"],
        "reason": ["Contains aggregate performance fields that are likely post-outcome and unsuitable for the impression-level "
                   "predictive base unless a later descriptive use is specifically justified."
        ]
    }
)

excluded_source_review

,source_table,current_use_in_impression_fact,reason
0,video_dimension_statistic,excluded,Contains aggregate performance fields that are...


In [47]:
# Checking how downstream engagement behaves once a click has happened

downstream_event_columns = ["is_meaningful_watch", "is_like", "is_follow", "is_comment",
                            "is_forward", "is_high_value_engagement"
]

click_dependency_by_traffic_source = (
    impression_fact
    .assign(click_status = np.where(impression_fact["is_click"].eq(1), "clicked", "not_clicked"))
    .groupby(["traffic_source", "click_status"], as_index = False)
    .agg(
        impression_count = ("user_id", "size"),
        meaningful_watch_rate_percent = ("is_meaningful_watch", lambda s: round(s.mean() * 100, 4)),
        like_rate_percent = ("is_like", lambda s: round(s.mean() * 100, 4)),
        follow_rate_percent = ("is_follow", lambda s: round(s.mean() * 100, 4)),
        comment_rate_percent = ("is_comment", lambda s: round(s.mean() * 100, 4)),
        forward_rate_percent = ("is_forward", lambda s: round(s.mean() * 100, 4)),
        high_value_engagement_rate_percent = ("is_high_value_engagement", lambda s: round(s.mean() * 100, 4))
    )
)

click_dependency_by_traffic_source

,traffic_source,click_status,impression_count,meaningful_watch_rate_percent,like_rate_percent,follow_rate_percent,comment_rate_percent,forward_rate_percent,high_value_engagement_rate_percent
0,random,clicked,208934,48.0554,1.6666,0.1101,0.1900,0.1374,0.4351
1,random,not_clicked,977115,0.0374,0.2261,0.0082,0.0014,0.0118,0.0213
2,standard,clicked,655796,72.0468,3.3977,0.2049,0.5381,0.1856,0.9200
3,standard,not_clicked,755854,0.2538,0.5379,0.0221,0.0134,0.0208,0.0562


In [48]:
# Cataloging the retained analysis fields with their business roles

analysis_field_catalog = pd.DataFrame(
    [
        ("user_id", "identifier", "Impression viewer identifier"),
        ("video_id", "identifier", "Video identifier"),
        ("creator_user_id", "identifier", "Video creator identifier"),
        ("event_timestamp_ms", "identifier", "Event timestamp in milliseconds"),
        ("event_date_key", "calendar", "Source event date key"),
        ("event_date", "calendar", "Source event date"),
        ("event_weekday_name", "calendar", "Weekday derived from source event date"),
        ("is_weekend", "calendar", "Weekend flag derived from source event date"),
        ("traffic_source", "delivery_context", "Traffic allocation source"),
        ("feed_tab_id", "delivery_context", "Feed tab identifier"),
        ("is_click", "outcome", "Primary engagement target"),
        ("is_meaningful_watch", "outcome", "Meaningful watch outcome"),
        ("is_social_engagement", "outcome", "Any social engagement outcome"),
        ("is_high_value_engagement", "outcome", "Follow or comment or forward"),
        ("is_like", "outcome", "Like outcome"),
        ("is_follow", "outcome", "Follow outcome"),
        ("is_comment", "outcome", "Comment outcome"),
        ("is_forward", "outcome", "Forward outcome"),
        ("is_hate", "outcome", "Negative reaction outcome"),
        ("is_profile_enter", "outcome", "Profile entry outcome"),
        ("play_time_ms", "behavior_measure", "Observed play time"),
        ("event_content_duration_ms", "behavior_measure", "Content duration recorded on the event"),
        ("playback_completion_rate", "behavior_measure", "Play time divided by content duration, capped to 1"),
        ("profile_stay_time", "behavior_measure", "Profile stay time"),
        ("comment_stay_time", "behavior_measure", "Comment area stay time"),
        ("has_zero_duration", "quality_flag", "Event content duration equals zero"),
        ("has_zero_play_time", "quality_flag", "Play time equals zero"),
        ("has_play_time_exceeding_duration", "quality_flag", "Play time exceeds content duration"),
        ("user_active_degree", "user_attribute", "User activity segment"),
        ("is_video_author", "user_attribute", "Whether the user is a video author"),
        ("follow_user_num", "user_attribute", "Followed users count"),
        ("follow_user_num_range", "user_attribute", "Followed users count band"),
        ("fans_user_num", "user_attribute", "Fans count"),
        ("fans_user_num_range", "user_attribute", "Fans count band"),
        ("friend_user_num", "user_attribute", "Friends count"),
        ("friend_user_num_range", "user_attribute", "Friends count band"),
        ("register_days", "user_attribute", "Days since registration"),
        ("register_days_range", "user_attribute", "Registration age band"),
        ("video_type", "content_attribute", "Video type"),
        ("video_upload_date", "content_attribute", "Video upload date"),
        ("video_upload_type", "content_attribute", "Video upload source type"),
        ("video_width", "content_attribute", "Video width"),
        ("video_height", "content_attribute", "Video height"),
        ("video_orientation", "content_attribute", "Video orientation"),
        ("music_type", "content_attribute", "Music type"),
        ("video_age_days", "content_attribute", "Video age in days at event date")
    ],
    columns = ["field_name", "field_group", "business_definition"]
)

field_profile = pd.DataFrame(
    {
        "field_name": impression_fact.columns,
        "data_type": impression_fact.dtypes.astype(str).values,
        "missing_count": impression_fact.isna().sum().values,
        "missing_rate_percent": (impression_fact.isna().mean().values * 100).round(4),
        "distinct_value_count": [impression_fact[column].nunique(dropna = False)
                                 for column in impression_fact.columns
        ]
    }
)

analysis_field_catalog = (analysis_field_catalog
                          .merge(field_profile, on = "field_name", how = "left")
                          .sort_values(["field_group", "field_name"])
                          .reset_index(drop = True)
)

analysis_field_catalog

,field_name,field_group,business_definition,data_type,missing_count,missing_rate_percent,distinct_value_count
0,comment_stay_time,behavior_measure,Comment area stay time,int32,0,0.0000,24967
1,event_content_duration_ms,behavior_measure,Content duration recorded on the event,int32,0,0.0000,5757
2,play_time_ms,behavior_measure,Observed play time,int32,0,0.0000,160587
3,playback_completion_rate,behavior_measure,"Play time divided by content duration, capped ...",float64,65778,2.5322,2171493
4,profile_stay_time,behavior_measure,Profile stay time,int32,0,0.0000,184
5,event_date,calendar,Source event date,datetime64[us],0,0.0000,30
6,event_date_key,calendar,Source event date key,int32,0,0.0000,30
7,event_weekday_name,calendar,Weekday derived from source event date,string,0,0.0000,7
8,is_weekend,calendar,Weekend flag derived from source event date,int8,0,0.0000,2
9,music_type,content_attribute,Music type,float64,79102,3.0451,6


In [49]:
# Recording which fields are being excluded from the analysis base and why

excluded_field_review = pd.DataFrame(
    [
        ("event_timestamp_utc", "Redundant with event_timestamp_ms"),
        ("event_hour_bucket", "Raw hour field is not reliable enough for main analysis"),
        ("event_hour_of_day", "Derived from event_hour_bucket"),
        ("video_duration_ms", "Duplicates event_content_duration_ms where present"),
        ("playback_completion_rate_capped", "Replaced by a single standardized playback_completion_rate field"),
        ("music_id", "Very high-cardinality identifier with limited direct analytical value at this stage"),
        ("video_tag", "Moved to a separate bridge table because one video can carry multiple tags"),
        ("video_dimension_statistic.*", "Aggregate performance fields are post-outcome summaries and unsuitable for the impression-level analytical base")
    ],
    columns = ["excluded_field", "reason"]
)

excluded_field_review

,excluded_field,reason
0,event_timestamp_utc,Redundant with event_timestamp_ms
1,event_hour_bucket,Raw hour field is not reliable enough for main...
2,event_hour_of_day,Derived from event_hour_bucket
3,video_duration_ms,Duplicates event_content_duration_ms where pre...
4,playback_completion_rate_capped,Replaced by a single standardized playback_com...
5,music_id,Very high-cardinality identifier with limited ...
6,video_tag,Moved to a separate bridge table because one v...
7,video_dimension_statistic.*,Aggregate performance fields are post-outcome ...


In [50]:
# Building the lean analysis base from the wider impression fact table

impression_analysis_base = impression_fact.copy()

impression_analysis_base["playback_completion_rate"] = np.where(
    impression_analysis_base["event_content_duration_ms"] > 0,
    (
        impression_analysis_base["play_time_ms"] / impression_analysis_base["event_content_duration_ms"]
    ).clip(lower = 0, upper = 1),
    np.nan
)


# Keeping only the fields that are justified for analysis and reporting
analysis_base_columns = ["user_id", "video_id", "creator_user_id", "event_timestamp_ms", "event_date_key", "event_date",
                         "traffic_source", "feed_tab_id",
                         "is_click", "is_meaningful_watch", "is_social_engagement", "is_high_value_engagement", "is_like", "is_follow",
                         "is_comment", "is_forward", "is_hate", "is_profile_enter",
                         "play_time_ms", "event_content_duration_ms", "playback_completion_rate", "profile_stay_time", "comment_stay_time",
                         "has_zero_duration", "has_zero_play_time", "has_play_time_exceeding_duration", "user_active_degree", 
                         "is_video_author", "follow_user_num", "follow_user_num_range", "fans_user_num", "fans_user_num_range", 
                         "friend_user_num", "friend_user_num_range", "register_days", "register_days_range", "video_type", 
                         "video_upload_date", "video_upload_type", "video_width", "video_height", "video_orientation", 
                         "music_type", "video_age_days"
]

impression_analysis_base = (impression_analysis_base[analysis_base_columns]
                            .sort_values(["event_date", "event_timestamp_ms", "user_id", "video_id"])
                            .reset_index(drop = True)
)

print(impression_analysis_base.shape)
impression_analysis_base.head()

(2597699, 44)


,user_id,video_id,creator_user_id,event_timestamp_ms,event_date_key,event_date,traffic_source,feed_tab_id,is_click,is_meaningful_watch,is_social_engagement,is_high_value_engagement,is_like,is_follow,is_comment,is_forward,is_hate,is_profile_enter,play_time_ms,event_content_duration_ms,playback_completion_rate,profile_stay_time,comment_stay_time,has_zero_duration,has_zero_play_time,has_play_time_exceeding_duration,user_active_degree,is_video_author,follow_user_num,follow_user_num_range,fans_user_num,fans_user_num_range,friend_user_num,friend_user_num_range,register_days,register_days_range,video_type,video_upload_date,video_upload_type,video_width,video_height,video_orientation,music_type,video_age_days
0,20853,222,6705448,1649475963278,20220409,2022-04-09,standard,1,0,0,0,0,0,0,0,0,0,0,1286,289866,0.004437,0,0,0,0,0,full_active,1,78,"(50,100]",66,"[10,100)",0,0,2479,730+,NORMAL,2022-04-09,Web,720.0,1280.0,vertical,9.0,0
1,8286,222,6705448,1649476012164,20220409,2022-04-09,standard,1,0,0,0,0,0,0,0,0,0,0,1857,289866,0.006406,0,0,0,0,0,2_14_day_new,1,64,"(50,100]",47,"[10,100)",36,"[30,60)",1420,730+,NORMAL,2022-04-09,Web,720.0,1280.0,vertical,9.0,0
2,7406,222,6705448,1649476421916,20220409,2022-04-09,standard,4,1,1,0,0,0,0,0,0,0,1,25750,289866,0.088834,0,0,0,0,0,full_active,1,32,"(10,50]",53,"[10,100)",32,"[30,60)",102,91-180,NORMAL,2022-04-09,Web,720.0,1280.0,vertical,9.0,0
3,25934,6580,4854468,1649476523839,20220409,2022-04-09,standard,0,0,0,0,0,0,0,0,0,0,0,0,55960,0.000000,0,0,0,1,0,full_active,0,78,"(50,100]",0,0,0,0,89,61-90,NORMAL,2022-04-09,LongImport,720.0,1280.0,vertical,9.0,0
4,2073,4566,5597283,1649476630340,20220409,2022-04-09,standard,2,0,1,0,0,0,0,0,0,0,0,14904,14700,1.000000,0,1641,0,0,1,high_active,1,153,"(150,250]",11,"[10,100)",1,"[1,5)",1711,730+,NORMAL,2022-04-09,Web,720.0,1280.0,vertical,4.0,0


In [51]:
# Splitting multi-tag videos into a bridge table for clean tag analysis

video_tag_bridge = (video_attributes[["video_id", "video_tag"]]
                    .dropna(subset = ["video_tag"])
                    .assign(video_tag = lambda frame: frame["video_tag"].astype("string").str.split(","))
                    .explode("video_tag")
                    .assign(video_tag = lambda frame: frame["video_tag"].str.strip())
                    .query("video_tag.notna()", engine = "python")
                    .drop_duplicates()
                    .reset_index(drop = True)
)

video_tag_bridge.head()

,video_id,video_tag
0,0,39
1,1,2
2,2,1
3,3,7
4,4,9


In [52]:
# Confirming that the analysis base is structurally ready to use
analysis_base_readiness = pd.DataFrame(
    {
        "metric_name": ["row_count", "column_count", "duplicate_event_key_count", 
                        "missing_event_date_count", "missing_playback_completion_rate_count",
                        "missing_music_type_count", "missing_video_age_days_count"
        ],
        "value": [int(len(impression_analysis_base)), 
                  int(impression_analysis_base.shape[1]),
                  int(impression_analysis_base.duplicated(
                      subset = ["user_id", "video_id", "event_timestamp_ms"]
                    ).sum()
                ),
                  int(impression_analysis_base["event_date"].isna().sum()),
                  int(impression_analysis_base["playback_completion_rate"].isna().sum()),
                  int(impression_analysis_base["music_type"].isna().sum()),
                  int(impression_analysis_base["video_age_days"].isna().sum()),
        ]
    }
)

analysis_base_readiness

,metric_name,value
0,row_count,2597699
1,column_count,44
2,duplicate_event_key_count,0
3,missing_event_date_count,0
4,missing_playback_completion_rate_count,65778
5,missing_music_type_count,79102
6,missing_video_age_days_count,0


In [53]:
# Summarizing the outcome rates that define the engagement landscape
target_summary = pd.DataFrame(
    {
        "target_name": ["is_click", "is_meaningful_watch", "is_social_engagement", "is_high_value_engagement", "is_like",
                        "is_follow", "is_comment", "is_forward", "is_hate", "is_profile_enter",
        ],
        "positive_count": [int(impression_analysis_base[column].sum())
                           for column in ["is_click", "is_meaningful_watch", "is_social_engagement", "is_high_value_engagement", "is_like", 
                                          "is_follow", "is_comment", "is_forward", "is_hate", "is_profile_enter"
            ]
        ],
        "positive_rate_percent": [round(impression_analysis_base[column].mean() * 100, 4)
                                  for column in ["is_click", "is_meaningful_watch", "is_social_engagement", "is_high_value_engagement", 
                                                 "is_like", "is_follow", "is_comment", "is_forward", "is_hate", "is_profile_enter"
            ]
        ],
    }
).sort_values("positive_rate_percent", ascending = False).reset_index(drop = True)

target_summary

,target_name,positive_count,positive_rate_percent
0,is_click,864730,33.2883
1,is_meaningful_watch,575167,22.1414
2,is_profile_enter,40915,1.5750
3,is_social_engagement,38023,1.4637
4,is_like,32039,1.2334
5,is_high_value_engagement,7575,0.2916
6,is_comment,4041,0.1556
7,is_hate,2056,0.0791
8,is_follow,1821,0.0701
9,is_forward,1776,0.0684


In [54]:
# Selecting the main categorical dimensions for segment-level analysis

selected_categorical_fields = ["traffic_source", "feed_tab_id", "user_active_degree", "follow_user_num_range", 
                               "fans_user_num_range", "friend_user_num_range", "register_days_range", 
                               "video_type", "video_upload_type", "video_orientation", "music_type"
]

categorical_field_quality = pd.DataFrame(
    {
        "field_name": selected_categorical_fields,
        "distinct_value_count": [impression_analysis_base[field].nunique(dropna = False)
                                 for field in selected_categorical_fields
        ],
        "missing_count": [int(impression_analysis_base[field].isna().sum())
                          for field in selected_categorical_fields
        ],
        "top_category": [impression_analysis_base[field].value_counts(dropna = False).index[0]
                         for field in selected_categorical_fields
        ],
        "top_category_count": [int(impression_analysis_base[field].value_counts(dropna = False).iloc[0])
                               for field in selected_categorical_fields
        ]
    }
)

categorical_field_quality

,field_name,distinct_value_count,missing_count,top_category,top_category_count
0,traffic_source,2,0,standard,1411650
1,feed_tab_id,15,0,1,2236411
2,user_active_degree,9,0,full_active,1889845
3,follow_user_num_range,8,0,500+,537421
4,fans_user_num_range,9,0,"[10,100)",991288
5,friend_user_num_range,7,0,"[5,30)",726049
6,register_days_range,8,0,730+,1909229
7,video_type,3,0,NORMAL,2568173
8,video_upload_type,14,0,LongImport,1070378
9,video_orientation,3,0,vertical,1959315


In [55]:
# Comparing engagement rates across the key categorical segments
categorical_target_profiles = []

for field_name in selected_categorical_fields:
    field_profile = (
        impression_analysis_base
        .groupby(field_name, dropna = False, as_index = False)
        .agg(impression_count = ("user_id", "size"),
             click_rate_percent = ("is_click", lambda s: round(s.mean() * 100, 4)),
             meaningful_watch_rate_percent = ("is_meaningful_watch", lambda s: round(s.mean() * 100, 4)),
             social_engagement_rate_percent = ("is_social_engagement", lambda s: round(s.mean() * 100, 4)),
             high_value_engagement_rate_percent = ("is_high_value_engagement", lambda s: round(s.mean() * 100, 4))
        )
        .sort_values("impression_count", ascending = False)
        .head(20)
        .reset_index(drop = True)
    )

    field_profile.insert(0, "field_name", field_name)
    categorical_target_profiles.append(field_profile)

categorical_target_profiles = pd.concat(categorical_target_profiles, ignore_index = True)
categorical_target_profiles.head(50)

,field_name,traffic_source,impression_count,click_rate_percent,meaningful_watch_rate_percent,social_engagement_rate_percent,high_value_engagement_rate_percent,feed_tab_id,user_active_degree,follow_user_num_range,fans_user_num_range,friend_user_num_range,register_days_range,video_type,video_upload_type,video_orientation,music_type
0,traffic_source,standard,1411650,46.4560,33.6059,2.2252,0.4575,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN
1,traffic_source,random,1186049,17.6160,8.4962,0.5574,0.0942,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN
2,feed_tab_id,NaN,2236411,34.0898,22.4224,1.5149,0.3000,1.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN
3,feed_tab_id,NaN,177846,9.0837,4.1210,0.4959,0.1198,0.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN
4,feed_tab_id,NaN,93165,61.5499,48.3261,1.8934,0.2769,4.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN
5,feed_tab_id,NaN,49128,47.4373,37.2659,2.4365,0.6493,2.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN
6,feed_tab_id,NaN,20236,17.7357,8.8703,0.9834,0.2669,6.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN
7,feed_tab_id,NaN,7470,0.0000,3.4003,0.3347,0.0669,11.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN
8,feed_tab_id,NaN,4008,2.2455,0.3992,0.0998,0.0250,3.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN
9,feed_tab_id,NaN,3909,4.0931,1.8931,0.1023,0.0000,8.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN


In [56]:
# Profiling the main numeric fields to understand scale and sparsity

selected_numeric_fields = ["play_time_ms", "event_content_duration_ms", "playback_completion_rate", "profile_stay_time",
                           "comment_stay_time", "follow_user_num", "fans_user_num",
                           "friend_user_num", "register_days", "video_width", "video_height", "video_age_days"
]

numeric_field_profile = []

for field_name in selected_numeric_fields:
    series = impression_analysis_base[field_name]

    numeric_field_profile.append(
        {
            "field_name": field_name,
            "missing_count": int(series.isna().sum()),
            "missing_rate_percent": round(series.isna().mean() * 100, 4),
            "minimum_value": series.min(),
            "p01_value": series.quantile(0.01),
            "median_value": series.median(),
            "p99_value": series.quantile(0.99),
            "maximum_value": series.max(),
            "zero_value_rate_percent": round((series.fillna(0).eq(0)).mean() * 100, 4)
        }
    )

numeric_field_profile = pd.DataFrame(numeric_field_profile)
numeric_field_profile

,field_name,missing_count,missing_rate_percent,minimum_value,p01_value,median_value,p99_value,maximum_value,zero_value_rate_percent
0,play_time_ms,0,0.0000,0.0,0.0,2865.000000,176995.02,1023809.0,7.5694
1,event_content_duration_ms,0,0.0000,0.0,0.0,73566.000000,465041.00,1177720.0,2.5322
2,playback_completion_rate,65778,2.5322,0.0,0.0,0.055252,1.00,1.0,9.9019
3,profile_stay_time,0,0.0000,0.0,0.0,0.000000,0.00,300000.0,99.9929
4,comment_stay_time,0,0.0000,0.0,0.0,0.000000,9727.04,300000.0,96.6682
5,follow_user_num,0,0.0000,0.0,1.0,151.000000,3993.00,5003.0,0.6674
6,fans_user_num,0,0.0000,0.0,0.0,55.000000,4109.00,1986249.0,3.8694
7,friend_user_num,0,0.0000,0.0,0.0,18.000000,1581.00,4995.0,11.5081
8,register_days,0,0.0000,14.0,47.0,1186.000000,2702.00,3624.0,0.0000
9,video_width,0,0.0000,270.0,720.0,720.000000,1280.00,2400.0,0.0000


In [57]:
# Checking simple numeric associations with click and high-value engagement
candidate_numeric_fields = ["follow_user_num", "fans_user_num", "friend_user_num", "register_days",
                            "event_content_duration_ms", "video_width", "video_height", "video_age_days"
]

candidate_numeric_association = []

for field_name in candidate_numeric_fields:
    valid_rows = impression_analysis_base[[field_name, "is_click", "is_high_value_engagement"]].dropna()

    candidate_numeric_association.append(
        {
            "field_name": field_name,
            "row_count_used": int(len(valid_rows)),
            "correlation_with_click": round(valid_rows[field_name].corr(valid_rows["is_click"]), 6),
            "correlation_with_high_value_engagement": round(
                valid_rows[field_name].corr(valid_rows["is_high_value_engagement"]),
                6
            )
        }
    )

candidate_numeric_association = (
    pd.DataFrame(candidate_numeric_association)
    .sort_values("correlation_with_click", key = lambda s: s.abs(), ascending = False)
    .reset_index(drop = True)
)

candidate_numeric_association

,field_name,row_count_used,correlation_with_click,correlation_with_high_value_engagement
0,video_age_days,2597699,-0.237346,-0.025258
1,register_days,2597699,-0.032549,-0.012668
2,video_height,2597699,-0.024519,0.000828
3,event_content_duration_ms,2597699,-0.011129,-0.004228
4,friend_user_num,2597699,-0.010206,0.001197
5,follow_user_num,2597699,0.003744,0.021147
6,video_width,2597699,-0.000830,-0.000073
7,fans_user_num,2597699,-0.000501,-0.000539


In [58]:
# Measuring how video tags are relating to impression-level outcomes
video_tag_performance = (
    impression_analysis_base[
        [
            "video_id", "is_click", "is_meaningful_watch", "is_social_engagement", "is_high_value_engagement"
        ]
    ]
    .merge(video_tag_bridge, on = "video_id", how = "inner")
    .groupby("video_tag", as_index = False)
    .agg(impression_count = ("video_id", "size"),
         distinct_video_count = ("video_id", "nunique"),
         click_rate_percent = ("is_click", lambda s: round(s.mean() * 100, 4)),
         meaningful_watch_rate_percent = ("is_meaningful_watch", lambda s: round(s.mean() * 100, 4)),
         social_engagement_rate_percent = ("is_social_engagement", lambda s: round(s.mean() * 100, 4)),
         high_value_engagement_rate_percent = ("is_high_value_engagement", lambda s: round(s.mean() * 100, 4))
    )
    .sort_values("impression_count", ascending = False)
    .reset_index(drop = True)
)

video_tag_performance.head(30)

,video_tag,impression_count,distinct_video_count,click_rate_percent,meaningful_watch_rate_percent,social_engagement_rate_percent,high_value_engagement_rate_percent
0,39,581755,1446,39.6003,26.8584,1.2433,0.2534
1,3,213848,732,33.0267,24.2738,1.6086,0.3362
2,12,212959,606,30.7130,19.1370,0.9406,0.2376
3,9,204384,419,44.1458,31.3782,1.7859,0.4247
4,20,170508,675,25.1736,16.2086,1.9389,0.3331
5,6,165654,400,35.8718,23.5165,1.8895,0.3495
6,68,119077,428,31.2344,18.8769,1.1530,0.1890
7,62,116579,489,21.6248,12.2990,0.9093,0.1964
8,7,109971,383,29.3023,18.8486,1.2612,0.1991
9,43,104551,428,32.0418,20.2064,0.8914,0.1884


In [59]:
# Testing whether music ID is analytically useful before excluding it

music_id_profile = pd.DataFrame(
    {
        "field_name": ["music_id"],
        "missing_count": [int(impression_fact["music_id"].isna().sum())],
        "missing_rate_percent": [round(impression_fact["music_id"].isna().mean() * 100, 4)],
        "distinct_value_count": [int(impression_fact["music_id"].nunique(dropna = False))],
        "top_value": [impression_fact["music_id"].value_counts(dropna = False).index[0]],
        "top_value_count": [int(impression_fact["music_id"].value_counts(dropna = False).iloc[0])]
    }
)

music_id_target_profile = (
    impression_fact
    .groupby("music_id", as_index = False)
    .agg(impression_count = ("user_id", "size"),
         click_rate_percent = ("is_click", lambda s: round(s.mean() * 100, 4)),
         meaningful_watch_rate_percent = ("is_meaningful_watch", lambda s: round(s.mean() * 100, 4)),
         social_engagement_rate_percent = ("is_social_engagement", lambda s: round(s.mean() * 100, 4)),
         high_value_engagement_rate_percent = ("is_high_value_engagement", lambda s: round(s.mean() * 100, 4))
    )
    .sort_values("impression_count", ascending = False)
    .head(20)
    .reset_index(drop = True)
)

display(music_id_profile)
display(music_id_target_profile)

,field_name,missing_count,missing_rate_percent,distinct_value_count,top_value,top_value_count
0,music_id,0,0.0,7202,0,81443


,music_id,impression_count,click_rate_percent,meaningful_watch_rate_percent,social_engagement_rate_percent,high_value_engagement_rate_percent
0,0,81443,37.2641,25.6462,1.3077,0.2689
1,9152913689,10515,57.0043,39.8859,1.5407,0.3233
2,9167116455,9911,60.8617,50.2068,3.9956,0.6962
3,9139255224,7824,63.5225,63.3308,1.1247,0.3579
4,6202189854,7769,48.3331,43.0042,0.6565,0.2188
5,9141209460,6561,47.1574,30.9252,1.5242,0.7011
6,9107419068,6211,28.5622,9.6925,1.8999,0.3059
7,9155224806,5014,60.3311,49.7407,2.9318,2.3933
8,9151301996,4975,54.4523,42.8342,3.5578,0.7236
9,9143427573,4798,58.4827,44.4977,1.5006,0.2501


In [60]:
# Preparing the leakage-safe base for click prediction work

click_model_base_columns = ["user_id", "video_id", "creator_user_id", "event_timestamp_ms", "traffic_source", "feed_tab_id",
                            "user_active_degree", "is_video_author", "follow_user_num", "follow_user_num_range", "fans_user_num",
                            "fans_user_num_range", "friend_user_num", "friend_user_num_range", "register_days", "register_days_range", 
                            "video_type", "video_upload_date", "video_upload_type", "video_width", "video_height", "video_orientation",
                            "music_type", "video_age_days", "is_click"
]

click_model_base = (impression_analysis_base[click_model_base_columns]
                    .sort_values(["event_timestamp_ms", "user_id", "video_id"])
                    .reset_index(drop = True)
)

pd.DataFrame(
    {
        "dataset_name": ["click_model_base"],
        "row_count": [len(click_model_base)],
        "column_count": [click_model_base.shape[1]],
        "duplicate_event_key_count": [
            int(click_model_base.duplicated(subset = ["user_id", "video_id", "event_timestamp_ms"]).sum())
        ],
        "missing_music_type_count": [int(click_model_base["music_type"].isna().sum())]
    }
)

,dataset_name,row_count,column_count,duplicate_event_key_count,missing_music_type_count
0,click_model_base,2597699,25,0,79102


In [61]:
# Saving the analysis-ready impression base for downstream use.
upload_dataframe_to_parquet_blob(impression_analysis_base,
                                 container_names["curated"],
                                 "impression_analysis_base.parquet"
)

pd.DataFrame(
    {
        "dataset_name": ["impression_analysis_base"],
        "row_count": [len(impression_analysis_base)],
        "column_count": [impression_analysis_base.shape[1]],
        "duplicate_event_key_count": 
            [
                int(impression_analysis_base.duplicated(
                        subset = ["user_id", "video_id", "event_timestamp_ms"]
                    ).sum()
                )
            ]
    }
)

,dataset_name,row_count,column_count,duplicate_event_key_count
0,impression_analysis_base,2597699,44,0


In [62]:
# Saving the tag bridge for tag-level analysis and reporting
upload_dataframe_to_parquet_blob(video_tag_bridge, 
                                 container_names["curated"], 
                                 "video_tag_bridge.parquet"
)

pd.DataFrame(
    {
        "dataset_name": ["video_tag_bridge"],
        "row_count": [len(video_tag_bridge)],
        "column_count": [video_tag_bridge.shape[1]],
        "duplicate_row_count": [int(video_tag_bridge.duplicated().sum())]
    }
)

,dataset_name,row_count,column_count,duplicate_row_count
0,video_tag_bridge,9310,2,0


In [63]:
# Verifying that the analysis base is complete and uniquely keyed

required_analysis_columns = ["user_id", "video_id", "event_timestamp_ms", "event_date", "traffic_source", "feed_tab_id",
                             "is_click", "is_meaningful_watch", "is_social_engagement", "is_high_value_engagement", "is_like",
                             "is_follow", "is_comment", "is_forward", "is_profile_enter", "user_active_degree",
                             "follow_user_num_range", "fans_user_num_range", "friend_user_num_range", "register_days_range",
                             "video_type", "video_upload_type", "video_orientation", "music_type", "video_age_days"
]

missing_analysis_columns = [column_name
                            for column_name in required_analysis_columns
                            if column_name not in impression_analysis_base.columns
]

if missing_analysis_columns:
    raise KeyError(f"Missing required analysis columns : {missing_analysis_columns}")

analysis_row_count = len(impression_analysis_base)
analysis_distinct_user_count = impression_analysis_base["user_id"].nunique()
analysis_distinct_video_count = impression_analysis_base["video_id"].nunique()

pd.DataFrame(
    {
        "metric_name": ["analysis_row_count", 
                        "analysis_distinct_user_count",
                        "analysis_distinct_video_count",
                        "duplicate_event_key_count"
        ],
        "value": [int(analysis_row_count),
                  int(analysis_distinct_user_count),
                  int(analysis_distinct_video_count),
                  int(impression_analysis_base.duplicated(
                        subset = ["user_id", "video_id", "event_timestamp_ms"]
                    ).sum()
                )
        ]
    }
)

,metric_name,value
0,analysis_row_count,2597699
1,analysis_distinct_user_count,27285
2,analysis_distinct_video_count,7583
3,duplicate_event_key_count,0


In [64]:
# Summarizing the overall impression-to-engagement outcome rates

overall_funnel_summary = pd.DataFrame(
    {
        "funnel_stage": ["impressions", 
                         "clicks",
                         "meaningful_watches",
                         "profile_enters",
                         "likes",
                         "high_value_engagements",
                         "comments",
                         "follows",
                         "forwards"
        ],
        "event_count": [int(len(impression_analysis_base)),
                        int(impression_analysis_base["is_click"].sum()),
                        int(impression_analysis_base["is_meaningful_watch"].sum()),
                        int(impression_analysis_base["is_profile_enter"].sum()),
                        int(impression_analysis_base["is_like"].sum()),
                        int(impression_analysis_base["is_high_value_engagement"].sum()),
                        int(impression_analysis_base["is_comment"].sum()),
                        int(impression_analysis_base["is_follow"].sum()),
                        int(impression_analysis_base["is_forward"].sum())
        ]
    }
)

overall_funnel_summary["rate_from_impressions_percent"] = (
    overall_funnel_summary["event_count"] / len(impression_analysis_base) * 100
).round(4)

overall_funnel_summary

,funnel_stage,event_count,rate_from_impressions_percent
0,impressions,2597699,100.0000
1,clicks,864730,33.2883
2,meaningful_watches,575167,22.1414
3,profile_enters,40915,1.5750
4,likes,32039,1.2334
5,high_value_engagements,7575,0.2916
6,comments,4041,0.1556
7,follows,1821,0.0701
8,forwards,1776,0.0684


In [65]:
# Building reusable grouped funnel summaries for comparison views

def build_funnel_summary(dataframe, group_columns):
    grouped_summary = (
        dataframe.groupby(group_columns, dropna = False, as_index = False)
        .agg(impression_count = ("user_id", "size"),
             click_count = ("is_click", "sum"),
             meaningful_watch_count = ("is_meaningful_watch", "sum"),
             profile_enter_count = ("is_profile_enter", "sum"),
             like_count = ("is_like", "sum"),
             high_value_engagement_count = ("is_high_value_engagement", "sum"),
             comment_count = ("is_comment", "sum"),
             follow_count = ("is_follow", "sum"),
             forward_count = ("is_forward", "sum")
        )
    )

    # Converting grouped counts into comparable impression-level rates
    grouped_summary["click_rate_percent"] = (
        grouped_summary["click_count"] / grouped_summary["impression_count"] * 100
    ).round(4)

    grouped_summary["meaningful_watch_rate_percent"] = (
        grouped_summary["meaningful_watch_count"] / grouped_summary["impression_count"] * 100
    ).round(4)

    grouped_summary["profile_enter_rate_percent"] = (
        grouped_summary["profile_enter_count"] / grouped_summary["impression_count"] * 100
    ).round(4)

    grouped_summary["like_rate_percent"] = (
        grouped_summary["like_count"] / grouped_summary["impression_count"] * 100
    ).round(4)

    grouped_summary["high_value_engagement_rate_percent"] = (
        grouped_summary["high_value_engagement_count"] / grouped_summary["impression_count"] * 100
    ).round(4)

    grouped_summary["comment_rate_percent"] = (
        grouped_summary["comment_count"] / grouped_summary["impression_count"] * 100
    ).round(4)

    grouped_summary["follow_rate_percent"] = (
        grouped_summary["follow_count"] / grouped_summary["impression_count"] * 100
    ).round(4)

    grouped_summary["forward_rate_percent"] = (
        grouped_summary["forward_count"] / grouped_summary["impression_count"] * 100
    ).round(4)

    return grouped_summary.sort_values("impression_count", ascending = False).reset_index(drop = True)


traffic_source_funnel_summary = build_funnel_summary(
    impression_analysis_base,
    ["traffic_source"]
)

feed_tab_funnel_summary = build_funnel_summary(
    impression_analysis_base,
    ["feed_tab_id"]
)

display(traffic_source_funnel_summary)
display(feed_tab_funnel_summary)

,traffic_source,impression_count,click_count,meaningful_watch_count,profile_enter_count,like_count,high_value_engagement_count,comment_count,follow_count,forward_count,click_rate_percent,meaningful_watch_rate_percent,profile_enter_rate_percent,like_rate_percent,high_value_engagement_rate_percent,comment_rate_percent,follow_rate_percent,forward_rate_percent
0,standard,1411650,655796,474398,34318,26348,6458,3630,1511,1374,46.456,33.6059,2.4311,1.8665,0.4575,0.2571,0.1070,0.0973
1,random,1186049,208934,100769,6597,5691,1117,411,310,402,17.616,8.4962,0.5562,0.4798,0.0942,0.0347,0.0261,0.0339


,feed_tab_id,impression_count,click_count,meaningful_watch_count,profile_enter_count,like_count,high_value_engagement_count,comment_count,follow_count,forward_count,click_rate_percent,meaningful_watch_rate_percent,profile_enter_rate_percent,like_rate_percent,high_value_engagement_rate_percent,comment_rate_percent,follow_rate_percent,forward_rate_percent
0,1,2236411,762389,501456,35685,28596,6710,3574,1599,1597,34.0898,22.4224,1.5956,1.2787,0.3000,0.1598,0.0715,0.0714
1,0,177846,16155,7329,1225,711,213,126,54,35,9.0837,4.1210,0.6888,0.3998,0.1198,0.0708,0.0304,0.0197
2,4,93165,57343,45023,2720,1540,258,179,40,39,61.5499,48.3261,2.9196,1.6530,0.2769,0.1921,0.0429,0.0419
3,2,49128,23305,18308,1138,945,319,136,88,95,47.4373,37.2659,2.3164,1.9235,0.6493,0.2768,0.1791,0.1934
4,6,20236,3589,1795,122,160,54,21,27,7,17.7357,8.8703,0.6029,0.7907,0.2669,0.1038,0.1334,0.0346
5,11,7470,0,254,0,23,5,0,5,0,0.0000,3.4003,0.0000,0.3079,0.0669,0.0000,0.0669,0.0000
6,3,4008,90,16,2,3,1,1,0,0,2.2455,0.3992,0.0499,0.0749,0.0250,0.0250,0.0000,0.0000
7,8,3909,160,74,0,4,0,0,0,0,4.0931,1.8931,0.0000,0.1023,0.0000,0.0000,0.0000,0.0000
8,5,2009,564,342,20,22,10,3,4,3,28.0737,17.0234,0.9955,1.0951,0.4978,0.1493,0.1991,0.1493
9,12,1472,200,151,0,6,1,0,1,0,13.5870,10.2582,0.0000,0.4076,0.0679,0.0000,0.0679,0.0000


In [66]:
# Comparing funnel outcomes across key user and content segments

segment_fields = ["user_active_degree", "follow_user_num_range", "fans_user_num_range", "friend_user_num_range",
                  "register_days_range", "video_type", "video_upload_type", "video_orientation", "music_type"
]

segment_funnel_summary_list = []

for field_name in segment_fields:
    segment_summary = build_funnel_summary(
        impression_analysis_base,
        [field_name]
    )
    segment_summary.insert(0, "segment_field", field_name)
    segment_funnel_summary_list.append(segment_summary)

segment_funnel_summary = pd.concat(segment_funnel_summary_list, ignore_index = True)

segment_funnel_summary.head(100)

,segment_field,user_active_degree,impression_count,click_count,meaningful_watch_count,profile_enter_count,like_count,high_value_engagement_count,comment_count,follow_count,forward_count,click_rate_percent,meaningful_watch_rate_percent,profile_enter_rate_percent,like_rate_percent,high_value_engagement_rate_percent,comment_rate_percent,follow_rate_percent,forward_rate_percent,follow_user_num_range,fans_user_num_range,friend_user_num_range,register_days_range,video_type,video_upload_type,video_orientation,music_type
0,user_active_degree,full_active,1889845,612323,406125,29569,20175,5049,2856,1041,1198,32.4007,21.4899,1.5646,1.0675,0.2672,0.1511,0.0551,0.0634,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN
1,user_active_degree,high_active,462463,161878,108521,7179,7407,1558,682,532,356,35.0034,23.4659,1.5523,1.6016,0.3369,0.1475,0.1150,0.0770,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN
2,user_active_degree,middle_active,166902,62070,41276,2869,3331,697,351,183,167,37.1895,24.7307,1.7190,1.9958,0.4176,0.2103,0.1096,0.1001,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN
3,user_active_degree,2_14_day_new,55735,19759,13355,898,703,171,97,37,38,35.4517,23.9616,1.6112,1.2613,0.3068,0.1740,0.0664,0.0682,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN
4,user_active_degree,low_active,15407,5932,3948,249,321,62,35,14,13,38.5020,25.6247,1.6161,2.0835,0.4024,0.2272,0.0909,0.0844,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN
5,user_active_degree,single_low_active,4481,1799,1313,93,78,24,10,10,4,40.1473,29.3015,2.0754,1.7407,0.5356,0.2232,0.2232,0.0893,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN
6,user_active_degree,30day_retention,1228,418,269,32,10,9,8,1,0,34.0391,21.9055,2.6059,0.8143,0.7329,0.6515,0.0814,0.0000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN
7,user_active_degree,day_new,1111,437,300,23,13,5,2,3,0,39.3339,27.0027,2.0702,1.1701,0.4500,0.1800,0.2700,0.0000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN
8,user_active_degree,UNKNOWN,527,114,60,3,1,0,0,0,0,21.6319,11.3852,0.5693,0.1898,0.0000,0.0000,0.0000,0.0000,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN
9,follow_user_num_range,<NA>,537421,182101,121245,8874,10768,2378,908,941,563,33.8842,22.5605,1.6512,2.0036,0.4425,0.1690,0.1751,0.1048,500+,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN


In [67]:
# Measuring what clicked impressions are converting into next

clicked_impressions = impression_analysis_base.loc[
    impression_analysis_base["is_click"] == 1
].copy()

post_click_conversion_summary = pd.DataFrame(
    {
        "conversion_name": ["click_to_meaningful_watch",
                            "click_to_profile_enter",
                            "click_to_like",
                            "click_to_high_value_engagement",
                            "click_to_comment",
                            "click_to_follow",
                            "click_to_forward"
        ],
        "converted_count": [int(clicked_impressions["is_meaningful_watch"].sum()),
                            int(clicked_impressions["is_profile_enter"].sum()),
                            int(clicked_impressions["is_like"].sum()),
                            int(clicked_impressions["is_high_value_engagement"].sum()),
                            int(clicked_impressions["is_comment"].sum()),
                            int(clicked_impressions["is_follow"].sum()),
                            int(clicked_impressions["is_forward"].sum())
        ]
    }
)

post_click_conversion_summary["clicked_impression_count"] = int(len(clicked_impressions))
post_click_conversion_summary["conversion_rate_percent"] = (
    post_click_conversion_summary["converted_count"] / len(clicked_impressions) * 100
).round(4)

post_click_conversion_summary

,conversion_name,converted_count,clicked_impression_count,conversion_rate_percent
0,click_to_meaningful_watch,572884,864730,66.2500
1,click_to_profile_enter,36707,864730,4.2449
2,click_to_like,25764,864730,2.9794
3,click_to_high_value_engagement,6942,864730,0.8028
4,click_to_comment,3926,864730,0.4540
5,click_to_follow,1574,864730,0.1820
6,click_to_forward,1504,864730,0.1739


In [68]:
# Tracking daily movement in core engagement rates

post_click_conversion_by_traffic_source = (
    clicked_impressions
    .groupby("traffic_source", as_index = False)
    .agg(clicked_impression_count = ("user_id", "size"),
         meaningful_watch_count = ("is_meaningful_watch", "sum"),
         profile_enter_count = ("is_profile_enter", "sum"),
         like_count = ("is_like", "sum"),
         high_value_engagement_count = ("is_high_value_engagement", "sum"),
         comment_count = ("is_comment", "sum"),follow_count = ("is_follow", "sum"),
         forward_count = ("is_forward", "sum")
    )
)

post_click_conversion_by_traffic_source["click_to_meaningful_watch_rate_percent"] = (
    post_click_conversion_by_traffic_source["meaningful_watch_count"] 
    / post_click_conversion_by_traffic_source["clicked_impression_count"] 
    * 100
).round(4)

post_click_conversion_by_traffic_source["click_to_profile_enter_rate_percent"] = (
    post_click_conversion_by_traffic_source["profile_enter_count"] 
    / post_click_conversion_by_traffic_source["clicked_impression_count"] 
    * 100
).round(4)

post_click_conversion_by_traffic_source["click_to_like_rate_percent"] = (
    post_click_conversion_by_traffic_source["like_count"]
    / post_click_conversion_by_traffic_source["clicked_impression_count"]
    * 100
).round(4)

post_click_conversion_by_traffic_source["click_to_high_value_engagement_rate_percent"] = (
    post_click_conversion_by_traffic_source["high_value_engagement_count"]
    / post_click_conversion_by_traffic_source["clicked_impression_count"]
    * 100
).round(4)

post_click_conversion_by_traffic_source["click_to_comment_rate_percent"] = (
    post_click_conversion_by_traffic_source["comment_count"]
    / post_click_conversion_by_traffic_source["clicked_impression_count"]
    * 100
).round(4)

post_click_conversion_by_traffic_source["click_to_follow_rate_percent"] = (
    post_click_conversion_by_traffic_source["follow_count"]
    / post_click_conversion_by_traffic_source["clicked_impression_count"]
    * 100
).round(4)

post_click_conversion_by_traffic_source["click_to_forward_rate_percent"] = (
    post_click_conversion_by_traffic_source["forward_count"]
    / post_click_conversion_by_traffic_source["clicked_impression_count"]
    * 100
).round(4)

post_click_conversion_by_traffic_source

,traffic_source,clicked_impression_count,meaningful_watch_count,profile_enter_count,like_count,high_value_engagement_count,comment_count,follow_count,forward_count,click_to_meaningful_watch_rate_percent,click_to_profile_enter_rate_percent,click_to_like_rate_percent,click_to_high_value_engagement_rate_percent,click_to_comment_rate_percent,click_to_follow_rate_percent,click_to_forward_rate_percent
0,random,208934,100404,4284,3482,909,397,230,287,48.0554,2.0504,1.6666,0.4351,0.1900,0.1101,0.1374
1,standard,655796,472480,32423,22282,6033,3529,1344,1217,72.0468,4.9441,3.3977,0.9200,0.5381,0.2049,0.1856


In [69]:
# Tracking daily movement in core engagement rates

daily_funnel_summary = (impression_analysis_base
                        .groupby("event_date", as_index = False)
                        .agg(impression_count = ("user_id", "size"),
                             click_count = ("is_click", "sum"),
                             meaningful_watch_count = ("is_meaningful_watch", "sum"),
                             social_engagement_count = ("is_social_engagement", "sum"),
                             high_value_engagement_count = ("is_high_value_engagement", "sum")
    )
    .sort_values("event_date")
    .reset_index(drop = True)
)

daily_funnel_summary["click_rate_percent"] = (
    daily_funnel_summary["click_count"] / daily_funnel_summary["impression_count"] * 100
).round(4)

daily_funnel_summary["meaningful_watch_rate_percent"] = (
    daily_funnel_summary["meaningful_watch_count"] / daily_funnel_summary["impression_count"] * 100
).round(4)

daily_funnel_summary["social_engagement_rate_percent"] = (
    daily_funnel_summary["social_engagement_count"] / daily_funnel_summary["impression_count"] * 100
).round(4)

daily_funnel_summary["high_value_engagement_rate_percent"] = (
    daily_funnel_summary["high_value_engagement_count"] / daily_funnel_summary["impression_count"] * 100
).round(4)

daily_funnel_summary

,event_date,impression_count,click_count,meaningful_watch_count,social_engagement_count,high_value_engagement_count,click_rate_percent,meaningful_watch_rate_percent,social_engagement_rate_percent,high_value_engagement_rate_percent
0,2022-04-09,52390,23510,17697,1084,213,44.8750,33.7793,2.0691,0.4066
1,2022-04-10,226043,106015,77428,5061,963,46.9004,34.2537,2.2390,0.4260
2,2022-04-11,275743,127042,92548,5865,1252,46.0726,33.5631,2.1270,0.4540
3,2022-04-12,162146,75021,54832,3521,762,46.2676,33.8164,2.1715,0.4699
4,2022-04-13,91714,41651,29785,2034,426,45.4140,32.4760,2.2178,0.4645
5,2022-04-14,68971,31551,22601,1497,331,45.7453,32.7688,2.1705,0.4799
6,2022-04-15,58239,29413,21320,1570,299,50.5040,36.6078,2.6958,0.5134
7,2022-04-16,60538,31308,22910,1710,301,51.7163,37.8440,2.8247,0.4972
8,2022-04-17,43567,21757,15970,1103,208,49.9392,36.6562,2.5317,0.4774
9,2022-04-18,24140,11259,8092,499,108,46.6404,33.5211,2.0671,0.4474


In [70]:
# Breaking daily performance by traffic source mix
daily_traffic_source_summary = (impression_analysis_base
                                .groupby(["event_date", "traffic_source"], as_index = False)
                                .agg(impression_count = ("user_id", "size"),
                                     click_count = ("is_click", "sum"),
                                     meaningful_watch_count = ("is_meaningful_watch", "sum"),
                                     social_engagement_count = ("is_social_engagement", "sum"),
                                     high_value_engagement_count = ("is_high_value_engagement", "sum")
    )
    .sort_values(["event_date", "traffic_source"])
    .reset_index(drop = True)
)

daily_traffic_source_summary["traffic_source_share_of_day_percent"] = (
    daily_traffic_source_summary["impression_count"]
    / daily_traffic_source_summary.groupby("event_date")["impression_count"].transform("sum")
    * 100
).round(4)

daily_traffic_source_summary["click_rate_percent"] = (
    daily_traffic_source_summary["click_count"]
    / daily_traffic_source_summary["impression_count"]
    * 100
).round(4)

daily_traffic_source_summary["meaningful_watch_rate_percent"] = (
    daily_traffic_source_summary["meaningful_watch_count"]
    / daily_traffic_source_summary["impression_count"]
    * 100
).round(4)

daily_traffic_source_summary["social_engagement_rate_percent"] = (
    daily_traffic_source_summary["social_engagement_count"]
    / daily_traffic_source_summary["impression_count"]
    * 100
).round(4)

daily_traffic_source_summary["high_value_engagement_rate_percent"] = (
    daily_traffic_source_summary["high_value_engagement_count"]
    / daily_traffic_source_summary["impression_count"]
    * 100
).round(4)

daily_traffic_source_summary

,event_date,traffic_source,impression_count,click_count,meaningful_watch_count,social_engagement_count,high_value_engagement_count,traffic_source_share_of_day_percent,click_rate_percent,meaningful_watch_rate_percent,social_engagement_rate_percent,high_value_engagement_rate_percent
0,2022-04-09,standard,52390,23510,17697,1084,213,100.0000,44.8750,33.7793,2.0691,0.4066
1,2022-04-10,standard,226043,106015,77428,5061,963,100.0000,46.9004,34.2537,2.2390,0.4260
2,2022-04-11,standard,275743,127042,92548,5865,1252,100.0000,46.0726,33.5631,2.1270,0.4540
3,2022-04-12,standard,162146,75021,54832,3521,762,100.0000,46.2676,33.8164,2.1715,0.4699
4,2022-04-13,standard,91714,41651,29785,2034,426,100.0000,45.4140,32.4760,2.2178,0.4645
5,2022-04-14,standard,68971,31551,22601,1497,331,100.0000,45.7453,32.7688,2.1705,0.4799
6,2022-04-15,standard,58239,29413,21320,1570,299,100.0000,50.5040,36.6078,2.6958,0.5134
7,2022-04-16,standard,60538,31308,22910,1710,301,100.0000,51.7163,37.8440,2.8247,0.4972
8,2022-04-17,standard,43567,21757,15970,1103,208,100.0000,49.9392,36.6562,2.5317,0.4774
9,2022-04-18,standard,24140,11259,8092,499,108,100.0000,46.6404,33.5211,2.0671,0.4474


In [71]:
# Reshaping daily traffic mix for later rate decomposition

daily_traffic_source_mix = (daily_traffic_source_summary
                            .pivot(index = "event_date",
                                   columns = "traffic_source",
                                   values = "traffic_source_share_of_day_percent"
                            )
                            .reset_index()
)

daily_traffic_source_mix.columns.name = None
daily_traffic_source_mix = daily_traffic_source_mix.rename(
    columns = {"random": "random_traffic_share_percent",
               "standard": "standard_traffic_share_percent"
    }
)

daily_traffic_source_mix

,event_date,random_traffic_share_percent,standard_traffic_share_percent
0,2022-04-09,NaN,100.0000
1,2022-04-10,NaN,100.0000
2,2022-04-11,NaN,100.0000
3,2022-04-12,NaN,100.0000
4,2022-04-13,NaN,100.0000
5,2022-04-14,NaN,100.0000
6,2022-04-15,NaN,100.0000
7,2022-04-16,NaN,100.0000
8,2022-04-17,NaN,100.0000
9,2022-04-18,NaN,100.0000


In [72]:
# Setting overall rate benchmarks for segment lift comparisons
overall_rate_benchmark = {"click_rate_percent": round(impression_analysis_base["is_click"].mean() * 100, 4),
                          "meaningful_watch_rate_percent": round(impression_analysis_base["is_meaningful_watch"].mean() * 100, 4),
                          "social_engagement_rate_percent": round(impression_analysis_base["is_social_engagement"].mean() * 100, 4),
                          "high_value_engagement_rate_percent": round(impression_analysis_base["is_high_value_engagement"].mean() * 100, 4)
}

overall_rate_benchmark

{'click_rate_percent': np.float64(33.2883),
 'meaningful_watch_rate_percent': np.float64(22.1414),
 'social_engagement_rate_percent': np.float64(1.4637),
 'high_value_engagement_rate_percent': np.float64(0.2916)}

In [73]:
# Building segment lifts against overall outcome benchmarks

segment_reporting_minimum_impression_count = 1000

def build_segment_rate_comparison(dataframe, segment_fields, minimum_impression_count):
    segment_rate_comparison_list = []

    for field_name in segment_fields:
        segment_summary = (dataframe
                           .groupby(field_name, dropna = False, as_index = False)
                           .agg(impression_count = ("user_id", "size"),
                                click_count = ("is_click", "sum"),
                                meaningful_watch_count = ("is_meaningful_watch", "sum"),
                                social_engagement_count = ("is_social_engagement", "sum"),
                                high_value_engagement_count = ("is_high_value_engagement", "sum")
            )
        )

        segment_summary["click_rate_percent"] = (
            segment_summary["click_count"] / segment_summary["impression_count"] * 100
        ).round(4)

        segment_summary["meaningful_watch_rate_percent"] = (
            segment_summary["meaningful_watch_count"] / segment_summary["impression_count"] * 100
        ).round(4)

        segment_summary["social_engagement_rate_percent"] = (
            segment_summary["social_engagement_count"] / segment_summary["impression_count"] * 100
        ).round(4)

        segment_summary["high_value_engagement_rate_percent"] = (
            segment_summary["high_value_engagement_count"] / segment_summary["impression_count"] * 100
        ).round(4)

        segment_summary["click_rate_lift_vs_overall"] = (
            segment_summary["click_rate_percent"] / overall_rate_benchmark["click_rate_percent"]
        ).round(4)

        segment_summary["meaningful_watch_rate_lift_vs_overall"] = (
            segment_summary["meaningful_watch_rate_percent"] / overall_rate_benchmark["meaningful_watch_rate_percent"]
        ).round(4)

        segment_summary["social_engagement_rate_lift_vs_overall"] = (
            segment_summary["social_engagement_rate_percent"] / overall_rate_benchmark["social_engagement_rate_percent"]
        ).round(4)

        segment_summary["high_value_engagement_rate_lift_vs_overall"] = (
            segment_summary["high_value_engagement_rate_percent"] / overall_rate_benchmark["high_value_engagement_rate_percent"]
        ).round(4)

        segment_summary = (
            segment_summary.loc[segment_summary["impression_count"] >= minimum_impression_count]
            .rename(columns = {field_name: "segment_value"})
            .copy()
        )

        segment_summary.insert(0, "segment_field", field_name)
        segment_rate_comparison_list.append(segment_summary)

    return (pd.concat(segment_rate_comparison_list, ignore_index = True)
            .sort_values(["segment_field", "impression_count"], ascending = [True, False])
            .reset_index(drop = True)
    )

segment_rate_comparison = build_segment_rate_comparison(impression_analysis_base,
                                                        segment_fields,
                                                        segment_reporting_minimum_impression_count
)

segment_rate_comparison

,segment_field,segment_value,impression_count,click_count,meaningful_watch_count,social_engagement_count,high_value_engagement_count,click_rate_percent,meaningful_watch_rate_percent,social_engagement_rate_percent,high_value_engagement_rate_percent,click_rate_lift_vs_overall,meaningful_watch_rate_lift_vs_overall,social_engagement_rate_lift_vs_overall,high_value_engagement_rate_lift_vs_overall
0,fans_user_num_range,"[10,100)",991288,332748,222162,13738,3011,33.5672,22.4114,1.3859,0.3037,1.0084,1.0122,0.9468,1.0415
1,fans_user_num_range,"[100,1k)",875764,285624,189124,15013,2647,32.6143,21.5953,1.7143,0.3023,0.9798,0.9753,1.1712,1.0367
2,fans_user_num_range,"[1,10)",467709,161395,108206,5636,1253,34.5076,23.1353,1.2050,0.2679,1.0366,1.0449,0.8233,0.9187
3,fans_user_num_range,"[1k,5k)",144442,45557,29496,2635,429,31.5400,20.4207,1.8243,0.2970,0.9475,0.9223,1.2464,1.0185
4,fans_user_num_range,0,100515,33904,22538,724,197,33.7303,22.4225,0.7203,0.1960,1.0133,1.0127,0.4921,0.6722
5,fans_user_num_range,"[5k,1w)",9102,2665,1761,112,18,29.2793,19.3474,1.2305,0.1978,0.8796,0.8738,0.8407,0.6783
6,fans_user_num_range,"[1w,10w)",7871,2483,1627,101,18,31.5462,20.6708,1.2832,0.2287,0.9477,0.9336,0.8767,0.7843
7,follow_user_num_range,500+,537421,182101,121245,12338,2378,33.8842,22.5605,2.2958,0.4425,1.0179,1.0189,1.5685,1.5175
8,follow_user_num_range,"(10,50]",479351,159907,106600,4454,1123,33.3591,22.2384,0.9292,0.2343,1.0021,1.0044,0.6348,0.8035
9,follow_user_num_range,"(250,500]",411614,136314,90412,6699,1235,33.1169,21.9652,1.6275,0.3000,0.9949,0.9920,1.1119,1.0288


In [74]:
# Keeping only segments with enough volume for reporting
clicked_impression_reporting_minimum_count = 500

def build_post_click_segment_conversion_comparison(dataframe, segment_fields, minimum_impression_count):
    clicked_dataframe = dataframe.loc[dataframe["is_click"] == 1].copy()
    segment_conversion_list = []

    overall_clicked_meaningful_watch_rate_percent = round(
        clicked_dataframe["is_meaningful_watch"].mean() * 100,
        4
    )
    overall_clicked_profile_enter_rate_percent = round(
        clicked_dataframe["is_profile_enter"].mean() * 100,
        4
    )
    overall_clicked_like_rate_percent = round(
        clicked_dataframe["is_like"].mean() * 100,
        4
    )
    overall_clicked_high_value_engagement_rate_percent = round(
        clicked_dataframe["is_high_value_engagement"].mean() * 100,
        4
    )

    for field_name in segment_fields:
        segment_summary = (clicked_dataframe
                           .groupby(field_name, dropna = False, as_index = False)
                           .agg(clicked_impression_count = ("user_id", "size"),
                                meaningful_watch_count = ("is_meaningful_watch", "sum"),
                                profile_enter_count = ("is_profile_enter", "sum"),
                                like_count = ("is_like", "sum"),
                                high_value_engagement_count = ("is_high_value_engagement", "sum")
            )
        )

        segment_summary["click_to_meaningful_watch_rate_percent"] = (
            segment_summary["meaningful_watch_count"] / segment_summary["clicked_impression_count"] * 100
        ).round(4)

        segment_summary["click_to_profile_enter_rate_percent"] = (
            segment_summary["profile_enter_count"] / segment_summary["clicked_impression_count"] * 100
        ).round(4)

        segment_summary["click_to_like_rate_percent"] = (
            segment_summary["like_count"] / segment_summary["clicked_impression_count"] * 100
        ).round(4)

        segment_summary["click_to_high_value_engagement_rate_percent"] = (
            segment_summary["high_value_engagement_count"] / segment_summary["clicked_impression_count"] * 100
        ).round(4)

        segment_summary["click_to_meaningful_watch_rate_lift_vs_overall"] = (
            segment_summary["click_to_meaningful_watch_rate_percent"] / overall_clicked_meaningful_watch_rate_percent
        ).round(4)

        segment_summary["click_to_profile_enter_rate_lift_vs_overall"] = (
            segment_summary["click_to_profile_enter_rate_percent"] / overall_clicked_profile_enter_rate_percent
        ).round(4)

        segment_summary["click_to_like_rate_lift_vs_overall"] = (
            segment_summary["click_to_like_rate_percent"] / overall_clicked_like_rate_percent
        ).round(4)

        segment_summary["click_to_high_value_engagement_rate_lift_vs_overall"] = (
            segment_summary["click_to_high_value_engagement_rate_percent"] / overall_clicked_high_value_engagement_rate_percent
        ).round(4)

        segment_summary = (
            segment_summary.loc[segment_summary["clicked_impression_count"] >= minimum_impression_count]
            .rename(columns = {field_name: "segment_value"})
            .copy()
        )

        segment_summary.insert(0, "segment_field", field_name)
        segment_conversion_list.append(segment_summary)

    return (
        pd.concat(segment_conversion_list, ignore_index = True)
        .sort_values(["segment_field", "clicked_impression_count"], ascending = [True, False])
        .reset_index(drop = True)
    )

post_click_segment_conversion_comparison = build_post_click_segment_conversion_comparison(
    impression_analysis_base,
    segment_fields,
    clicked_impression_reporting_minimum_count,
)

post_click_segment_conversion_comparison

,segment_field,segment_value,clicked_impression_count,meaningful_watch_count,profile_enter_count,like_count,high_value_engagement_count,click_to_meaningful_watch_rate_percent,click_to_profile_enter_rate_percent,click_to_like_rate_percent,click_to_high_value_engagement_rate_percent,click_to_meaningful_watch_rate_lift_vs_overall,click_to_profile_enter_rate_lift_vs_overall,click_to_like_rate_lift_vs_overall,click_to_high_value_engagement_rate_lift_vs_overall
0,fans_user_num_range,"[10,100)",332748,221285,13033,9266,2789,66.5023,3.9168,2.7847,0.8382,1.0038,0.9227,0.9347,1.0441
1,fans_user_num_range,"[100,1k)",285624,188320,14048,10177,2391,65.9328,4.9184,3.5631,0.8371,0.9952,1.1587,1.1959,1.0427
2,fans_user_num_range,"[1,10)",161395,107816,5864,3860,1148,66.8026,3.6333,2.3916,0.7113,1.0083,0.8559,0.8027,0.8860
3,fans_user_num_range,"[1k,5k)",45557,29368,2348,1793,400,64.4643,5.1540,3.9357,0.8780,0.9730,1.2142,1.3210,1.0937
4,fans_user_num_range,0,33904,22473,1155,455,176,66.2842,3.4067,1.3420,0.5191,1.0005,0.8025,0.4504,0.6466
5,fans_user_num_range,"[5k,1w)",2665,1750,98,87,18,65.6660,3.6773,3.2645,0.6754,0.9912,0.8663,1.0957,0.8413
6,fans_user_num_range,"[1w,10w)",2483,1620,125,67,18,65.2437,5.0342,2.6983,0.7249,0.9848,1.1859,0.9057,0.9030
7,follow_user_num_range,500+,182101,120722,8024,8625,2105,66.2940,4.4063,4.7364,1.1560,1.0007,1.0380,1.5897,1.4400
8,follow_user_num_range,"(10,50]",159907,106197,6419,2725,1030,66.4117,4.0142,1.7041,0.6441,1.0024,0.9457,0.5720,0.8023
9,follow_user_num_range,"(250,500]",136314,90065,6048,4634,1164,66.0717,4.4368,3.3995,0.8539,0.9973,1.0452,1.1410,1.0637


In [75]:
# Ranking segments by high-value engagement potential

segment_priority_view = (
    segment_rate_comparison[
        ["segment_field", "segment_value", "impression_count", 
         "click_rate_percent", "click_rate_lift_vs_overall",
         "meaningful_watch_rate_percent", "meaningful_watch_rate_lift_vs_overall", 
         "high_value_engagement_rate_percent",
         "high_value_engagement_rate_lift_vs_overall"
        ]
    ]
    .sort_values(["high_value_engagement_rate_lift_vs_overall", "impression_count"],
                 ascending = [False, False]
    )
    .reset_index(drop = True)
)

segment_priority_view

,segment_field,segment_value,impression_count,click_rate_percent,click_rate_lift_vs_overall,meaningful_watch_rate_percent,meaningful_watch_rate_lift_vs_overall,high_value_engagement_rate_percent,high_value_engagement_rate_lift_vs_overall
0,music_type,11.0,1167,22.7078,0.6822,9.8543,0.4451,0.9426,3.2325
1,user_active_degree,30day_retention,1228,34.0391,1.0226,21.9055,0.9893,0.7329,2.5134
2,user_active_degree,single_low_active,4481,40.1473,1.2060,29.3015,1.3234,0.5356,1.8368
3,register_days_range,31-60,32011,37.3153,1.1210,25.2538,1.1406,0.4780,1.6392
4,register_days_range,91-180,97700,36.0297,1.0824,24.2815,1.0967,0.4504,1.5446
5,user_active_degree,day_new,1111,39.3339,1.1816,27.0027,1.2196,0.4500,1.5432
6,follow_user_num_range,500+,537421,33.8842,1.0179,22.5605,1.0189,0.4425,1.5175
7,register_days_range,181-365,179607,36.7914,1.1052,24.8259,1.1212,0.4376,1.5007
8,user_active_degree,middle_active,166902,37.1895,1.1172,24.7307,1.1169,0.4176,1.4321
9,video_orientation,square,38328,27.0533,0.8127,15.8474,0.7157,0.4122,1.4136


In [76]:
# Establishing traffic-source baselines for fair comparisons
traffic_source_benchmark = (impression_analysis_base
                            .groupby("traffic_source", as_index = False)
                            .agg(impression_count = ("user_id", "size"),
                                 click_rate = ("is_click", "mean"),
                                 meaningful_watch_rate = ("is_meaningful_watch", "mean"),
                                 social_engagement_rate = ("is_social_engagement", "mean"),
                                 high_value_engagement_rate = ("is_high_value_engagement", "mean")
    )
)

traffic_source_benchmark["click_rate_percent"] = (
    traffic_source_benchmark["click_rate"] * 100
).round(4)

traffic_source_benchmark["meaningful_watch_rate_percent"] = (
    traffic_source_benchmark["meaningful_watch_rate"] * 100
).round(4)

traffic_source_benchmark["social_engagement_rate_percent"] = (
    traffic_source_benchmark["social_engagement_rate"] * 100
).round(4)

traffic_source_benchmark["high_value_engagement_rate_percent"] = (
    traffic_source_benchmark["high_value_engagement_rate"] * 100
).round(4)

traffic_source_benchmark

,traffic_source,impression_count,click_rate,meaningful_watch_rate,social_engagement_rate,high_value_engagement_rate,click_rate_percent,meaningful_watch_rate_percent,social_engagement_rate_percent,high_value_engagement_rate_percent
0,random,1186049,0.17616,0.084962,0.005574,0.000942,17.616,8.4962,0.5574,0.0942
1,standard,1411650,0.46456,0.336059,0.022252,0.004575,46.456,33.6059,2.2252,0.4575


In [77]:
# Separating daily rate shifts from traffic-mix effects

daily_rate_decomposition = daily_funnel_summary.merge(daily_traffic_source_mix,
                                                      on = "event_date",
                                                      how = "left"
)

daily_rate_decomposition["random_traffic_share_percent"] = (
    daily_rate_decomposition["random_traffic_share_percent"].fillna(0)
)

daily_rate_decomposition["standard_traffic_share_percent"] = (
    daily_rate_decomposition["standard_traffic_share_percent"].fillna(100)
)

traffic_source_rate_lookup = (traffic_source_benchmark
                              .set_index("traffic_source")[
                                    ["click_rate", "meaningful_watch_rate", "social_engagement_rate", "high_value_engagement_rate"]
    ]
    .to_dict("index")
)

daily_rate_decomposition["expected_click_rate_from_traffic_mix_percent"] = (
    daily_rate_decomposition["standard_traffic_share_percent"] / 100 * traffic_source_rate_lookup["standard"]["click_rate"]
    + daily_rate_decomposition["random_traffic_share_percent"] / 100 * traffic_source_rate_lookup["random"]["click_rate"]
) * 100

daily_rate_decomposition["expected_meaningful_watch_rate_from_traffic_mix_percent"] = (
    daily_rate_decomposition["standard_traffic_share_percent"] / 100 * traffic_source_rate_lookup["standard"]["meaningful_watch_rate"]
    + daily_rate_decomposition["random_traffic_share_percent"] / 100 * traffic_source_rate_lookup["random"]["meaningful_watch_rate"]
) * 100

daily_rate_decomposition["expected_social_engagement_rate_from_traffic_mix_percent"] = (
    daily_rate_decomposition["standard_traffic_share_percent"] / 100 * traffic_source_rate_lookup["standard"]["social_engagement_rate"]
    + daily_rate_decomposition["random_traffic_share_percent"] / 100 * traffic_source_rate_lookup["random"]["social_engagement_rate"]
) * 100

daily_rate_decomposition["expected_high_value_engagement_rate_from_traffic_mix_percent"] = (
    daily_rate_decomposition["standard_traffic_share_percent"] / 100 * traffic_source_rate_lookup["standard"]["high_value_engagement_rate"]
    + daily_rate_decomposition["random_traffic_share_percent"] / 100 * traffic_source_rate_lookup["random"]["high_value_engagement_rate"]
) * 100

daily_rate_decomposition["click_rate_gap_vs_mix_expectation_percent_points"] = (
    daily_rate_decomposition["click_rate_percent"] - daily_rate_decomposition["expected_click_rate_from_traffic_mix_percent"]
).round(4)

daily_rate_decomposition["meaningful_watch_rate_gap_vs_mix_expectation_percent_points"] = (
    daily_rate_decomposition["meaningful_watch_rate_percent"] - daily_rate_decomposition["expected_meaningful_watch_rate_from_traffic_mix_percent"]
).round(4)

daily_rate_decomposition["social_engagement_rate_gap_vs_mix_expectation_percent_points"] = (
    daily_rate_decomposition["social_engagement_rate_percent"] - daily_rate_decomposition["expected_social_engagement_rate_from_traffic_mix_percent"]
).round(4)

daily_rate_decomposition["high_value_engagement_rate_gap_vs_mix_expectation_percent_points"] = (
    daily_rate_decomposition["high_value_engagement_rate_percent"] - daily_rate_decomposition["expected_high_value_engagement_rate_from_traffic_mix_percent"]
).round(4)

daily_rate_decomposition[
    [
        "event_date",
        "impression_count",
        "standard_traffic_share_percent",
        "random_traffic_share_percent",
        "click_rate_percent",
        "expected_click_rate_from_traffic_mix_percent",
        "click_rate_gap_vs_mix_expectation_percent_points",
        "meaningful_watch_rate_percent",
        "expected_meaningful_watch_rate_from_traffic_mix_percent",
        "meaningful_watch_rate_gap_vs_mix_expectation_percent_points",
        "social_engagement_rate_percent",
        "expected_social_engagement_rate_from_traffic_mix_percent",
        "social_engagement_rate_gap_vs_mix_expectation_percent_points",
        "high_value_engagement_rate_percent",
        "expected_high_value_engagement_rate_from_traffic_mix_percent",
        "high_value_engagement_rate_gap_vs_mix_expectation_percent_points"
    ]
]

,event_date,impression_count,standard_traffic_share_percent,random_traffic_share_percent,click_rate_percent,expected_click_rate_from_traffic_mix_percent,click_rate_gap_vs_mix_expectation_percent_points,meaningful_watch_rate_percent,expected_meaningful_watch_rate_from_traffic_mix_percent,meaningful_watch_rate_gap_vs_mix_expectation_percent_points,social_engagement_rate_percent,expected_social_engagement_rate_from_traffic_mix_percent,social_engagement_rate_gap_vs_mix_expectation_percent_points,high_value_engagement_rate_percent,expected_high_value_engagement_rate_from_traffic_mix_percent,high_value_engagement_rate_gap_vs_mix_expectation_percent_points
0,2022-04-09,52390,100.0000,0.0000,44.8750,46.455991,-1.5810,33.7793,33.605922,0.1734,2.0691,2.225197,-0.1561,0.4066,0.457479,-0.0509
1,2022-04-10,226043,100.0000,0.0000,46.9004,46.455991,0.4444,34.2537,33.605922,0.6478,2.2390,2.225197,0.0138,0.4260,0.457479,-0.0315
2,2022-04-11,275743,100.0000,0.0000,46.0726,46.455991,-0.3834,33.5631,33.605922,-0.0428,2.1270,2.225197,-0.0982,0.4540,0.457479,-0.0035
3,2022-04-12,162146,100.0000,0.0000,46.2676,46.455991,-0.1884,33.8164,33.605922,0.2105,2.1715,2.225197,-0.0537,0.4699,0.457479,0.0124
4,2022-04-13,91714,100.0000,0.0000,45.4140,46.455991,-1.0420,32.4760,33.605922,-1.1299,2.2178,2.225197,-0.0074,0.4645,0.457479,0.0070
5,2022-04-14,68971,100.0000,0.0000,45.7453,46.455991,-0.7107,32.7688,33.605922,-0.8371,2.1705,2.225197,-0.0547,0.4799,0.457479,0.0224
6,2022-04-15,58239,100.0000,0.0000,50.5040,46.455991,4.0480,36.6078,33.605922,3.0019,2.6958,2.225197,0.4706,0.5134,0.457479,0.0559
7,2022-04-16,60538,100.0000,0.0000,51.7163,46.455991,5.2603,37.8440,33.605922,4.2381,2.8247,2.225197,0.5995,0.4972,0.457479,0.0397
8,2022-04-17,43567,100.0000,0.0000,49.9392,46.455991,3.4832,36.6562,33.605922,3.0503,2.5317,2.225197,0.3065,0.4774,0.457479,0.0199
9,2022-04-18,24140,100.0000,0.0000,46.6404,46.455991,0.1844,33.5211,33.605922,-0.0848,2.0671,2.225197,-0.1581,0.4474,0.457479,-0.0101


In [78]:
# Calculating uncertainty bounds for segment-level rate comparisons

def wilson_interval(success_count, total_count, z_value = 1.96):
    if total_count == 0:
        return np.nan, np.nan

    proportion = success_count / total_count
    denominator = 1 + (z_value ** 2) / total_count
    center = (proportion + (z_value ** 2) / (2 * total_count)) / denominator
    margin = (z_value * np.sqrt((
                                    proportion * (1 - proportion) + (z_value ** 2) / (4 * total_count)
                                ) / total_count
        )
    ) / denominator

    return center, margin


def build_segment_rate_comparison_within_traffic_source(dataframe, segment_fields, minimum_impression_count):
    traffic_source_baseline = (dataframe
                               .groupby("traffic_source", as_index = False)
                               .agg(traffic_source_impression_count = ("user_id", "size"),
                                    traffic_source_click_rate = ("is_click", "mean"),
                                    traffic_source_meaningful_watch_rate = ("is_meaningful_watch", "mean"),
                                    traffic_source_social_engagement_rate = ("is_social_engagement", "mean"),
                                    traffic_source_high_value_engagement_rate = ("is_high_value_engagement", "mean")
        )
    )

    comparison_tables = []

    for field_name in segment_fields:
        segment_summary = (dataframe
                           .groupby(["traffic_source", field_name], dropna = False, as_index = False)
                           .agg(impression_count = ("user_id", "size"),
                                click_count = ("is_click", "sum"),
                                meaningful_watch_count = ("is_meaningful_watch", "sum"),
                                social_engagement_count = ("is_social_engagement", "sum"),
                                high_value_engagement_count = ("is_high_value_engagement", "sum")
            )
            .rename(columns = {field_name: "segment_value"})
        )

        segment_summary = segment_summary.loc[
            segment_summary["impression_count"] >= minimum_impression_count
        ].copy()

        segment_summary["click_rate_percent"] = (
            segment_summary["click_count"] / segment_summary["impression_count"] * 100
        ).round(4)

        segment_summary["meaningful_watch_rate_percent"] = (
            segment_summary["meaningful_watch_count"] / segment_summary["impression_count"] * 100
        ).round(4)

        segment_summary["social_engagement_rate_percent"] = (
            segment_summary["social_engagement_count"] / segment_summary["impression_count"] * 100
        ).round(4)

        segment_summary["high_value_engagement_rate_percent"] = (
            segment_summary["high_value_engagement_count"] / segment_summary["impression_count"] * 100
        ).round(4)

        click_interval = segment_summary.apply(
            lambda row: wilson_interval(row["click_count"], row["impression_count"]),
            axis = 1
        )

        high_value_interval = segment_summary.apply(
            lambda row: wilson_interval(row["high_value_engagement_count"], row["impression_count"]),
            axis = 1
        )

        segment_summary["click_rate_center"] = [interval[0] for interval in click_interval]
        segment_summary["click_rate_margin"] = [interval[1] for interval in click_interval]
        segment_summary["high_value_engagement_rate_center"] = [interval[0] for interval in high_value_interval]
        segment_summary["high_value_engagement_rate_margin"] = [interval[1] for interval in high_value_interval]

        segment_summary["click_rate_lower_bound_percent"] = (
            (segment_summary["click_rate_center"] - segment_summary["click_rate_margin"]) * 100
        ).round(4)

        segment_summary["click_rate_upper_bound_percent"] = (
            (segment_summary["click_rate_center"] + segment_summary["click_rate_margin"]) * 100
        ).round(4)

        segment_summary["high_value_engagement_rate_lower_bound_percent"] = (
            (
                segment_summary["high_value_engagement_rate_center"] - segment_summary["high_value_engagement_rate_margin"]
            ) * 100
        ).round(4)

        segment_summary["high_value_engagement_rate_upper_bound_percent"] = (
            (
                segment_summary["high_value_engagement_rate_center"] + segment_summary["high_value_engagement_rate_margin"]
            ) * 100
        ).round(4)

        segment_summary = segment_summary.merge(traffic_source_baseline,
                                                on = "traffic_source", 
                                                how = "left"
        )

        segment_summary["traffic_source_click_rate_percent"] = (
            segment_summary["traffic_source_click_rate"] * 100
        ).round(4)

        segment_summary["traffic_source_high_value_engagement_rate_percent"] = (
            segment_summary["traffic_source_high_value_engagement_rate"] * 100
        ).round(4)

        segment_summary["click_rate_lift_within_traffic_source"] = (
            segment_summary["click_rate_percent"] / segment_summary["traffic_source_click_rate_percent"]
        ).round(4)

        segment_summary["high_value_engagement_rate_lift_within_traffic_source"] = (
            segment_summary["high_value_engagement_rate_percent"] / segment_summary["traffic_source_high_value_engagement_rate_percent"]
        ).round(4)

        segment_summary["click_rate_confidently_above_traffic_source"] = (
            segment_summary["click_rate_lower_bound_percent"] > segment_summary["traffic_source_click_rate_percent"]
        )

        segment_summary["high_value_engagement_rate_confidently_above_traffic_source"] = (
            segment_summary["high_value_engagement_rate_lower_bound_percent"] > segment_summary["traffic_source_high_value_engagement_rate_percent"]
        )

        segment_summary.insert(0, "segment_field", field_name)
        comparison_tables.append(segment_summary)

    return (pd.concat(comparison_tables, ignore_index = True)
            .sort_values(["segment_field", "traffic_source", "impression_count"],
                         ascending = [True, True, False]
        )
        .reset_index(drop = True)
    )


segment_rate_comparison_within_traffic_source = build_segment_rate_comparison_within_traffic_source(
    impression_analysis_base,
    segment_fields,
    minimum_impression_count = 1000
)

segment_rate_comparison_within_traffic_source[
    [
        "segment_field", 
        "traffic_source",
        "segment_value",
        "impression_count",
        "click_rate_percent",
        "traffic_source_click_rate_percent",
        "click_rate_lift_within_traffic_source",
        "click_rate_lower_bound_percent",
        "click_rate_upper_bound_percent",
        "click_rate_confidently_above_traffic_source",
        "high_value_engagement_rate_percent",
        "traffic_source_high_value_engagement_rate_percent",
        "high_value_engagement_rate_lift_within_traffic_source",
        "high_value_engagement_rate_lower_bound_percent",
        "high_value_engagement_rate_upper_bound_percent",
        "high_value_engagement_rate_confidently_above_traffic_source"
    ]
]

,segment_field,traffic_source,segment_value,impression_count,click_rate_percent,traffic_source_click_rate_percent,click_rate_lift_within_traffic_source,click_rate_lower_bound_percent,click_rate_upper_bound_percent,click_rate_confidently_above_traffic_source,high_value_engagement_rate_percent,traffic_source_high_value_engagement_rate_percent,high_value_engagement_rate_lift_within_traffic_source,high_value_engagement_rate_lower_bound_percent,high_value_engagement_rate_upper_bound_percent,high_value_engagement_rate_confidently_above_traffic_source
0,fans_user_num_range,random,"[10,100)",457791,17.5471,17.616,0.9961,17.4372,17.6575,False,0.1044,0.0942,1.1083,0.0955,0.1142,True
1,fans_user_num_range,random,"[100,1k)",380239,17.5455,17.616,0.9960,17.4250,17.6668,False,0.0934,0.0942,0.9915,0.0841,0.1036,False
2,fans_user_num_range,random,"[1,10)",225066,18.0680,17.616,1.0257,17.9096,18.2275,True,0.0942,0.0942,1.0000,0.0823,0.1077,False
3,fans_user_num_range,random,"[1k,5k)",65279,17.2659,17.616,0.9801,16.9779,17.5578,False,0.0797,0.0942,0.8461,0.0608,0.1044,False
4,fans_user_num_range,random,0,49483,17.7758,17.616,1.0091,17.4415,18.1152,False,0.0384,0.0942,0.4076,0.0246,0.0600,False
5,fans_user_num_range,random,"[5k,1w)",4198,12.1486,17.616,0.6896,11.1948,13.1717,False,0.0238,0.0942,0.2527,0.0042,0.1348,False
6,fans_user_num_range,random,"[1w,10w)",3562,15.7496,17.616,0.8941,14.5903,16.9827,False,0.0000,0.0942,0.0000,0.0000,0.1077,False
7,fans_user_num_range,standard,"[10,100)",533497,47.3140,46.456,1.0185,47.1801,47.4480,True,0.4748,0.4575,1.0378,0.4567,0.4936,False
8,fans_user_num_range,standard,"[100,1k)",495525,44.1772,46.456,0.9509,44.0390,44.3155,False,0.4625,0.4575,1.0109,0.4440,0.4818,False
9,fans_user_num_range,standard,"[1,10)",242643,49.7562,46.456,1.0710,49.5573,49.9552,True,0.4290,0.4575,0.9377,0.4038,0.4558,False


In [79]:
# Comparing post-click segment performance within each traffic source

def build_post_click_segment_conversion_within_traffic_source(dataframe, segment_fields, minimum_clicked_impression_count):
    clicked_dataframe = dataframe.loc[dataframe["is_click"] == 1].copy()

    traffic_source_post_click_baseline = (clicked_dataframe
                                          .groupby("traffic_source", as_index = False)
                                          .agg(clicked_impression_count_total = ("user_id", "size"),
                                               traffic_source_click_to_meaningful_watch_rate = ("is_meaningful_watch", "mean"),
                                               traffic_source_click_to_profile_enter_rate = ("is_profile_enter", "mean"),
                                               traffic_source_click_to_like_rate = ("is_like", "mean"),
                                               traffic_source_click_to_high_value_engagement_rate = ("is_high_value_engagement", "mean")
        )
    )

    comparison_tables = []

    for field_name in segment_fields:
        segment_summary = (
            clicked_dataframe
            .groupby(["traffic_source", field_name], dropna = False, as_index = False)
            .agg(clicked_impression_count = ("user_id", "size"),
                 meaningful_watch_count = ("is_meaningful_watch", "sum"),
                 profile_enter_count = ("is_profile_enter", "sum"),
                 like_count = ("is_like", "sum"),
                 high_value_engagement_count = ("is_high_value_engagement", "sum")
            )
            .rename(columns = {field_name: "segment_value"})
        )

        segment_summary = segment_summary.loc[
            segment_summary["clicked_impression_count"] >= minimum_clicked_impression_count
        ].copy()

        segment_summary["click_to_meaningful_watch_rate_percent"] = (
            segment_summary["meaningful_watch_count"] / segment_summary["clicked_impression_count"] * 100
        ).round(4)

        segment_summary["click_to_profile_enter_rate_percent"] = (
            segment_summary["profile_enter_count"] / segment_summary["clicked_impression_count"] * 100
        ).round(4)

        segment_summary["click_to_like_rate_percent"] = (
            segment_summary["like_count"] / segment_summary["clicked_impression_count"] * 100
        ).round(4)

        segment_summary["click_to_high_value_engagement_rate_percent"] = (
            segment_summary["high_value_engagement_count"] / segment_summary["clicked_impression_count"] * 100
        ).round(4)

        # Estimating confidence bounds for post-click high-value conversion
        high_value_interval = segment_summary.apply(
            lambda row: wilson_interval(
                row["high_value_engagement_count"],
                row["clicked_impression_count"]
            ),
            axis = 1
        )

        segment_summary["click_to_high_value_engagement_rate_center"] = [
            interval[0] for interval in high_value_interval
        ]
        segment_summary["click_to_high_value_engagement_rate_margin"] = [
            interval[1] for interval in high_value_interval
        ]

        segment_summary["click_to_high_value_engagement_rate_lower_bound_percent"] = (
            (
                segment_summary["click_to_high_value_engagement_rate_center"] - segment_summary["click_to_high_value_engagement_rate_margin"]
            ) * 100
        ).round(4)

        segment_summary["click_to_high_value_engagement_rate_upper_bound_percent"] = (
            (
                segment_summary["click_to_high_value_engagement_rate_center"] + segment_summary["click_to_high_value_engagement_rate_margin"]
            ) * 100
        ).round(4)

        segment_summary = segment_summary.merge(traffic_source_post_click_baseline,
                                                on = "traffic_source",
                                                how = "left"
        )

        segment_summary["traffic_source_click_to_high_value_engagement_rate_percent"] = (
            segment_summary["traffic_source_click_to_high_value_engagement_rate"] * 100
        ).round(4)

        segment_summary["click_to_high_value_engagement_rate_lift_within_traffic_source"] = (
            segment_summary["click_to_high_value_engagement_rate_percent"] 
            / segment_summary["traffic_source_click_to_high_value_engagement_rate_percent"]
        ).round(4)

        segment_summary["click_to_high_value_engagement_confidently_above_traffic_source"] = (
            segment_summary["click_to_high_value_engagement_rate_lower_bound_percent"] 
            > segment_summary["traffic_source_click_to_high_value_engagement_rate_percent"]
        )

        segment_summary.insert(0, "segment_field", field_name)
        comparison_tables.append(segment_summary)

    return (pd.concat(comparison_tables, ignore_index = True)
            .sort_values(["segment_field", "traffic_source", "clicked_impression_count"],
                         ascending = [True, True, False]
            )
            .reset_index(drop = True)
    )


post_click_segment_conversion_within_traffic_source = build_post_click_segment_conversion_within_traffic_source(
    impression_analysis_base,
    segment_fields,
    minimum_clicked_impression_count = 500
)

post_click_segment_conversion_within_traffic_source[
    [
        "segment_field",
        "traffic_source",
        "segment_value",
        "clicked_impression_count",
        "click_to_meaningful_watch_rate_percent",
        "click_to_profile_enter_rate_percent",
        "click_to_like_rate_percent",
        "click_to_high_value_engagement_rate_percent",
        "traffic_source_click_to_high_value_engagement_rate_percent",
        "click_to_high_value_engagement_rate_lift_within_traffic_source",
        "click_to_high_value_engagement_rate_lower_bound_percent",
        "click_to_high_value_engagement_rate_upper_bound_percent",
        "click_to_high_value_engagement_confidently_above_traffic_source"
    ]
]

,segment_field,traffic_source,segment_value,clicked_impression_count,click_to_meaningful_watch_rate_percent,click_to_profile_enter_rate_percent,click_to_like_rate_percent,click_to_high_value_engagement_rate_percent,traffic_source_click_to_high_value_engagement_rate_percent,click_to_high_value_engagement_rate_lift_within_traffic_source,click_to_high_value_engagement_rate_lower_bound_percent,click_to_high_value_engagement_rate_upper_bound_percent,click_to_high_value_engagement_confidently_above_traffic_source
0,fans_user_num_range,random,"[10,100)",80329,48.2839,1.9022,1.5785,0.4880,0.4351,1.1216,0.4421,0.5386,True
1,fans_user_num_range,random,"[100,1k)",66715,48.1826,2.2634,2.0355,0.4392,0.4351,1.0094,0.3918,0.4923,False
2,fans_user_num_range,random,"[1,10)",40665,47.8274,1.8075,1.3919,0.4131,0.4351,0.9494,0.3553,0.4803,False
3,fans_user_num_range,random,"[1k,5k)",11271,47.0411,2.6173,1.8543,0.3549,0.4351,0.8157,0.2607,0.4829,False
4,fans_user_num_range,random,0,8796,47.4989,1.8986,0.7390,0.1705,0.4351,0.3919,0.1034,0.2812,False
5,fans_user_num_range,random,"[1w,10w)",561,47.0588,4.4563,1.0695,0.0000,0.4351,0.0000,0.0000,0.6801,False
6,fans_user_num_range,random,"[5k,1w)",510,45.8824,2.7451,0.5882,0.1961,0.4351,0.4507,0.0346,1.1022,False
7,fans_user_num_range,standard,"[10,100)",252419,72.3000,4.5579,3.1685,0.9496,0.9200,1.0322,0.9125,0.9882,False
8,fans_user_num_range,standard,"[100,1k)",218909,71.3424,5.7275,4.0286,0.9584,0.9200,1.0417,0.9184,1.0001,False
9,fans_user_num_range,standard,"[1,10)",120730,73.1939,4.2483,2.7284,0.8117,0.9200,0.8823,0.7627,0.8639,False


In [80]:
# Filtering pre-click priority segments to credible opportunities

minimum_priority_impression_count = 10000

pre_click_priority_segments = (
    segment_rate_comparison_within_traffic_source[
        [
            "segment_field",
            "traffic_source",
            "segment_value",
            "impression_count",
            "click_rate_percent",
            "traffic_source_click_rate_percent",
            "click_rate_lift_within_traffic_source",
            "click_rate_lower_bound_percent",
            "click_rate_upper_bound_percent",
            "click_rate_confidently_above_traffic_source",
            "high_value_engagement_rate_percent",
            "traffic_source_high_value_engagement_rate_percent",
            "high_value_engagement_rate_lift_within_traffic_source",
            "high_value_engagement_rate_lower_bound_percent",
            "high_value_engagement_rate_upper_bound_percent",
            "high_value_engagement_rate_confidently_above_traffic_source"
        ]
    ]
    .loc[lambda frame: frame["impression_count"] >= minimum_priority_impression_count]
    .loc[
        lambda frame:
        frame["click_rate_confidently_above_traffic_source"] | frame["high_value_engagement_rate_confidently_above_traffic_source"]
    ]
    .sort_values(
        [
            "high_value_engagement_rate_lift_within_traffic_source",
            "click_rate_lift_within_traffic_source",
            "impression_count"
        ],
        ascending = [False, False, False]
    )
    .reset_index(drop = True)
)

pre_click_priority_segments

,segment_field,traffic_source,segment_value,impression_count,click_rate_percent,traffic_source_click_rate_percent,click_rate_lift_within_traffic_source,click_rate_lower_bound_percent,click_rate_upper_bound_percent,click_rate_confidently_above_traffic_source,high_value_engagement_rate_percent,traffic_source_high_value_engagement_rate_percent,high_value_engagement_rate_lift_within_traffic_source,high_value_engagement_rate_lower_bound_percent,high_value_engagement_rate_upper_bound_percent,high_value_engagement_rate_confidently_above_traffic_source
0,register_days_range,random,31-60,15095,20.2915,17.616,1.1519,19.6575,20.9406,True,0.1590,0.0942,1.6879,0.1069,0.2365,True
1,register_days_range,standard,31-60,16916,52.5065,46.456,1.1302,51.7535,53.2584,True,0.7626,0.4575,1.6669,0.6422,0.9053,True
2,register_days_range,standard,91-180,49566,52.3423,46.456,1.1267,51.9025,52.7818,True,0.7485,0.4575,1.6361,0.6763,0.8283,True
3,follow_user_num_range,standard,500+,279008,48.1610,46.456,1.0367,47.9756,48.3464,True,0.7215,0.4575,1.5770,0.6908,0.7536,True
4,user_active_degree,standard,middle_active,84057,54.3679,46.456,1.1703,54.0310,54.7044,True,0.7209,0.4575,1.5757,0.6660,0.7804,True
5,register_days_range,standard,61-90,16879,54.5293,46.456,1.1738,53.7771,55.2794,True,0.7109,0.4575,1.5539,0.5949,0.8494,True
6,register_days_range,standard,181-365,95789,52.5071,46.456,1.1303,52.1907,52.8232,True,0.7036,0.4575,1.5379,0.6526,0.7586,True
7,register_days_range,random,91-180,48134,19.2317,17.616,1.0917,18.8821,19.5863,True,0.1433,0.0942,1.5212,0.1133,0.1814,True
8,follow_user_num_range,random,500+,258413,18.4697,17.616,1.0485,18.3205,18.6197,True,0.1412,0.0942,1.4989,0.1275,0.1565,True
9,video_orientation,standard,square,21537,38.9562,46.456,0.8386,38.3070,39.6094,False,0.6640,0.4575,1.4514,0.5640,0.7816,True


In [81]:
# Filtering post-click priority segments to credible opportunities

minimum_priority_clicked_impression_count = 3000

post_click_priority_segments = (
    post_click_segment_conversion_within_traffic_source[
        [
            "segment_field",
            "traffic_source",
            "segment_value",
            "clicked_impression_count",
            "click_to_meaningful_watch_rate_percent",
            "click_to_profile_enter_rate_percent",
            "click_to_like_rate_percent",
            "click_to_high_value_engagement_rate_percent",
            "traffic_source_click_to_high_value_engagement_rate_percent",
            "click_to_high_value_engagement_rate_lift_within_traffic_source",
            "click_to_high_value_engagement_rate_lower_bound_percent",
            "click_to_high_value_engagement_rate_upper_bound_percent",
            "click_to_high_value_engagement_confidently_above_traffic_source"
        ]
    ]
    .loc[lambda frame: frame["clicked_impression_count"] >= minimum_priority_clicked_impression_count]
    .loc[lambda frame: frame["click_to_high_value_engagement_confidently_above_traffic_source"]]
    .sort_values(
        [
            "click_to_high_value_engagement_rate_lift_within_traffic_source",
            "clicked_impression_count"
        ],
        ascending = [False, False]
    )
    .reset_index(drop = True)
)

post_click_priority_segments

,segment_field,traffic_source,segment_value,clicked_impression_count,click_to_meaningful_watch_rate_percent,click_to_profile_enter_rate_percent,click_to_like_rate_percent,click_to_high_value_engagement_rate_percent,traffic_source_click_to_high_value_engagement_rate_percent,click_to_high_value_engagement_rate_lift_within_traffic_source,click_to_high_value_engagement_rate_lower_bound_percent,click_to_high_value_engagement_rate_upper_bound_percent,click_to_high_value_engagement_confidently_above_traffic_source
0,video_upload_type,standard,LongPicture,4993,32.0449,5.4076,11.2157,2.2632,0.9200,2.4600,1.8859,2.7139,True
1,video_orientation,standard,square,8390,61.6091,5.0775,8.8439,1.5375,0.9200,1.6712,1.2956,1.8239,True
2,follow_user_num_range,standard,500+,134373,72.4669,5.1796,5.4460,1.3485,0.9200,1.4658,1.2882,1.4116,True
3,register_days_range,standard,91-180,25944,73.8784,4.3401,4.2977,1.3452,0.9200,1.4622,1.2121,1.4928,True
4,register_days_range,random,91-180,9257,48.1906,2.0309,1.9553,0.6266,0.4351,1.4401,0.4850,0.8090,True
5,register_days_range,standard,31-60,8882,73.8235,4.3909,4.2333,1.3173,0.9200,1.4318,1.1003,1.5763,True
6,register_days_range,random,181-365,15784,47.9726,2.0084,2.1541,0.6209,0.4351,1.4270,0.5098,0.7560,True
7,follow_user_num_range,random,500+,47728,48.9147,2.2293,2.7384,0.6139,0.4351,1.4109,0.5477,0.6881,True
8,register_days_range,standard,181-365,50296,73.2881,4.6505,4.9785,1.2884,0.9200,1.4004,1.1935,1.3907,True
9,register_days_range,standard,61-90,9204,74.2286,4.2156,2.9770,1.2603,0.9200,1.3699,1.0519,1.5094,True


In [82]:
# Combining pre-click and post-click opportunities into one view

priority_segment_opportunities = (
    pre_click_priority_segments[
        [
            "segment_field",
            "traffic_source",
            "segment_value",
            "impression_count",
            "click_rate_percent",
            "click_rate_lift_within_traffic_source",
            "high_value_engagement_rate_percent",
            "high_value_engagement_rate_lift_within_traffic_source",
            "click_rate_confidently_above_traffic_source",
            "high_value_engagement_rate_confidently_above_traffic_source"
        ]
    ]
    .merge(
        post_click_priority_segments[
            [
                "segment_field",
                "traffic_source",
                "segment_value",
                "clicked_impression_count",
                "click_to_high_value_engagement_rate_percent",
                "click_to_high_value_engagement_rate_lift_within_traffic_source"
            ]
        ],
        on = ["segment_field", "traffic_source", "segment_value"],
        how = "left"
    )
    .sort_values(
        [
            "high_value_engagement_rate_lift_within_traffic_source",
            "click_rate_lift_within_traffic_source",
            "impression_count"
        ],
        ascending = [False, False, False]
    )
    .reset_index(drop = True)
)

priority_segment_opportunities

,segment_field,traffic_source,segment_value,impression_count,click_rate_percent,click_rate_lift_within_traffic_source,high_value_engagement_rate_percent,high_value_engagement_rate_lift_within_traffic_source,click_rate_confidently_above_traffic_source,high_value_engagement_rate_confidently_above_traffic_source,clicked_impression_count,click_to_high_value_engagement_rate_percent,click_to_high_value_engagement_rate_lift_within_traffic_source
0,register_days_range,random,31-60,15095,20.2915,1.1519,0.1590,1.6879,True,True,NaN,NaN,NaN
1,register_days_range,standard,31-60,16916,52.5065,1.1302,0.7626,1.6669,True,True,8882.0,1.3173,1.4318
2,register_days_range,standard,91-180,49566,52.3423,1.1267,0.7485,1.6361,True,True,25944.0,1.3452,1.4622
3,follow_user_num_range,standard,500+,279008,48.1610,1.0367,0.7215,1.5770,True,True,134373.0,1.3485,1.4658
4,user_active_degree,standard,middle_active,84057,54.3679,1.1703,0.7209,1.5757,True,True,45700.0,1.2276,1.3343
5,register_days_range,standard,61-90,16879,54.5293,1.1738,0.7109,1.5539,True,True,9204.0,1.2603,1.3699
6,register_days_range,standard,181-365,95789,52.5071,1.1303,0.7036,1.5379,True,True,50296.0,1.2884,1.4004
7,register_days_range,random,91-180,48134,19.2317,1.0917,0.1433,1.5212,True,True,9257.0,0.6266,1.4401
8,follow_user_num_range,random,500+,258413,18.4697,1.0485,0.1412,1.4989,True,True,47728.0,0.6139,1.4109
9,video_orientation,standard,square,21537,38.9562,0.8386,0.6640,1.4514,False,True,8390.0,1.5375,1.6712


In [83]:
# Preparing the click modeling dataset and feature scope

click_target_column = "is_click"

click_reference_columns = [
    "user_id", "video_id", "creator_user_id", "event_timestamp_ms", "event_date", "traffic_source"
]

click_predictor_columns = [
    "feed_tab_id",
    "user_active_degree",
    "is_video_author",
    "follow_user_num",
    "follow_user_num_range",
    "fans_user_num",
    "fans_user_num_range",
    "friend_user_num",
    "friend_user_num_range",
    "register_days",
    "register_days_range",
    "video_type",
    "video_upload_date",
    "video_upload_type",
    "video_width",
    "video_height",
    "video_orientation",
    "music_type",
    "video_age_days",
    "event_content_duration_ms"
]

click_model_dataset = (
    impression_analysis_base[
        click_reference_columns + click_predictor_columns + [click_target_column]
    ]
    .sort_values(["event_date", "event_timestamp_ms", "user_id", "video_id"])
    .reset_index(drop = True)
)

pd.DataFrame(
    {
        "metric_name": ["row_count",
                        "column_count",
                        "duplicate_event_key_count",
                        "missing_target_count"
        ],
        "value": [int(len(click_model_dataset)),
                  int(click_model_dataset.shape[1]),
                  int(
                      click_model_dataset.duplicated(
                          subset = ["user_id", "video_id", "event_timestamp_ms"]
                    ).sum()
                ),
                  int(click_model_dataset[click_target_column].isna().sum())
        ]
    }
)

,metric_name,value
0,row_count,2597699
1,column_count,27
2,duplicate_event_key_count,0
3,missing_target_count,0


In [84]:
# Documenting model roles for reference, predictors and target

click_model_field_catalog = pd.DataFrame(
    [
        ("user_id", "reference", "Viewer identifier for tracking and later diagnostics"),
        ("video_id", "reference", "Video identifier for tracking and later diagnostics"),
        ("creator_user_id", "reference", "Creator identifier for tracking and later diagnostics"),
        ("event_timestamp_ms", "reference", "Event timestamp for chronological ordering"),
        ("event_date", "reference", "Event date for chronological split and daily diagnostics"),
        ("traffic_source", "predictor", "Delivery source available at impression time"),
        ("feed_tab_id", "predictor", "Feed tab context available at impression time"),
        ("user_active_degree", "predictor", "User activity segment"),
        ("is_video_author", "predictor", "Whether the viewer is also a video author"),
        ("follow_user_num", "predictor", "Viewer follow count"),
        ("follow_user_num_range", "predictor", "Viewer follow count band"),
        ("fans_user_num", "predictor", "Viewer fans count"),
        ("fans_user_num_range", "predictor", "Viewer fans count band"),
        ("friend_user_num", "predictor", "Viewer friend count"),
        ("friend_user_num_range", "predictor", "Viewer friend count band"),
        ("register_days", "predictor", "Viewer registration age in days"),
        ("register_days_range", "predictor", "Viewer registration age band"),
        ("video_type", "predictor", "Video type"),
        ("video_upload_date", "predictor", "Video upload date"),
        ("video_upload_type", "predictor", "Video upload source type"),
        ("video_width", "predictor", "Video width"),
        ("video_height", "predictor", "Video height"),
        ("video_orientation", "predictor", "Video orientation"),
        ("music_type", "predictor", "Music type"),
        ("video_age_days", "predictor", "Video age at impression date"),
        ("event_content_duration_ms", "predictor", "Content duration available on the event"),
        ("is_click", "target", "Primary prediction target")
    ],
    columns = ["field_name", "model_role", "business_definition"]
)

click_model_field_catalog

,field_name,model_role,business_definition
0,user_id,reference,Viewer identifier for tracking and later diagn...
1,video_id,reference,Video identifier for tracking and later diagno...
2,creator_user_id,reference,Creator identifier for tracking and later diag...
3,event_timestamp_ms,reference,Event timestamp for chronological ordering
4,event_date,reference,Event date for chronological split and daily d...
5,traffic_source,predictor,Delivery source available at impression time
6,feed_tab_id,predictor,Feed tab context available at impression time
7,user_active_degree,predictor,User activity segment
8,is_video_author,predictor,Whether the viewer is also a video author
9,follow_user_num,predictor,Viewer follow count


In [85]:
# Creating chronological train, validation and test splits

train_end_date = pd.Timestamp("2022-04-28")
validation_end_date = pd.Timestamp("2022-05-03")

click_model_dataset["model_split"] = np.select(
    [
        click_model_dataset["event_date"] <= train_end_date,
        (click_model_dataset["event_date"] > train_end_date) & (click_model_dataset["event_date"] <= validation_end_date),
        click_model_dataset["event_date"] > validation_end_date
    ],
    [
        "train", "validation", "test"
    ],
    default = "unassigned"
)

split_summary = (click_model_dataset
                 .groupby("model_split", as_index = False)
                 .agg(row_count = ("user_id", "size"),
                      distinct_user_count = ("user_id", "nunique"),
                      distinct_video_count = ("video_id", "nunique"),
                      click_count = ("is_click", "sum")
    )
)

split_summary["click_rate_percent"] = (
    split_summary["click_count"] / split_summary["row_count"] * 100
).round(4)

split_summary

,model_split,row_count,distinct_user_count,distinct_video_count,click_count,click_rate_percent
0,test,600478,26362,7320,128259,21.3595
1,train,1533888,26842,7583,628565,40.9785
2,validation,463333,26103,7392,107906,23.2891


In [86]:
# Checking split composition and traffic-source balance
split_traffic_source_summary = (click_model_dataset
                                .groupby(["model_split", "traffic_source"], as_index = False)
                                .agg(row_count = ("user_id", "size"),
                                     click_count = ("is_click", "sum")
    )
)

split_traffic_source_summary["click_rate_percent"] = (
    split_traffic_source_summary["click_count"] / split_traffic_source_summary["row_count"] * 100
).round(4)

split_traffic_source_summary["split_traffic_share_percent"] = (
    split_traffic_source_summary["row_count"] / split_traffic_source_summary.groupby("model_split")["row_count"].transform("sum") * 100
).round(4)

display(split_summary)
display(split_traffic_source_summary.sort_values(["model_split", "traffic_source"]).reset_index(drop = True))

,model_split,row_count,distinct_user_count,distinct_video_count,click_count,click_rate_percent
0,test,600478,26362,7320,128259,21.3595
1,train,1533888,26842,7583,628565,40.9785
2,validation,463333,26103,7392,107906,23.2891


,model_split,traffic_source,row_count,click_count,click_rate_percent,split_traffic_share_percent
0,test,random,527865,95922,18.1717,87.9075
1,test,standard,72613,32337,44.5333,12.0925
2,train,random,288335,48140,16.6959,18.7977
3,train,standard,1245553,580425,46.5998,81.2023
4,validation,random,369849,64872,17.5401,79.8236
5,validation,standard,93484,43034,46.0335,20.1764


In [87]:
# Saving the full click modeling dataset for reuse

upload_dataframe_to_parquet_blob(click_model_dataset,
                                 container_names["curated"],
                                 "click_model_dataset.parquet"
)

pd.DataFrame(
    {
        "dataset_name": ["click_model_dataset"],
        "row_count": [len(click_model_dataset)],
        "column_count": [click_model_dataset.shape[1]],
        "split_count": [click_model_dataset["model_split"].nunique()]
    }
)

,dataset_name,row_count,column_count,split_count
0,click_model_dataset,2597699,28,3


In [88]:
# Diagnosing split drift before modeling decisions

click_model_split_diagnostics = (
    split_traffic_source_summary[
        [
            "model_split",
            "traffic_source",
            "row_count",
            "split_traffic_share_percent",
            "click_rate_percent"
        ]
    ]
    .sort_values(["model_split", "traffic_source"])
    .reset_index(drop = True)
)

split_drift_summary = pd.DataFrame(
    {
        "metric_name": ["train_standard_share_percent", "validation_standard_share_percent", "test_standard_share_percent",
                        "train_click_rate_percent", "validation_click_rate_percent", "test_click_rate_percent",
                        "standard_share_range_percent_points", "click_rate_range_percent_points",
        ],
        "value": [
            float(
                click_model_split_diagnostics.loc[
                    (click_model_split_diagnostics["model_split"] == "train") 
                    & (click_model_split_diagnostics["traffic_source"] == "standard"),
                    "split_traffic_share_percent"
                ].iloc[0]
            ),
            float(
                click_model_split_diagnostics.loc[
                    (click_model_split_diagnostics["model_split"] == "validation")
                    & (click_model_split_diagnostics["traffic_source"] == "standard"),
                    "split_traffic_share_percent"
                ].iloc[0]
            ),
            float(
                click_model_split_diagnostics.loc[
                    (click_model_split_diagnostics["model_split"] == "test")
                    & (click_model_split_diagnostics["traffic_source"] == "standard"),
                    "split_traffic_share_percent"
                ].iloc[0]
            ),
            float(
                split_summary.loc[
                    split_summary["model_split"] == "train",
                    "click_rate_percent"
                ].iloc[0]
            ),
            float(
                split_summary.loc[
                    split_summary["model_split"] == "validation",
                    "click_rate_percent"
                ].iloc[0]
            ),
            float(
                split_summary.loc[
                    split_summary["model_split"] == "test",
                    "click_rate_percent"
                ].iloc[0]
            ),
            round(
                split_traffic_source_summary.loc[
                    split_traffic_source_summary["traffic_source"] == "standard",
                    "split_traffic_share_percent"
                ].max()
                - split_traffic_source_summary.loc[
                    split_traffic_source_summary["traffic_source"] == "standard",
                    "split_traffic_share_percent"
                ].min(),
                4
            ),
            round(
                split_summary["click_rate_percent"].max()
                - split_summary["click_rate_percent"].min(),
                4
            )
        ]
    }
)

display(click_model_split_diagnostics)
display(split_drift_summary)

,model_split,traffic_source,row_count,split_traffic_share_percent,click_rate_percent
0,test,random,527865,87.9075,18.1717
1,test,standard,72613,12.0925,44.5333
2,train,random,288335,18.7977,16.6959
3,train,standard,1245553,81.2023,46.5998
4,validation,random,369849,79.8236,17.5401
5,validation,standard,93484,20.1764,46.0335


,metric_name,value
0,train_standard_share_percent,81.2023
1,validation_standard_share_percent,20.1764
2,test_standard_share_percent,12.0925
3,train_click_rate_percent,40.9785
4,validation_click_rate_percent,23.2891
5,test_click_rate_percent,21.3595
6,standard_share_range_percent_points,69.1098
7,click_rate_range_percent_points,19.6190


In [89]:
# Isolating standard traffic for the main click model path

click_model_dataset_standard_traffic = (click_model_dataset.loc[click_model_dataset["traffic_source"] == "standard"]
                                        .copy()
                                        .sort_values(["event_date", "event_timestamp_ms", "user_id", "video_id"])
                                        .reset_index(drop = True)
)

standard_traffic_split_summary = (click_model_dataset_standard_traffic
                                  .groupby("model_split", as_index = False)
                                  .agg(row_count = ("user_id", "size"),
                                       distinct_user_count = ("user_id", "nunique"),
                                       distinct_video_count = ("video_id", "nunique"),
                                       click_count = ("is_click", "sum")
    )
)

standard_traffic_split_summary["click_rate_percent"] = (
    standard_traffic_split_summary["click_count"] / standard_traffic_split_summary["row_count"] * 100
).round(4)

standard_traffic_split_summary

,model_split,row_count,distinct_user_count,distinct_video_count,click_count,click_rate_percent
0,test,72613,19357,5217,32337,44.5333
1,train,1245553,26631,7545,580425,46.5998
2,validation,93484,20550,5277,43034,46.0335


In [90]:
# Confirming that the standard-traffic split is stable enough

standard_traffic_split_diagnostics = pd.DataFrame(
    {
        "metric_name": ["train_row_count", "validation_row_count", "test_row_count",
                        "train_click_rate_percent", "validation_click_rate_percent", "test_click_rate_percent",
                        "click_rate_range_percent_points", "duplicate_event_key_count", "missing_target_count"
        ],
        "value": [
            int(
                standard_traffic_split_summary.loc[
                    standard_traffic_split_summary["model_split"] == "train",
                    "row_count"
                ].iloc[0]
            ),
            int(
                standard_traffic_split_summary.loc[
                    standard_traffic_split_summary["model_split"] == "validation",
                    "row_count"
                ].iloc[0]
            ),
            int(
                standard_traffic_split_summary.loc[
                    standard_traffic_split_summary["model_split"] == "test",
                    "row_count"
                ].iloc[0]
            ),
            float(
                standard_traffic_split_summary.loc[
                    standard_traffic_split_summary["model_split"] == "train",
                    "click_rate_percent"
                ].iloc[0]
            ),
            float(
                standard_traffic_split_summary.loc[
                    standard_traffic_split_summary["model_split"] == "validation",
                    "click_rate_percent"
                ].iloc[0]
            ),
            float(
                standard_traffic_split_summary.loc[
                    standard_traffic_split_summary["model_split"] == "test",
                    "click_rate_percent"
                ].iloc[0]
            ),
            round(
                standard_traffic_split_summary["click_rate_percent"].max()
                - standard_traffic_split_summary["click_rate_percent"].min(),
                4
            ),
            int(
                click_model_dataset_standard_traffic.duplicated(
                    subset = ["user_id", "video_id", "event_timestamp_ms"]
                ).sum()
            ),
            int(click_model_dataset_standard_traffic["is_click"].isna().sum())
        ]
    }
)

standard_traffic_split_diagnostics

,metric_name,value
0,train_row_count,1.245553e+06
1,validation_row_count,9.348400e+04
2,test_row_count,7.261300e+04
3,train_click_rate_percent,4.659980e+01
4,validation_click_rate_percent,4.603350e+01
5,test_click_rate_percent,4.453330e+01
6,click_rate_range_percent_points,2.066500e+00
7,duplicate_event_key_count,0.000000e+00
8,missing_target_count,0.000000e+00


In [91]:
# Saving the standard-traffic modeling dataset for production use
upload_dataframe_to_parquet_blob(click_model_dataset_standard_traffic,
                                 container_names["curated"],
                                 "click_model_dataset_standard_traffic.parquet"
)

pd.DataFrame(
    {
        "dataset_name": ["click_model_dataset_standard_traffic"],
        "row_count": [len(click_model_dataset_standard_traffic)],
        "column_count": [click_model_dataset_standard_traffic.shape[1]],
        "split_count": [click_model_dataset_standard_traffic["model_split"].nunique()]
    }
)

,dataset_name,row_count,column_count,split_count
0,click_model_dataset_standard_traffic,1411650,28,3


In [92]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (average_precision_score,
                             brier_score_loss,
                             log_loss,
                             roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

In [93]:
# Separating modeling inputs, target, and chronological splits

modeling_base = click_model_dataset_standard_traffic.copy()

target_column = "is_click"
split_column = "model_split"

reference_columns = ["user_id", "video_id", "creator_user_id", "event_timestamp_ms", "event_date",
                     "video_upload_date", split_column
]

feature_columns = ["traffic_source", "feed_tab_id", "user_active_degree", "is_video_author", "follow_user_num",
                   "follow_user_num_range", "fans_user_num", "fans_user_num_range", "friend_user_num", "friend_user_num_range",
                   "register_days", "register_days_range", "video_type", "video_upload_type", "video_width", "video_height",
                   "video_orientation", "music_type", "video_age_days", "event_content_duration_ms"
]

missing_model_columns = [column_name
                         for column_name in feature_columns + [target_column, split_column]
                         if column_name not in modeling_base.columns
]

if missing_model_columns:
    raise KeyError(f"Missing required modeling columns : {missing_model_columns}")


train_mask = modeling_base[split_column].eq("train")
validation_mask = modeling_base[split_column].eq("validation")
test_mask = modeling_base[split_column].eq("test")

X_train = modeling_base.loc[train_mask, feature_columns].copy()
y_train = modeling_base.loc[train_mask, target_column].copy()

X_validation = modeling_base.loc[validation_mask, feature_columns].copy()
y_validation = modeling_base.loc[validation_mask, target_column].copy()

X_test = modeling_base.loc[test_mask, feature_columns].copy()
y_test = modeling_base.loc[test_mask, target_column].copy()


pd.DataFrame(
    {
        "split_name": ["train", "validation", "test"],
        "row_count": [len(X_train), len(X_validation), len(X_test)],
        "click_rate_percent": [round(y_train.mean() * 100, 4),
                               round(y_validation.mean() * 100, 4),
                               round(y_test.mean() * 100, 4)
        ]
    }
)

,split_name,row_count,click_rate_percent
0,train,1245553,46.5998
1,validation,93484,46.0335
2,test,72613,44.5333


In [94]:
# Defining the preprocessing plan for mixed feature types
categorical_features = ["traffic_source",
                        "feed_tab_id",
                        "user_active_degree",
                        "follow_user_num_range",
                        "fans_user_num_range",
                        "friend_user_num_range",
                        "register_days_range",
                        "video_type",
                        "video_upload_type",
                        "video_orientation",
                        "music_type"
]

numeric_features = ["is_video_author",
                    "follow_user_num",
                    "fans_user_num",
                    "friend_user_num",
                    "register_days",
                    "video_width",
                    "video_height",
                    "video_age_days",
                    "event_content_duration_ms"
]

preprocessor = ColumnTransformer(
    transformers = [
        (
            "categorical",
            Pipeline(
                steps = [
                    ("imputer", SimpleImputer(strategy = "most_frequent")),
                    ("encoder", OneHotEncoder(handle_unknown = "ignore"))
                ]
            ),
            categorical_features
        ),
        (
            "numeric",
            Pipeline(
                steps = [
                    ("imputer", SimpleImputer(strategy = "median")),
                    ("scaler", StandardScaler())
                ]
            ),
            numeric_features
        )
    ],
    remainder = "drop"
)

In [95]:
# Training baseline and logistic click models for comparison

dummy_model = Pipeline(
    steps = [
        ("preprocessor", preprocessor),
        ("classifier", DummyClassifier(strategy = "prior"))
    ]
)

logistic_click_model = Pipeline(
    steps = [
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(max_iter = 1000,
                               class_weight = "balanced",
                               random_state = 42
            )
        )
    ]
)

dummy_model.fit(X_train, y_train)
logistic_click_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int8](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](20,)","['traffic_source','feed_tab_id','user_active_degree',...,'music_type', 'video_age_days','event_content_duration_ms']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,20
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default o

In [96]:
# Evaluating model quality with ranking and calibration metrics

def evaluate_binary_classifier(model, X_frame, y_true, split_name):
    predicted_probability = model.predict_proba(X_frame)[ : , 1]

    return {"split_name": split_name,
            "row_count": int(len(y_true)),
            "positive_rate_percent": round(y_true.mean() * 100, 4),
            "roc_auc": round(roc_auc_score(y_true, predicted_probability), 6),
            "average_precision": round(average_precision_score(y_true, predicted_probability), 6),
            "log_loss": round(log_loss(y_true, predicted_probability), 6),
            "brier_score": round(brier_score_loss(y_true, predicted_probability), 6)
    }

model_evaluation_summary = pd.DataFrame(
    [
        {"model_name": "dummy_click_model", **evaluate_binary_classifier(dummy_model, X_validation, y_validation, "validation")},
        {"model_name": "dummy_click_model", **evaluate_binary_classifier(dummy_model, X_test, y_test, "test")},
        {"model_name": "logistic_click_model", **evaluate_binary_classifier(logistic_click_model, X_validation, y_validation, "validation")},
        {"model_name": "logistic_click_model", **evaluate_binary_classifier(logistic_click_model, X_test, y_test, "test")}
    ]
)

model_evaluation_summary

,model_name,split_name,row_count,positive_rate_percent,roc_auc,average_precision,log_loss,brier_score
0,dummy_click_model,validation,93484,46.0335,0.500000,0.460335,0.690062,0.248459
1,dummy_click_model,test,72613,44.5333,0.500000,0.445333,0.688018,0.247439
2,logistic_click_model,validation,93484,46.0335,0.656781,0.583613,0.637296,0.225342
3,logistic_click_model,test,72613,44.5333,0.646363,0.563044,0.641065,0.227126


In [97]:
# Scoring test impressions and checking score separation by decile

test_scored_impressions = modeling_base.loc[test_mask, reference_columns + feature_columns + [target_column]].copy()
test_scored_impressions["predicted_click_probability"] = logistic_click_model.predict_proba(X_test)[ : , 1]

test_scored_impressions["score_decile"] = pd.qcut(
    test_scored_impressions["predicted_click_probability"],
    q = 10,
    labels = False,
    duplicates = "drop"
)

test_score_decile_summary = (test_scored_impressions
                             .groupby("score_decile", as_index = False)
                             .agg(impression_count = ("user_id", "size"),
                                  actual_click_count = ("is_click", "sum"),
                                  average_predicted_click_probability = ("predicted_click_probability", "mean")
    )
    .sort_values("score_decile", ascending = False)
    .reset_index(drop = True)
)

test_score_decile_summary["actual_click_rate_percent"] = (
    test_score_decile_summary["actual_click_count"] / test_score_decile_summary["impression_count"] * 100
).round(4)

test_score_decile_summary["average_predicted_click_probability_percent"] = (
    test_score_decile_summary["average_predicted_click_probability"] * 100
).round(4)

test_score_decile_summary

,score_decile,impression_count,actual_click_count,average_predicted_click_probability,actual_click_rate_percent,average_predicted_click_probability_percent
0,9,7262,4491,0.640171,61.8425,64.0171
1,8,7261,4101,0.590873,56.4798,59.0873
2,7,7261,3913,0.563557,53.8906,56.3557
3,6,7261,3581,0.541127,49.3183,54.1127
4,5,7261,3434,0.522790,47.2938,52.2790
5,4,7262,3413,0.507010,46.9981,50.7010
6,3,7261,3301,0.489726,45.4621,48.9726
7,2,7261,3097,0.462436,42.6525,46.2436
8,1,7261,2386,0.355366,32.8605,35.5366
9,0,7262,620,0.088517,8.5376,8.8517


In [98]:
# Extracting directional model signals for business interpretation
encoded_feature_names = logistic_click_model.named_steps["preprocessor"].get_feature_names_out()
logistic_coefficients = logistic_click_model.named_steps["classifier"].coef_[0]

logistic_feature_importance = pd.DataFrame(
    {
        "feature_name": encoded_feature_names,
        "coefficient": logistic_coefficients,
        "absolute_coefficient": np.abs(logistic_coefficients)
    }
).sort_values("absolute_coefficient", ascending = False).reset_index(drop = True)

top_positive_click_drivers = logistic_feature_importance.sort_values("coefficient", ascending = False).head(20).reset_index(drop = True)
top_negative_click_drivers = logistic_feature_importance.sort_values("coefficient", ascending = True).head(20).reset_index(drop = True)

display(top_positive_click_drivers)
display(top_negative_click_drivers)

,feature_name,coefficient,absolute_coefficient
0,categorical__feed_tab_id_9,3.435293,3.435293
1,categorical__feed_tab_id_10,1.611975,1.611975
2,categorical__feed_tab_id_4,1.478394,1.478394
3,categorical__feed_tab_id_1,1.132836,1.132836
4,categorical__feed_tab_id_2,0.884750,0.884750
5,categorical__video_type_AD,0.686651,0.686651
6,categorical__feed_tab_id_7,0.654618,0.654618
7,categorical__video_upload_type_Web,0.592489,0.592489
8,categorical__video_upload_type_Kmovie,0.555294,0.555294
9,categorical__video_upload_type_LongImport,0.550769,0.550769


,feature_name,coefficient,absolute_coefficient
0,categorical__feed_tab_id_3,-2.865176,2.865176
1,categorical__feed_tab_id_8,-2.333047,2.333047
2,categorical__feed_tab_id_11,-1.914471,1.914471
3,categorical__feed_tab_id_0,-1.240870,1.240870
4,categorical__user_active_degree_UNKNOWN,-1.065955,1.065955
5,categorical__feed_tab_id_12,-0.796327,0.796327
6,categorical__video_upload_type_PictureSet,-0.692668,0.692668
7,categorical__video_upload_type_UNKNOWN,-0.619232,0.619232
8,categorical__video_upload_type_FollowShoot,-0.570328,0.570328
9,categorical__music_type_8.0,-0.560285,0.560285


In [99]:
# Rebuilding the standard-traffic modeling frame for the final path

standard_modeling_base = click_model_dataset_standard_traffic.copy()

standard_target_column = "is_click"
standard_split_column = "model_split"

standard_reference_columns = ["user_id", "video_id", "creator_user_id",
                              "event_timestamp_ms", "event_date",
                              "video_upload_date", standard_split_column
]

standard_feature_columns = ["feed_tab_id", "user_active_degree", "is_video_author", 
                            "follow_user_num", "follow_user_num_range",
                            "fans_user_num", "fans_user_num_range", 
                            "friend_user_num", "friend_user_num_range", "register_days",
                            "register_days_range", 
                            "video_type", "video_upload_type", "video_width", "video_height", "video_orientation", 
                            "music_type", "video_age_days", "event_content_duration_ms"
]

missing_standard_model_columns = [
    column_name
    for column_name in standard_feature_columns + [standard_target_column, standard_split_column]
    if column_name not in standard_modeling_base.columns
]

if missing_standard_model_columns:
    raise KeyError(f"Missing required modeling columns : {missing_standard_model_columns}")


standard_train_mask = standard_modeling_base[standard_split_column].eq("train")
standard_validation_mask = standard_modeling_base[standard_split_column].eq("validation")
standard_test_mask = standard_modeling_base[standard_split_column].eq("test")

X_train_standard = standard_modeling_base.loc[standard_train_mask, standard_feature_columns].copy()
y_train_standard = standard_modeling_base.loc[standard_train_mask, standard_target_column].copy()

X_validation_standard = standard_modeling_base.loc[standard_validation_mask, standard_feature_columns].copy()
y_validation_standard = standard_modeling_base.loc[standard_validation_mask, standard_target_column].copy()

X_test_standard = standard_modeling_base.loc[standard_test_mask, standard_feature_columns].copy()
y_test_standard = standard_modeling_base.loc[standard_test_mask, standard_target_column].copy()


pd.DataFrame(
    {
        "split_name": ["train", "validation", "test"],
        "row_count": [len(X_train_standard), len(X_validation_standard), len(X_test_standard)],
        "click_rate_percent": [round(y_train_standard.mean() * 100, 4),
                               round(y_validation_standard.mean() * 100, 4),
                               round(y_test_standard.mean() * 100, 4)
        ]
    }
)

,split_name,row_count,click_rate_percent
0,train,1245553,46.5998
1,validation,93484,46.0335
2,test,72613,44.5333


In [100]:
# Defining and fitting the final standard-traffic model pipeline

standard_categorical_features = ["feed_tab_id", "user_active_degree", "follow_user_num_range", "fans_user_num_range",
                                 "friend_user_num_range", "register_days_range", "video_type",
                                 "video_upload_type", "video_orientation", "music_type"
]

standard_numeric_features = ["is_video_author", "follow_user_num", "fans_user_num", "friend_user_num", "register_days",
                             "video_width", "video_height", "video_age_days", "event_content_duration_ms"
]

standard_preprocessor = ColumnTransformer(
    transformers = [
        (
            "categorical",
            Pipeline(
                steps = [
                    ("imputer", SimpleImputer(strategy = "most_frequent")),
                    ("encoder", OneHotEncoder(handle_unknown = "ignore"))
                ]
            ),
            standard_categorical_features
        ),
        (
            "numeric",
            Pipeline(
                steps = [
                    ("imputer", SimpleImputer(strategy = "median")),
                    ("scaler", StandardScaler())
                ]
            ),
            standard_numeric_features
        )
    ],
    remainder = "drop"
)

standard_dummy_model = Pipeline(
    steps = [
        ("preprocessor", standard_preprocessor),
        ("classifier", DummyClassifier(strategy = "prior"))
    ]
)

standard_logistic_click_model = Pipeline(
    steps = [
        ("preprocessor", standard_preprocessor),
        (
            "classifier",
            LogisticRegression(max_iter = 1000,
                               class_weight = "balanced",
                               random_state = 42
            )
        )
    ]
)

standard_dummy_model.fit(X_train_standard, y_train_standard)
standard_logistic_click_model.fit(X_train_standard, y_train_standard)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int8](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](19,)","['feed_tab_id','user_active_degree','is_video_author',...,'music_type', 'video_age_days','event_content_duration_ms']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,19
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default 

In [101]:
# Summarizing final model performance on validation and test splits
standard_model_evaluation_summary = pd.DataFrame(
    [
        {
            "model_name": "standard_dummy_click_model",
            **evaluate_binary_classifier(standard_dummy_model,
                                         X_validation_standard,
                                         y_validation_standard,
                                         "validation"
            )
        },
        {
            "model_name": "standard_dummy_click_model",
            **evaluate_binary_classifier(standard_dummy_model,
                                         X_test_standard,
                                         y_test_standard,
                                         "test"
            )
        },
        {
            "model_name": "standard_logistic_click_model",
            **evaluate_binary_classifier(standard_logistic_click_model,
                                         X_validation_standard,
                                         y_validation_standard,
                                         "validation"
            )
        },
        {
            "model_name": "standard_logistic_click_model",
            **evaluate_binary_classifier(standard_logistic_click_model,
                                         X_test_standard,
                                         y_test_standard,
                                         "test"
            )
        }
    ]
)

standard_model_evaluation_summary

,model_name,split_name,row_count,positive_rate_percent,roc_auc,average_precision,log_loss,brier_score
0,standard_dummy_click_model,validation,93484,46.0335,0.500000,0.460335,0.690062,0.248459
1,standard_dummy_click_model,test,72613,44.5333,0.500000,0.445333,0.688018,0.247439
2,standard_logistic_click_model,validation,93484,46.0335,0.656571,0.583219,0.637392,0.225380
3,standard_logistic_click_model,test,72613,44.5333,0.646208,0.562669,0.641121,0.227147


In [102]:
# Scoring the final test set and validating probability ranking

standard_test_scored_impressions = (
    standard_modeling_base.loc[
        standard_test_mask,
        standard_reference_columns + standard_feature_columns + [standard_target_column]
    ]
    .copy()
)

standard_test_scored_impressions["predicted_click_probability"] = (
    standard_logistic_click_model.predict_proba(X_test_standard)[ : , 1]
)

standard_test_scored_impressions["score_decile"] = pd.qcut(
    standard_test_scored_impressions["predicted_click_probability"],
    q = 10,
    labels = False,
    duplicates = "drop"
)

standard_test_score_decile_summary = (
    standard_test_scored_impressions
    .groupby("score_decile", as_index = False)
    .agg(impression_count = ("user_id", "size"),
         actual_click_count = ("is_click", "sum"),
         average_predicted_click_probability = ("predicted_click_probability", "mean")
    )
    .sort_values("score_decile", ascending = False)
    .reset_index(drop = True)
)

standard_test_score_decile_summary["actual_click_rate_percent"] = (
    standard_test_score_decile_summary["actual_click_count"] / standard_test_score_decile_summary["impression_count"] * 100
).round(4)

standard_test_score_decile_summary["average_predicted_click_probability_percent"] = (
    standard_test_score_decile_summary["average_predicted_click_probability"] * 100
).round(4)

standard_test_score_decile_summary

,score_decile,impression_count,actual_click_count,average_predicted_click_probability,actual_click_rate_percent,average_predicted_click_probability_percent
0,9,7262,4485,0.640438,61.7598,64.0438
1,8,7261,4099,0.590985,56.4523,59.0985
2,7,7261,3903,0.563592,53.7529,56.3592
3,6,7261,3599,0.540991,49.5662,54.0991
4,5,7261,3440,0.522807,47.3764,52.2807
5,4,7262,3398,0.507114,46.7915,50.7114
6,3,7261,3315,0.489823,45.6549,48.9823
7,2,7261,3093,0.462371,42.5974,46.2371
8,1,7261,2385,0.354834,32.8467,35.4834
9,0,7262,620,0.088690,8.5376,8.8690


In [103]:
# Extracting final model drivers for reporting use

standard_encoded_feature_names = (
    standard_logistic_click_model.named_steps["preprocessor"].get_feature_names_out()
)

standard_logistic_coefficients = (
    standard_logistic_click_model.named_steps["classifier"].coef_[0]
)

standard_logistic_feature_importance = (
    pd.DataFrame(
        {
            "feature_name": standard_encoded_feature_names,
            "coefficient": standard_logistic_coefficients,
            "absolute_coefficient": np.abs(standard_logistic_coefficients),
        }
    )
    .sort_values("absolute_coefficient", ascending = False)
    .reset_index(drop = True)
)

standard_top_positive_click_drivers = (standard_logistic_feature_importance
                                       .sort_values("coefficient", ascending = False)
                                       .head(20)
                                       .reset_index(drop = True)
)

standard_top_negative_click_drivers = (standard_logistic_feature_importance
                                       .sort_values("coefficient", ascending = True)
                                       .head(20)
                                       .reset_index(drop = True)
)

display(standard_top_positive_click_drivers)
display(standard_top_negative_click_drivers)

,feature_name,coefficient,absolute_coefficient
0,categorical__feed_tab_id_9,3.324646,3.324646
1,categorical__feed_tab_id_10,1.497227,1.497227
2,categorical__feed_tab_id_4,1.474444,1.474444
3,categorical__feed_tab_id_1,1.133910,1.133910
4,categorical__feed_tab_id_2,0.881882,0.881882
5,categorical__feed_tab_id_7,0.655028,0.655028
6,categorical__video_type_AD,0.647854,0.647854
7,categorical__video_upload_type_Web,0.575487,0.575487
8,categorical__video_upload_type_Kmovie,0.540185,0.540185
9,categorical__video_upload_type_LongImport,0.533568,0.533568


,feature_name,coefficient,absolute_coefficient
0,categorical__feed_tab_id_3,-2.858486,2.858486
1,categorical__feed_tab_id_8,-2.360317,2.360317
2,categorical__feed_tab_id_11,-1.825047,1.825047
3,categorical__feed_tab_id_0,-1.238606,1.238606
4,categorical__user_active_degree_UNKNOWN,-1.033879,1.033879
5,categorical__feed_tab_id_12,-0.803857,0.803857
6,categorical__video_upload_type_PictureSet,-0.688190,0.688190
7,categorical__video_upload_type_UNKNOWN,-0.633513,0.633513
8,categorical__video_type_NORMAL,-0.590379,0.590379
9,categorical__music_type_8.0,-0.588924,0.588924


In [104]:
# Saving scored test impressions for downstream consumption
upload_dataframe_to_parquet_blob(standard_test_scored_impressions,
                                 container_names["model_output"],
                                 "click_model_scored_standard_traffic_test.parquet"
)

pd.DataFrame(
    {
        "dataset_name": ["click_model_scored_standard_traffic_test"],
        "row_count": [len(standard_test_scored_impressions)],
        "column_count": [standard_test_scored_impressions.shape[1]]
    }
)

,dataset_name,row_count,column_count
0,click_model_scored_standard_traffic_test,72613,29


In [105]:
# Ranking creators by standard-traffic engagement quality

creator_performance_summary = (
    impression_analysis_base.loc[impression_analysis_base["traffic_source"] == "standard"]
    .groupby("creator_user_id", as_index = False)
    .agg(impression_count = ("user_id", "size"),
         distinct_video_count = ("video_id", "nunique"),
         click_count = ("is_click", "sum"),
         meaningful_watch_count = ("is_meaningful_watch", "sum"),
         high_value_engagement_count = ("is_high_value_engagement", "sum")
    )
)

creator_performance_summary["click_rate_percent"] = (
    creator_performance_summary["click_count"] / creator_performance_summary["impression_count"] * 100
).round(4)

creator_performance_summary["meaningful_watch_rate_percent"] = (
    creator_performance_summary["meaningful_watch_count"] / creator_performance_summary["impression_count"] * 100
).round(4)

creator_performance_summary["high_value_engagement_rate_percent"] = (
    creator_performance_summary["high_value_engagement_count"] / creator_performance_summary["impression_count"] * 100
).round(4)

creator_hve_interval = creator_performance_summary.apply(
    lambda row: wilson_interval(
        row["high_value_engagement_count"],
        row["impression_count"]
    ),
    axis = 1
)

creator_performance_summary["high_value_engagement_rate_center"] = [
    interval[0] for interval in creator_hve_interval
]

creator_performance_summary["high_value_engagement_rate_margin"] = [
    interval[1] for interval in creator_hve_interval
]

creator_performance_summary["high_value_engagement_rate_lower_bound_percent"] = (
    (
        creator_performance_summary["high_value_engagement_rate_center"] - creator_performance_summary["high_value_engagement_rate_margin"]
    ) * 100
).round(4)

creator_priority_summary = (
    creator_performance_summary.loc[creator_performance_summary["impression_count"] >= 1000]
    .sort_values(
        [
            "high_value_engagement_rate_lower_bound_percent",
            "click_rate_percent",
            "impression_count",
        ],
        ascending = [False, False, False]
    )
    .reset_index(drop = True)
)

creator_priority_summary.head(30)

,creator_user_id,impression_count,distinct_video_count,click_count,meaningful_watch_count,high_value_engagement_count,click_rate_percent,meaningful_watch_rate_percent,high_value_engagement_rate_percent,high_value_engagement_rate_center,high_value_engagement_rate_margin,high_value_engagement_rate_lower_bound_percent
0,8346076,4887,1,2991,2472,120,61.2032,50.5832,2.4555,0.024928,0.004354,2.0575
1,663099,2008,1,1303,1021,45,64.8904,50.8466,2.2410,0.023322,0.006532,1.6790
2,7813686,1284,1,723,393,27,56.3084,30.6075,2.1028,0.022457,0.007965,1.4491
3,5468688,1647,1,875,628,29,53.1269,38.1299,1.7608,0.018730,0.006443,1.2287
4,2517717,2218,2,955,662,35,43.0568,29.8467,1.5780,0.016617,0.005249,1.1368
5,23123,1544,2,1081,797,23,70.0130,51.6192,1.4896,0.016100,0.006154,0.9946
6,5570290,1185,1,698,585,18,58.9030,49.3671,1.5190,0.016756,0.007127,0.9630
7,7941871,2509,2,1529,1175,33,60.9406,46.8314,1.3153,0.013897,0.004516,0.9381
8,2562264,1823,1,565,358,23,30.9929,19.6380,1.2617,0.013641,0.005220,0.8422
9,803086,2121,1,1447,1259,26,68.2225,59.3588,1.2258,0.013140,0.004761,0.8379


In [106]:
# Exporting presentation-ready tables for Power BI use

presentation_tables = {"overall_funnel_summary": overall_funnel_summary,
                       "traffic_source_funnel_summary": traffic_source_funnel_summary,
                       "feed_tab_funnel_summary": feed_tab_funnel_summary,
                       "daily_rate_decomposition": daily_rate_decomposition,
                       "priority_segment_opportunities": priority_segment_opportunities,
                       "post_click_priority_segments": post_click_priority_segments,
                       "post_click_conversion_by_traffic_source": post_click_conversion_by_traffic_source,
                       "video_tag_performance": video_tag_performance,
                       "creator_priority_summary": creator_priority_summary,
                       "standard_model_evaluation_summary": standard_model_evaluation_summary,
                       "standard_test_score_decile_summary": standard_test_score_decile_summary,
                       "standard_top_positive_click_drivers": standard_top_positive_click_drivers,
                       "standard_top_negative_click_drivers": standard_top_negative_click_drivers
}

def make_parquet_safe(dataframe):
    parquet_safe_frame = dataframe.copy()

    for column_name in parquet_safe_frame.columns:
        if parquet_safe_frame[column_name].dtype == "object":
            parquet_safe_frame[column_name] = parquet_safe_frame[column_name].astype("string")

    return parquet_safe_frame

presentation_export_summary = []


# Converting export tables into parquet-safe formats before saving
for table_name, table_frame in presentation_tables.items():
    parquet_safe_table = make_parquet_safe(table_frame)

    upload_dataframe_to_parquet_blob(
        parquet_safe_table,
        container_names["presentation"],
        f"{table_name}.parquet"
    )

    presentation_export_summary.append(
        {
            "table_name": table_name,
            "row_count": len(parquet_safe_table),
            "column_count": parquet_safe_table.shape[1]
        }
    )

pd.DataFrame(presentation_export_summary)

,table_name,row_count,column_count
0,overall_funnel_summary,9,3
1,traffic_source_funnel_summary,2,18
2,feed_tab_funnel_summary,15,18
3,daily_rate_decomposition,30,20
4,priority_segment_opportunities,46,13
5,post_click_priority_segments,25,13
6,post_click_conversion_by_traffic_source,2,16
7,video_tag_performance,46,7
8,creator_priority_summary,260,12
9,standard_model_evaluation_summary,4,8
